# Regression (RNN, GRU, LSTM, Mamba) Models

### OSI Features Only (Regression) -- training + test + validation

In [ ]:
base_template = [
    "OSI_mean_TW{}", "OSI_std_TW{}"
]

tw_features = {
    i: [f.format(i) for f in base_template]
    for i in range(1, 7)
}

### Vent_Time Features Only (Regression) -- training + test + validation

In [ ]:
base_template = [
    "PIP_mean_TW{}", "PIP_std_TW{}",
    "PEEP_mean_TW{}", "PEEP_std_TW{}", "TV_mean_TW{}(mL/Kg)", "TV_std_TW{}(mL/Kg)",
    "Avg_NegFlowDur_TW{}", "Std_NegFlowDur_TW{}", "Avg_PeakInterval_TW{}", "Std_PeakInterval_TW{}"
]

tw_features = {
    i: [f.format(i) for f in base_template]
    for i in range(1, 7)
}

### Vent_Freq Features Only (Regression) -- training + test + validation

In [ ]:
base_template = [
    "PIP_mean_TW{}", "PIP_std_TW{}",
    "PEEP_mean_TW{}", "PEEP_std_TW{}", "TV_mean_TW{}(mL/Kg)", "TV_std_TW{}(mL/Kg)"
]

tw_features = {
    i: [f.format(i) for f in base_template] + [f"w{i}_SubBandEnergy_row{j}" for j in range(1, 17)]
    for i in range(1, 7)
}

### Vent_All Features Only (Regression) -- training + test + validation

In [ ]:
base_template = [
    "PIP_mean_TW{}", "PIP_std_TW{}",
    "PEEP_mean_TW{}", "PEEP_std_TW{}", "TV_mean_TW{}(mL/Kg)", "TV_std_TW{}(mL/Kg)",
    "Avg_NegFlowDur_TW{}", "Std_NegFlowDur_TW{}", "Avg_PeakInterval_TW{}", "Std_PeakInterval_TW{}"
]

tw_features = {
    i: [f.format(i) for f in base_template] + [f"w{i}_SubBandEnergy_row{j}" for j in range(1, 17)]
    for i in range(1, 7)
}

### OSI+Vent_Time Features Only (Regression) -- training + test + validation

In [ ]:
base_template = [
    "OSI_mean_TW{}", "OSI_std_TW{}", "PIP_mean_TW{}", "PIP_std_TW{}",
    "PEEP_mean_TW{}", "PEEP_std_TW{}", "TV_mean_TW{}(mL/Kg)", "TV_std_TW{}(mL/Kg)",
    "Avg_NegFlowDur_TW{}", "Std_NegFlowDur_TW{}", "Avg_PeakInterval_TW{}", "Std_PeakInterval_TW{}"
]

tw_features = {
    i: [f.format(i) for f in base_template]
    for i in range(1, 7)
}

### OSI+Vent_Freq Features Only (Regression) -- training + test + validation

In [ ]:

base_template = [
    "OSI_mean_TW{}", "OSI_std_TW{}", "PIP_mean_TW{}", "PIP_std_TW{}",
    "PEEP_mean_TW{}", "PEEP_std_TW{}", "TV_mean_TW{}(mL/Kg)", "TV_std_TW{}(mL/Kg)"
]

tw_features = {
    i: [f.format(i) for f in base_template] + [f"w{i}_SubBandEnergy_row{j}" for j in range(1, 17)]
    for i in range(1, 7)
}

### OSI+Vent_All Features Only (Regression) -- training + test + validation (_8)

In [ ]:
base_template = [
    "OSI_mean_TW{}", "OSI_std_TW{}", "PIP_mean_TW{}", "PIP_std_TW{}",
    "PEEP_mean_TW{}", "PEEP_std_TW{}", "TV_mean_TW{}(mL/Kg)", "TV_std_TW{}(mL/Kg)",
    "Avg_NegFlowDur_TW{}", "Std_NegFlowDur_TW{}", "Avg_PeakInterval_TW{}", "Std_PeakInterval_TW{}"
]

tw_features = {
    i: [f.format(i) for f in base_template] + [f"w{i}_SubBandEnergy_row{j}" for j in range(1, 17)]
    for i in range(1, 7)
}

In [ ]:
# ======================================================================================
# TBME Major Revision Utilities (REGRESSION: OSI + VentALL feature set; NO PCA)
# - Save BEST config+weights per model type (RNN/LSTM/GRU/Transformer/Mamba)
# - Save TRAINING CV predictions + summary
# - Evaluate BEST per model on TEST + TEMPORAL VALIDATION and save y_true/y_pred
# - Learning curve (20/40/60/80/100% of training patients)
# - NO PCA sensitivity (disabled)
# - Patient-level bootstrap CIs for RMSE/MAE on TEST/VAL preds
# ======================================================================================

import os
import math
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from mamba_ssm import Mamba

# ======================================================================================
# ================================ USER CONTROLS ======================================
# ======================================================================================

RUN_GRID_SEARCH       = True     # trains many configs and saves best-per-model-type
RUN_EVAL_BEST_MODELS  = True     # runs TEST+VALIDATION for best-per-model-type (no retraining)
RUN_LEARNING_CURVES   = True     # trains best-config per model_type on fractions of patients
RUN_PCA_SENSITIVITY   = False    # <-- DISABLED (NO PCA)
RUN_BOOTSTRAP_CI_EVAL = True     # patient-level bootstrap CI for RMSE/MAE on TEST/VAL preds

# For learning curves, you probably don't want to redo all 5 models.
LEARNING_MODELS = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]    # or ["Mamba"]

LEARNING_FRACTIONS = [0.2, 0.4, 0.6, 0.8, 1.0]
LEARNING_SEEDS     = [0]  # repeats per fraction

# Bootstrap settings
BOOTSTRAP_B = 2000

# ======================================================================================
# ================================ PATHS / I/O ========================================
# ======================================================================================

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Training data
xlsx_train  = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1V2_df.xlsx"
sheet_train = "Sheet5"

# Temporal validation data
xlsx_val  = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"
sheet_val = "Sheet15"

# Output roots
out_root_models = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels"
save_root = os.path.join(out_root_models, "SavedModels_OSIandVentALL_8")  # NEW folder
os.makedirs(save_root, exist_ok=True)

out_revision_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/20251105_TBME_Submission/20260301_Major_Revision"
os.makedirs(out_revision_dir, exist_ok=True)

# ======================================================================================
# ================================= DATA LOADING ======================================
# ======================================================================================

target_col = "OSI_V2_12th_avg"

# ---------------------------
# OSI + VentALL feature set per TW (TW1..TW6)
# ---------------------------
base_template = [
    "OSI_mean_TW{}", "OSI_std_TW{}", "PIP_mean_TW{}", "PIP_std_TW{}",
    "PEEP_mean_TW{}", "PEEP_std_TW{}", "TV_mean_TW{}(mL/Kg)", "TV_std_TW{}(mL/Kg)",
    "Avg_NegFlowDur_TW{}", "Std_NegFlowDur_TW{}", "Avg_PeakInterval_TW{}", "Std_PeakInterval_TW{}"
]

tw_features = {
    i: [f.format(i) for f in base_template] + [f"w{i}_SubBandEnergy_row{j}" for j in range(1, 17)]
    for i in range(1, 7)
}

def build_sequences_from_df(
    df: pd.DataFrame,
    label_col: str,
    tw_features: dict
):
    """
    Returns:
      X6: (N,6,D_perTW)
      y:  (N,)
      groups: (N,)
    """
    # Pre-check required columns
    required_cols = [label_col]
    for tw in range(1, 7):
        required_cols += list(tw_features[tw])

    missing_global = [c for c in required_cols if c not in df.columns]
    if missing_global:
        raise ValueError(f"Missing {len(missing_global)} columns in dataframe. First 30: {missing_global[:30]}")

    # Drop rows with any missing in required columns
    df2 = df.dropna(subset=required_cols).copy()
    if len(df2) == 0:
        raise ValueError("No usable rows after dropna on required columns.")

    X_list = [df2[tw_features[tw]].to_numpy(dtype=np.float32) for tw in range(1, 7)]
    X6 = np.stack(X_list, axis=1)  # (N,6,D)

    y = df2[label_col].to_numpy(dtype=np.float32)
    groups = df2["ResearchID"].to_numpy() if "ResearchID" in df2.columns else np.arange(len(df2))

    return X6.astype(np.float32), y.astype(np.float32), np.asarray(groups)

def add_delta_window(X6: np.ndarray) -> np.ndarray:
    # X6: (N, 6, D)
    delta = (X6[:, -1, :] - X6[:, 0, :])[:, np.newaxis, :]  # (N,1,D)
    return np.concatenate([X6, delta], axis=1).astype(np.float32)  # (N,7,D)

# Load train dataframe
df = pd.read_excel(xlsx_train, sheet_name=sheet_train)

X6, y, groups = build_sequences_from_df(df, target_col, tw_features)
X = add_delta_window(X6)  # (N,7,D_total)

print(f"✅ Total usable samples: {len(X)}")
print(f"✅ Input shape: {X.shape} (N,T,D_total)")
print(f"✅ D_perTW (VentALL): {X.shape[2]}")

# ======================================================================================
# ============================= PREPROCESSING HELPERS =================================
# ======================================================================================

def fit_scaler_on_train_X(X_train: np.ndarray) -> StandardScaler:
    scaler = StandardScaler()
    N, T, D = X_train.shape
    scaler.fit(X_train.reshape(-1, D))
    return scaler

def apply_scaler_X(scaler: StandardScaler, X_in: np.ndarray) -> np.ndarray:
    N, T, D = X_in.shape
    return scaler.transform(X_in.reshape(-1, D)).reshape(N, T, D).astype(np.float32)

def fit_scaler_on_train_y(y_train: np.ndarray) -> StandardScaler:
    scaler = StandardScaler()
    scaler.fit(y_train.reshape(-1, 1))
    return scaler

def apply_scaler_y(scaler: StandardScaler, y_in: np.ndarray) -> np.ndarray:
    return scaler.transform(y_in.reshape(-1, 1)).flatten().astype(np.float32)

def invert_scaler_y(scaler: StandardScaler, y_scaled: np.ndarray) -> np.ndarray:
    return scaler.inverse_transform(y_scaled.reshape(-1, 1)).flatten().astype(np.float32)

def augment_data(X_in, y_in, groups_in, noise_std=0.005, seed=0):
    rng = np.random.RandomState(seed)
    noise = rng.normal(0, noise_std, X_in.shape).astype(np.float32)
    return (
        np.concatenate([X_in, X_in + noise], axis=0),
        np.concatenate([y_in, y_in], axis=0),
        np.concatenate([groups_in, groups_in], axis=0)
    )

# ======================================================================================
# ================================== MODELS ===========================================
# ======================================================================================

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        w = torch.softmax(self.attn(x), dim=1)  # (B,T,1)
        return torch.sum(w * x, dim=1)          # (B,H)

class RNNRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, rnn_type="RNN", num_layers=1, dropout=0.0):
        super().__init__()
        if num_layers == 1:
            dropout = 0.0
        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[rnn_type]
        self.rnn = rnn_cls(input_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.attn(out)
        return self.fc(out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, nhead, dropout):
        super().__init__()
        enc = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=nhead, dropout=dropout,
            batch_first=True, layer_norm_eps=1e-5
        )
        self.encoder = nn.TransformerEncoder(enc, num_layers=1)

    def forward(self, x):
        return self.encoder(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0, nhead=None):
        super().__init__()
        if nhead is None:
            if hidden_dim % 8 == 0:
                nhead = max(1, hidden_dim // 8)
            elif hidden_dim % 4 == 0:
                nhead = max(1, hidden_dim // 4)
            else:
                nhead = 1

        self.input_proj = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim))
        self.pe = PositionalEncoding(hidden_dim)

        if num_layers == 1:
            self.stack = nn.Sequential(TransformerBlock(hidden_dim, nhead, 0.0))
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(TransformerBlock(hidden_dim, nhead, dropout), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])

        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pe(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

class MambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba = Mamba(d_model=dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return self.norm(self.mamba(x))

class MambaRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        if num_layers == 1:
            self.stack = nn.Sequential(MambaBlock(hidden_dim))
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(MambaBlock(hidden_dim), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.norm(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

def build_model(model_type, input_dim, hidden_dim, num_layers, dropout):
    if model_type == "Mamba":
        return MambaRegressor(input_dim, hidden_dim, num_layers, dropout)
    if model_type == "Transformer":
        return TransformerModel(input_dim, hidden_dim, num_layers, dropout)
    return RNNRegressor(input_dim, hidden_dim, model_type, num_layers, dropout)

def get_loader(X_in, y_in, batch_size, shuffle=True):
    X_tensor = torch.tensor(X_in, dtype=torch.float32)
    y_tensor = torch.tensor(y_in, dtype=torch.float32).view(-1, 1)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=shuffle)

# ======================================================================================
# ============================== REGRESSION METRICS ===================================
# ======================================================================================

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))

def eval_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {"RMSE": rmse(y_true, y_pred), "MAE": mae(y_true, y_pred), "N": int(len(y_true))}

def bootstrap_ci_regression_grouped(y_true, y_pred, groups, B=2000, seed=0):
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    groups = np.asarray(groups)

    uniq = np.unique(groups)
    point = eval_regression_metrics(y_true, y_pred)

    dist_rmse, dist_mae = [], []
    for _ in range(B):
        sampled_groups = rng.choice(uniq, size=len(uniq), replace=True)
        mask = np.isin(groups, sampled_groups)
        yt = y_true[mask]
        yp = y_pred[mask]
        if len(yt) < 2:
            continue
        dist_rmse.append(rmse(yt, yp))
        dist_mae.append(mae(yt, yp))

    def ci(arr):
        if len(arr) == 0:
            return (np.nan, np.nan)
        return (float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5)))

    rmse_lo, rmse_hi = ci(dist_rmse)
    mae_lo, mae_hi = ci(dist_mae)

    return pd.DataFrame([
        {"metric": "RMSE", "point": point["RMSE"], "ci_low": rmse_lo, "ci_high": rmse_hi, "n_boot": len(dist_rmse)},
        {"metric": "MAE",  "point": point["MAE"],  "ci_low": mae_lo,  "ci_high": mae_hi,  "n_boot": len(dist_mae)},
    ])

# ======================================================================================
# ================================ TRAIN/TEST SPLIT ===================================
# ======================================================================================

X_train_raw, X_test_raw, y_train_raw, y_test_raw, groups_train, groups_test = train_test_split(
    X, y, groups, test_size=0.2, random_state=42
)
print(f"Train N={len(X_train_raw)} | Test N={len(X_test_raw)}")

scaler_x = fit_scaler_on_train_X(X_train_raw)
scaler_y = fit_scaler_on_train_y(y_train_raw)

X_train = apply_scaler_X(scaler_x, X_train_raw)
X_test  = apply_scaler_X(scaler_x, X_test_raw)

y_train = apply_scaler_y(scaler_y, y_train_raw)
y_test  = apply_scaler_y(scaler_y, y_test_raw)

X_train_aug, y_train_aug, groups_train_aug = augment_data(X_train, y_train, groups_train, seed=0)
print(f"Train after augmentation N={len(X_train_aug)}")

# ======================================================================================
# ============================== GRID SEARCH TRAINING =================================
# ======================================================================================

def train_one_config_cv_regression(
    model_type, X_train_in, y_train_in, groups_train_in,
    hidden_dim, dropout, num_layers, batch_size, lr, wd, optimizer_name,
    epochs=200, n_splits=5, early_stop_patience=10,
    collect_cv_preds=True,
    scaler_y_for_inverse=None
):
    gkf = GroupKFold(n_splits=n_splits)

    rmse_list, mae_list = [], []
    used_epochs_all = []

    combo_best_val_loss = np.inf
    combo_best_state = None

    cv_pred_records = []

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train_in, y_train_in, groups_train_in)):

        model = build_model(model_type, X_train_in.shape[2], hidden_dim, num_layers, dropout).to(device)

        opt_cls = torch.optim.Adam if optimizer_name == "Adam" else torch.optim.SGD
        optimizer = opt_cls(model.parameters(), lr=lr, weight_decay=wd)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5)
        criterion = nn.MSELoss()

        train_loader = get_loader(X_train_in[tr_idx], y_train_in[tr_idx], batch_size=batch_size, shuffle=True)

        val_x_t = torch.tensor(X_train_in[val_idx], dtype=torch.float32).to(device)
        val_y_t = torch.tensor(y_train_in[val_idx], dtype=torch.float32).view(-1, 1).to(device)

        best_val_loss = float("inf")
        wait = 0
        used_epochs = 0
        best_state_fold = None

        for _ in range(epochs):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                pred = model(xb)
                loss = criterion(pred, yb)
                if torch.isnan(loss) or torch.isinf(loss):
                    continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            model.eval()
            with torch.no_grad():
                val_pred = model(val_x_t)
                val_loss = float(criterion(val_pred, val_y_t).item())

            scheduler.step(val_loss)
            used_epochs += 1

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                wait = 0
                best_state_fold = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                wait += 1
                if wait >= early_stop_patience:
                    break

        used_epochs_all.append(used_epochs)

        if best_state_fold is not None:
            model.load_state_dict(best_state_fold)

        model.eval()
        with torch.no_grad():
            preds_scaled = model(val_x_t).squeeze().detach().cpu().numpy()

        yt_scaled = y_train_in[val_idx].reshape(-1)

        if scaler_y_for_inverse is None:
            raise ValueError("scaler_y_for_inverse is required to compute RMSE/MAE in original domain.")
        y_pred = invert_scaler_y(scaler_y_for_inverse, preds_scaled)
        y_true = invert_scaler_y(scaler_y_for_inverse, yt_scaled)

        rmse_list.append(rmse(y_true, y_pred))
        mae_list.append(mae(y_true, y_pred))

        if collect_cv_preds:
            cv_pred_records.append({
                "model": model_type,
                "hidden_dim": int(hidden_dim),
                "dropout": float(dropout),
                "num_layers": int(num_layers),
                "batch_size": int(batch_size),
                "lr": float(lr),
                "weight_decay": float(wd),
                "optimizer": optimizer_name,
                "fold": int(fold),
                "y_true": y_true.tolist(),
                "y_pred": y_pred.tolist()
            })

        if best_val_loss < combo_best_val_loss and best_state_fold is not None:
            combo_best_val_loss = best_val_loss
            combo_best_state = best_state_fold

    summary = {
        "RMSE_mean": float(np.mean(rmse_list)) if len(rmse_list) else np.inf,
        "RMSE_std":  float(np.std(rmse_list))  if len(rmse_list) else np.nan,
        "MAE_mean":  float(np.mean(mae_list))  if len(mae_list)  else np.inf,
        "MAE_std":   float(np.std(mae_list))   if len(mae_list)  else np.nan,
        "n_folds_done": int(len(rmse_list)),
        "expected_folds": int(n_splits),
        "used_epochs_mean": float(np.mean(used_epochs_all)) if used_epochs_all else np.nan,
        "used_epochs_std":  float(np.std(used_epochs_all))  if used_epochs_all else np.nan,
        "combo_best_val_loss": float(combo_best_val_loss),
        "has_state": combo_best_state is not None
    }
    return summary, combo_best_state, cv_pred_records

# ======================================================================================
# ============================ GRID SEARCH + SAVE BEST PER MODEL =======================
# ======================================================================================

model_types = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]

best_per_model = {m: {"rmse": np.inf, "cfg": None, "state": None} for m in model_types}
grid_summary_rows = []
cv_predictions_all = []

if RUN_GRID_SEARCH:
    epochs = 200
    n_splits = 5

    for model_type in model_types:
        for hidden_dim in [16, 32, 64, 128]:
            for dropout in [0.0, 0.2]:
                for num_layers in [1, 2, 3]:
                    for batch_size in [16, 32]:
                        for lr in [0.001, 0.01]:
                            for wd in [0, 1e-4]:
                                for optimizer_name in ["Adam", "SGD"]:

                                    summary, best_state, cv_pred_records = train_one_config_cv_regression(
                                        model_type,
                                        X_train_aug, y_train_aug, groups_train_aug,
                                        hidden_dim, dropout, num_layers, batch_size, lr, wd, optimizer_name,
                                        epochs=epochs, n_splits=n_splits, early_stop_patience=10,
                                        collect_cv_preds=True,
                                        scaler_y_for_inverse=scaler_y
                                    )

                                    if cv_pred_records:
                                        cv_predictions_all.extend(cv_pred_records)

                                    row = {
                                        "model": model_type,
                                        "hidden_dim": hidden_dim,
                                        "dropout": dropout,
                                        "num_layers": num_layers,
                                        "batch_size": batch_size,
                                        "lr": lr,
                                        "weight_decay": wd,
                                        "optimizer": optimizer_name,
                                        "epochs": epochs,
                                        "folds": n_splits,
                                        **summary
                                    }
                                    grid_summary_rows.append(row)

                                    rmse_mean = row["RMSE_mean"]
                                    if np.isfinite(rmse_mean) and rmse_mean < best_per_model[model_type]["rmse"] and summary["has_state"]:
                                        best_per_model[model_type]["rmse"] = float(rmse_mean)
                                        best_per_model[model_type]["cfg"] = {
                                            "model": model_type,
                                            "hidden_dim": int(hidden_dim),
                                            "dropout": float(dropout),
                                            "num_layers": int(num_layers),
                                            "batch_size": int(batch_size),
                                            "lr": float(lr),
                                            "weight_decay": float(wd),
                                            "optimizer": optimizer_name,
                                            "epochs": int(epochs),
                                            "folds": int(n_splits),
                                            "target_col": target_col,
                                            "timesteps": int(X_train_aug.shape[1]),
                                            "input_dim": int(X_train_aug.shape[2]),
                                            "feature_repr": "OSI+VentALL",
                                            "pca_components": None,
                                            "pca_mode": None
                                        }
                                        best_per_model[model_type]["state"] = best_state
                                        print(f"\n🏆 BEST for {model_type}: RMSE_mean={rmse_mean:.4f}")

        bm = best_per_model[model_type]
        if bm["cfg"] is not None:
            model_dir = os.path.join(save_root, f"best_{model_type}")
            os.makedirs(model_dir, exist_ok=True)

            torch.save({"state_dict": bm["state"], "config": bm["cfg"]}, os.path.join(model_dir, "best_model_state.pt"))
            with open(os.path.join(model_dir, "best_model_config.json"), "w") as f:
                json.dump(bm["cfg"], f, indent=2)

            joblib.dump(scaler_x, os.path.join(model_dir, "scaler_x.joblib"))
            joblib.dump(scaler_y, os.path.join(model_dir, "scaler_y.joblib"))

            print(f"✅ Saved best artifacts for {model_type} -> {model_dir}")

    df_grid = pd.DataFrame(grid_summary_rows)
    grid_csv_out = os.path.join(out_root_models, "Regression_grid_search_results(OSIandVentALL)_8.csv")
    df_grid.to_csv(grid_csv_out, index=False)
    print("✅ Grid search summary saved:", grid_csv_out)

    cv_json_out = os.path.join(out_root_models, "Regression_cv_predictions(OSIandVentALL)_8.json")
    with open(cv_json_out, "w") as f:
        json.dump(cv_predictions_all, f)
    print(f"✅ CV predictions saved: {len(cv_predictions_all)} records -> {cv_json_out}")

# ======================================================================================
# ============================= LOAD BEST-PER-MODEL ARTIFACTS ==========================
# ======================================================================================

def load_best_for_model(model_type: str):
    model_dir = os.path.join(save_root, f"best_{model_type}")
    ckpt = os.path.join(model_dir, "best_model_state.pt")
    cfgp = os.path.join(model_dir, "best_model_config.json")
    sxp  = os.path.join(model_dir, "scaler_x.joblib")
    syp  = os.path.join(model_dir, "scaler_y.joblib")

    if not (os.path.exists(ckpt) and os.path.exists(cfgp) and os.path.exists(sxp) and os.path.exists(syp)):
        raise FileNotFoundError(f"Missing best artifacts for {model_type} in {model_dir}")

    with open(cfgp, "r") as f:
        cfg = json.load(f)

    sx = joblib.load(sxp)
    sy = joblib.load(syp)
    state = torch.load(ckpt, map_location="cpu")

    return cfg, sx, sy, state["state_dict"]

# ======================================================================================
# =============================== VALIDATION BUILD (V3) ================================
# ======================================================================================

def load_validation_set(scaler_train_x: StandardScaler, scaler_train_y: StandardScaler):
    df_val = pd.read_excel(xlsx_val, sheet_name=sheet_val)
    train_target_col = target_col
    val_target_col = "OSI_V3_12th_avg"

    if train_target_col in df_val.columns:
        label_col = train_target_col
    elif val_target_col in df_val.columns:
        print(f"NOTE: '{train_target_col}' not found in {sheet_val}; using '{val_target_col}' instead.")
        label_col = val_target_col
    else:
        raise ValueError(f"Missing target in {sheet_val}: neither {train_target_col} nor {val_target_col}")

    all_cols = [c for tw in range(1, 7) for c in tw_features[tw]]
    missing = [c for c in all_cols if c not in df_val.columns]
    if missing:
        raise ValueError(f"Missing {len(missing)} VentALL feature cols in {sheet_val}. First 20: {missing[:20]}")

    df_val_model = df_val.dropna(subset=[label_col] + all_cols).copy()
    if len(df_val_model) == 0:
        raise ValueError("No validation rows left after dropna.")

    X_list = [df_val_model[tw_features[tw]].to_numpy(dtype=np.float32) for tw in range(1, 7)]
    X6_val = np.stack(X_list, axis=1)  # (N,6,D)
    X7_val = add_delta_window(X6_val)  # (N,7,D)

    y_val_raw = df_val_model[label_col].to_numpy(dtype=np.float32)

    X7_val_scaled = apply_scaler_X(scaler_train_x, X7_val)
    y_val_scaled  = apply_scaler_y(scaler_train_y, y_val_raw)

    g_val = df_val_model["ResearchID"].to_numpy() if "ResearchID" in df_val_model.columns else np.arange(len(df_val_model))
    return X7_val_scaled, y_val_scaled, y_val_raw, g_val, label_col

# ======================================================================================
# =============================== EVALUATION: TEST + VAL ===============================
# ======================================================================================

def predict_scaled(model: nn.Module, X_in: np.ndarray, batch_size=32):
    X_tensor = torch.tensor(X_in, dtype=torch.float32)
    loader = DataLoader(TensorDataset(X_tensor), batch_size=batch_size, shuffle=False)
    preds = []
    model.eval()
    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            yhat = model(xb).detach().cpu().numpy().reshape(-1)
            preds.append(yhat)
    return np.concatenate(preds, axis=0)

def evaluate_and_save_best_per_model():
    test_payloads = []
    val_payloads  = []
    metrics_rows_test = []
    metrics_rows_val  = []

    y_test_raw_local = invert_scaler_y(scaler_y, y_test)

    for m in model_types:
        cfg, sx, sy, state_dict = load_best_for_model(m)

        model = build_model(
            cfg["model"],
            input_dim=int(cfg["input_dim"]),
            hidden_dim=int(cfg["hidden_dim"]),
            num_layers=int(cfg["num_layers"]),
            dropout=float(cfg["dropout"])
        ).to(device)
        model.load_state_dict(state_dict)
        model.eval()

        bs = int(cfg.get("batch_size", 32))

        y_pred_test_scaled = predict_scaled(model, X_test, batch_size=bs)
        y_pred_test = invert_scaler_y(sy, y_pred_test_scaled)
        y_true_test = y_test_raw_local

        test_payloads.append({
            "model": m,
            "config": cfg,
            "y_true": y_true_test.tolist(),
            "y_pred": y_pred_test.tolist(),
            "groups": groups_test.tolist()
        })
        metrics_rows_test.append({"model": m, **eval_regression_metrics(y_true_test, y_pred_test)})

        X_val_scaled, y_val_scaled, y_val_raw, g_val, label_col_used = load_validation_set(sx, sy)

        y_pred_val_scaled = predict_scaled(model, X_val_scaled, batch_size=bs)
        y_pred_val = invert_scaler_y(sy, y_pred_val_scaled)
        y_true_val = y_val_raw

        val_payloads.append({
            "model": m,
            "config": cfg,
            "label_col_used": label_col_used,
            "y_true": y_true_val.tolist(),
            "y_pred": y_pred_val.tolist(),
            "groups": g_val.tolist()
        })
        metrics_rows_val.append({"model": m, **eval_regression_metrics(y_true_val, y_pred_val), "label_col_used": label_col_used})

    test_preds_json = os.path.join(out_root_models, "Regression_best_model_test_predictions(OSIandVentALL)_8.json")
    val_preds_json  = os.path.join(out_revision_dir, "Regression_best_model_validation_predictions(OSIandVentALL)_8.json")

    with open(test_preds_json, "w") as f:
        json.dump(test_payloads, f, indent=2)
    with open(val_preds_json, "w") as f:
        json.dump(val_payloads, f, indent=2)

    pd.DataFrame(metrics_rows_test).to_csv(
        os.path.join(out_root_models, "Regression_best_models_TEST_metrics_point(OSIandVentALL)_8.csv"),
        index=False
    )
    pd.DataFrame(metrics_rows_val).to_csv(
        os.path.join(out_revision_dir, "Regression_best_models_VALIDATION_metrics_point(OSIandVentALL)_8.csv"),
        index=False
    )

    print("✅ Saved ALL5 TEST preds:", test_preds_json)
    print("✅ Saved ALL5 VAL preds:",  val_preds_json)
    return test_preds_json, val_preds_json

# ======================================================================================
# ============================ BOOTSTRAP CI (GROUPED) =================================
# ======================================================================================

def run_bootstrap_ci(preds_json_path: str, split_name: str, out_dir: str):
    with open(preds_json_path, "r") as f:
        payloads = json.load(f)

    all_rows = []
    for item in payloads:
        model_name = item["model"]
        y_true = np.array(item["y_true"], dtype=float)
        y_pred = np.array(item["y_pred"], dtype=float)
        groups = np.array(item.get("groups", np.arange(len(y_true))), dtype=object)

        ci_df = bootstrap_ci_regression_grouped(y_true, y_pred, groups, B=BOOTSTRAP_B, seed=0)
        ci_df.insert(0, "split", split_name)
        ci_df.insert(1, "model", model_name)
        all_rows.append(ci_df)

    out = pd.concat(all_rows, ignore_index=True)
    out_csv = os.path.join(out_dir, f"BootstrapCI_{split_name}_OSIandVentALL_8.csv")
    out.to_csv(out_csv, index=False)
    print(f"✅ Bootstrap CI saved: {out_csv}")

# ======================================================================================
# ================================ LEARNING CURVES =====================================
# ======================================================================================

def train_single_model_once_regression(
    model_type: str,
    cfg: dict,
    X_train_in: np.ndarray,
    y_train_in: np.ndarray,      # scaled y
    groups_train_in: np.ndarray,
    X_test_in: np.ndarray,
    y_test_raw: np.ndarray,      # raw y
    X_val_in: np.ndarray,
    y_val_raw: np.ndarray,       # raw y
    sy: StandardScaler,          # y scaler for inverse
    seed: int = 0
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = build_model(
        model_type,
        input_dim=X_train_in.shape[2],
        hidden_dim=int(cfg["hidden_dim"]),
        num_layers=int(cfg["num_layers"]),
        dropout=float(cfg["dropout"])
    ).to(device)

    optimizer_name = cfg["optimizer"]
    lr = float(cfg["lr"])
    wd = float(cfg["weight_decay"])
    batch_size = int(cfg["batch_size"])
    epochs = int(cfg["epochs"])

    opt_cls = torch.optim.Adam if optimizer_name == "Adam" else torch.optim.SGD
    optimizer = opt_cls(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5)
    criterion = nn.MSELoss()

    loader = get_loader(X_train_in, y_train_in, batch_size=batch_size, shuffle=True)

    best_proxy = float("inf")
    wait = 0
    best_state = None

    for _ in range(epochs):
        model.train()
        losses = []
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            if torch.isnan(loss) or torch.isinf(loss):
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            losses.append(loss.item())

        avg_loss = float(np.mean(losses)) if len(losses) else np.inf
        scheduler.step(avg_loss)

        if avg_loss < best_proxy:
            best_proxy = avg_loss
            wait = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= 10:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    y_pred_test_scaled = predict_scaled(model, X_test_in, batch_size=batch_size)
    y_pred_test = invert_scaler_y(sy, y_pred_test_scaled)
    m_test = eval_regression_metrics(y_test_raw, y_pred_test)

    y_pred_val_scaled = predict_scaled(model, X_val_in, batch_size=batch_size)
    y_pred_val = invert_scaler_y(sy, y_pred_val_scaled)
    m_val = eval_regression_metrics(y_val_raw, y_pred_val)

    return m_test, m_val

def run_learning_curves():
    X_val_scaled, y_val_scaled, y_val_raw, g_val, label_col_used = load_validation_set(scaler_x, scaler_y)
    y_test_raw_local = invert_scaler_y(scaler_y, y_test)

    rows = []
    unique_groups = np.unique(groups_train)

    for model_type in LEARNING_MODELS:
        cfg, _, sy, _ = load_best_for_model(model_type)

        for frac in LEARNING_FRACTIONS:
            n_groups = max(2, int(len(unique_groups) * frac))
            for seed in LEARNING_SEEDS:
                rng = np.random.RandomState(seed)
                chosen_groups = rng.choice(unique_groups, size=n_groups, replace=False)
                mask = np.isin(groups_train, chosen_groups)

                X_sub = X_train[mask]
                y_sub = y_train[mask]
                g_sub = groups_train[mask]

                X_sub_aug, y_sub_aug, g_sub_aug = augment_data(X_sub, y_sub, g_sub, seed=seed)

                m_test, m_val = train_single_model_once_regression(
                    model_type, cfg,
                    X_sub_aug, y_sub_aug, g_sub_aug,
                    X_test, y_test_raw_local,
                    X_val_scaled, y_val_raw,
                    sy=sy,
                    seed=seed
                )

                rows.append({
                    "model": model_type,
                    "fraction": frac,
                    "seed": seed,
                    "label_col_val": label_col_used,
                    **{f"test_{k}": v for k, v in m_test.items()},
                    **{f"val_{k}": v for k, v in m_val.items()},
                    "n_train_patients": int(n_groups),
                    "n_train_samples": int(len(X_sub))
                })

                print(f"✅ LearningCurve {model_type} frac={frac} seed={seed} "
                      f"test_RMSE={m_test['RMSE']:.3f} val_RMSE={m_val['RMSE']:.3f}")

    df_lc = pd.DataFrame(rows)
    lc_csv = os.path.join(out_revision_dir, "LearningCurve_OSIandVentALL_8.csv")
    df_lc.to_csv(lc_csv, index=False)
    print("✅ Learning curve results saved:", lc_csv)

# ======================================================================================
# ===================================== RUN ===========================================
# ======================================================================================

test_preds_json = None
val_preds_json = None

if RUN_EVAL_BEST_MODELS:
    test_preds_json, val_preds_json = evaluate_and_save_best_per_model()

if RUN_BOOTSTRAP_CI_EVAL:
    if test_preds_json is None or val_preds_json is None:
        test_preds_json = os.path.join(out_root_models, "Regression_best_model_test_predictions(OSIandVentALL)_8.json")
        val_preds_json  = os.path.join(out_revision_dir, "Regression_best_model_validation_predictions(OSIandVentALL)_8.json")
    run_bootstrap_ci(test_preds_json, "TEST", out_root_models)
    run_bootstrap_ci(val_preds_json, "VALIDATION", out_revision_dir)

if RUN_LEARNING_CURVES:
    run_learning_curves()

print("✅ All done.")

### OSI+Vent_All Features Only (Regression) -- training + test + validation (_7)

In [ ]:
# === Try more (Regression) WITH BEST-MODEL WEIGHT + SCALER SAVING + TEST + EXTERNAL VALIDATION ===
# Updates applied:
#  1) Early stopping + scheduler now use **VALIDATION loss** (not training loss)
#  2) Bestå weights per fold chosen by **lowest val loss**
#  3) Best weights per hyperparam combo chosen by **lowest val loss**
#  4) External validation block enabled AND handles V3 target name:
#        uses OSI_V3_12th_avg if OSI_V2_12th_avg is missing
#  5) Adds sanity checks: feature dimension matches training input_dim; timestep matches 7
#  6) Keeps your file structure/output style the same

import os
import math
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from mamba_ssm import Mamba


# =============================
# Device
# =============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# =============================
# Load Data (TRAINING DATA)
# =============================
df = pd.read_excel(
    "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1V2_df.xlsx",
    sheet_name="Sheet5"
)

target = "OSI_V2_12th_avg"

base_template = [
    "OSI_mean_TW{}", "OSI_std_TW{}", "PIP_mean_TW{}", "PIP_std_TW{}",
    "PEEP_mean_TW{}", "PEEP_std_TW{}", "TV_mean_TW{}(mL/Kg)", "TV_std_TW{}(mL/Kg)",
    "Avg_NegFlowDur_TW{}", "Std_NegFlowDur_TW{}", "Avg_PeakInterval_TW{}", "Std_PeakInterval_TW{}"
]

tw_features = {
    i: [f.format(i) for f in base_template] + [f"w{i}_SubBandEnergy_row{j}" for j in range(1, 17)]
    for i in range(1, 7)
}

# Build sequences
X, y, groups = [], [], []
for idx, row in df.iterrows():
    seq = []
    for tw in range(1, 7):
        cols = tw_features[tw]
        if row[cols].isnull().any():
            break
        seq.append(row[cols].values)
    if len(seq) == 6 and not pd.isna(row[target]):
        X.append(seq)
        y.append(row[target])
        groups.append(row["ResearchID"] if "ResearchID" in df.columns else idx)

X = np.array(X, dtype=np.float32)   # (N,6,D)
y = np.array(y, dtype=np.float32)   # (N,)
groups = np.array(groups)


# =============================
# Feature Engineering: add delta timestep (7th step)
# =============================
delta_features = (X[:, -1, :] - X[:, 0, :])[:, np.newaxis, :]
X = np.concatenate([X, delta_features], axis=1)  # (N,7,D)


# =============================
# Scaling (fit on all data here, as in your code)
# =============================
scaler_x = StandardScaler()
X_scaled = scaler_x.fit_transform(X.reshape(-1, X.shape[2])).reshape(X.shape)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()


# =============================
# Train/Test Split (ONLY ONCE, keep for later test)
# =============================
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X_scaled, y_scaled, groups, test_size=0.2, random_state=42
)

# =============================
# Data Augmentation (train only)
# =============================
def augment_data(Xa, ya, ga):
    noise = np.random.normal(0, 0.005, Xa.shape).astype(np.float32)
    return (
        np.concatenate([Xa, Xa + noise]),
        np.concatenate([ya, ya]),
        np.concatenate([ga, ga])
    )

X_train, y_train, groups_train = augment_data(X_train, y_train, groups_train)


# =============================
# Model Definitions
# =============================
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        w = torch.softmax(self.attn(x), dim=1)
        return torch.sum(w * x, dim=1)

class RNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, rnn_type="RNN", num_layers=1, dropout=0.0):
        super().__init__()
        if num_layers == 1:
            dropout = 0.0
        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[rnn_type]
        self.rnn = rnn_cls(input_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.attn(out)
        return self.fc(out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, nhead, dropout):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=nhead, dropout=dropout,
            batch_first=True, layer_norm_eps=1e-5
        )
        self.enc = nn.TransformerEncoder(layer, num_layers=1)
    def forward(self, x):
        return self.enc(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0, nhead=None):
        super().__init__()
        if nhead is None:
            if hidden_dim % 8 == 0:
                nhead = hidden_dim // 8
            elif hidden_dim % 4 == 0:
                nhead = hidden_dim // 4
            else:
                nhead = 1

        self.input_proj = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim))
        self.pe = PositionalEncoding(hidden_dim)

        if num_layers == 1:
            self.stack = nn.Sequential(TransformerBlock(hidden_dim, nhead, 0.0))
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(TransformerBlock(hidden_dim, nhead, dropout), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])

        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pe(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

class MambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba = Mamba(d_model=dim)
        self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        return self.norm(self.mamba(x))

class MambaModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        if num_layers == 1:
            self.stack = nn.Sequential(*[MambaBlock(hidden_dim) for _ in range(num_layers)])
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(MambaBlock(hidden_dim), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)
    def forward(self, x):
        x = self.input_proj(x)
        x = self.norm(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)


# =============================
# Utility
# =============================
def get_loader(Xa, ya, batch_size, shuffle=True):
    Xt = torch.tensor(Xa, dtype=torch.float32)
    yt = torch.tensor(ya, dtype=torch.float32).view(-1, 1)
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=shuffle)


# =============================
# Save root for best artifacts
# =============================
save_root = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/SavedModels_Stats_7"
os.makedirs(save_root, exist_ok=True)

global_best_rmse = np.inf
global_best_info = None
global_best_state_dict = None


# =============================
# Grid search + best save
# =============================
def run_grid_search():
    models = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]  # remove "Mamba" if env issues
    hidden_dims = [16, 32, 64, 128]
    dropouts = [0.0, 0.2]
    num_layers_list = [1, 2, 3]
    batch_sizes = [16, 32]
    lrs = [0.001, 0.01]
    weight_decays = [0, 1e-4]
    optimizers = ["Adam", "SGD"]
    epochs = 200

    results = {"cv_predictions": [], "summary": []}
    gkf = GroupKFold(n_splits=5)

    global global_best_rmse, global_best_info, global_best_state_dict

    for model_type in models:
        for hidden_dim in hidden_dims:
            for dropout in dropouts:
                for num_layers in num_layers_list:
                    for batch_size in batch_sizes:
                        for lr in lrs:
                            for wd in weight_decays:
                                for opt_name in optimizers:
                                    rmse_list, mae_list = [], []
                                    used_epochs_all = []

                                    # choose best weights for this combo by best VAL loss (avg MSE in scaled space)
                                    combo_best_val_loss = np.inf
                                    combo_best_state_dict = None

                                    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups_train)):
                                        # init model
                                        if model_type == "Mamba":
                                            model = MambaModel(X_train.shape[2], hidden_dim, num_layers, dropout).to(device)
                                        elif model_type == "Transformer":
                                            model = TransformerModel(X_train.shape[2], hidden_dim, num_layers, dropout).to(device)
                                        else:
                                            model = RNNModel(X_train.shape[2], hidden_dim, model_type, num_layers, dropout).to(device)

                                        opt_cls = torch.optim.Adam if opt_name == "Adam" else torch.optim.SGD
                                        optimizer = opt_cls(model.parameters(), lr=lr, weight_decay=wd)
                                        criterion = nn.MSELoss()

                                        train_loader = get_loader(X_train[train_idx], y_train[train_idx], batch_size, shuffle=True)

                                        # pre-make val tensors once
                                        val_x_t = torch.tensor(X_train[val_idx], dtype=torch.float32).to(device)
                                        val_y_t = torch.tensor(y_train[val_idx], dtype=torch.float32).view(-1, 1).to(device)

                                        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5)

                                        best_val_loss = float("inf")
                                        wait = 0
                                        used_epochs = 0
                                        best_state_dict_fold = None

                                        for epoch in range(epochs):
                                            model.train()
                                            for xb, yb in train_loader:
                                                xb, yb = xb.to(device), yb.to(device)
                                                optimizer.zero_grad()
                                                pred = model(xb)
                                                loss = criterion(pred, yb)
                                                if torch.isnan(loss) or torch.isinf(loss):
                                                    continue
                                                loss.backward()
                                                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                                                optimizer.step()

                                            # ---- VALIDATION loss used for early stop + scheduler ----
                                            model.eval()
                                            with torch.no_grad():
                                                val_pred = model(val_x_t)
                                                val_loss = criterion(val_pred, val_y_t).item()

                                            scheduler.step(val_loss)
                                            used_epochs += 1

                                            if val_loss < best_val_loss:
                                                best_val_loss = val_loss
                                                wait = 0
                                                best_state_dict_fold = {k: v.detach().cpu().clone()
                                                                       for k, v in model.state_dict().items()}
                                            else:
                                                wait += 1
                                                if wait >= 10:
                                                    break

                                        used_epochs_all.append(used_epochs)

                                        if best_state_dict_fold is not None:
                                            model.load_state_dict(best_state_dict_fold)

                                        # val predict -> inverse transform to original scale for metrics
                                        model.eval()
                                        with torch.no_grad():
                                            preds_scaled = model(val_x_t).squeeze().detach().cpu().numpy()

                                        preds = scaler_y.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
                                        y_true = scaler_y.inverse_transform(y_train[val_idx].reshape(-1, 1)).flatten()

                                        results["cv_predictions"].append({
                                            "model": model_type,
                                            "hidden_dim": hidden_dim,
                                            "dropout": dropout,
                                            "num_layers": num_layers,
                                            "batch_size": batch_size,
                                            "lr": lr,
                                            "weight_decay": wd,
                                            "optimizer": opt_name,
                                            "fold": fold,
                                            "y_true": y_true.tolist(),
                                            "y_pred": preds.tolist()
                                        })

                                        rmse = float(np.sqrt(mean_squared_error(y_true, preds)))
                                        mae = float(mean_absolute_error(y_true, preds))
                                        rmse_list.append(rmse)
                                        mae_list.append(mae)

                                        # track best fold weights for this hyperparam combo by best_val_loss
                                        if best_val_loss < combo_best_val_loss and best_state_dict_fold is not None:
                                            combo_best_val_loss = best_val_loss
                                            combo_best_state_dict = best_state_dict_fold

                                    rmse_mean = float(np.mean(rmse_list)) if rmse_list else float("inf")
                                    rmse_std  = float(np.std(rmse_list))  if rmse_list else float("nan")
                                    mae_mean  = float(np.mean(mae_list))  if mae_list else float("inf")
                                    mae_std   = float(np.std(mae_list))   if mae_list else float("nan")

                                    results["summary"].append({
                                        "model": model_type, "hidden_dim": hidden_dim, "dropout": dropout,
                                        "num_layers": num_layers, "batch_size": batch_size, "lr": lr,
                                        "weight_decay": wd, "optimizer": opt_name, "epochs": epochs,
                                        "rmse_mean": rmse_mean, "rmse_std": rmse_std,
                                        "mae_mean": mae_mean, "mae_std": mae_std,
                                        "used_epochs_mean": float(np.mean(used_epochs_all)) if used_epochs_all else float("nan"),
                                        "used_epochs_std": float(np.std(used_epochs_all))  if used_epochs_all else float("nan"),
                                        "combo_best_val_loss": float(combo_best_val_loss),
                                    })

                                    print(f"{model_type} HD={hidden_dim} DO={dropout} NL={num_layers} OPT={opt_name} "
                                          f"=> RMSE: {rmse_mean:.4f} ± {rmse_std:.4f}, MAE: {mae_mean:.4f} ± {mae_std:.4f}")

                                    # save global best by rmse_mean
                                    if rmse_mean < global_best_rmse and combo_best_state_dict is not None:
                                        global_best_rmse = rmse_mean
                                        global_best_info = {
                                            "model": model_type,
                                            "hidden_dim": int(hidden_dim),
                                            "dropout": float(dropout),
                                            "num_layers": int(num_layers),
                                            "batch_size": int(batch_size),
                                            "lr": float(lr),
                                            "weight_decay": float(wd),
                                            "optimizer": opt_name,
                                            "epochs": int(epochs),
                                            "rmse_mean": float(rmse_mean),
                                            "mae_mean": float(mae_mean),
                                            "input_dim": int(X_train.shape[2]),
                                            "timesteps": int(X_train.shape[1]),  # 7
                                            "target": target,
                                        }
                                        global_best_state_dict = combo_best_state_dict

                                        print("\n🏆 NEW GLOBAL BEST FOUND:", global_best_info)

                                        torch.save(
                                            {"state_dict": global_best_state_dict, "config": global_best_info},
                                            os.path.join(save_root, "best_model_state.pt")
                                        )
                                        with open(os.path.join(save_root, "best_model_config.json"), "w") as f:
                                            json.dump(global_best_info, f, indent=2)

                                        joblib.dump(scaler_x, os.path.join(save_root, "scaler_x.joblib"))
                                        joblib.dump(scaler_y, os.path.join(save_root, "scaler_y.joblib"))

    return pd.DataFrame(results["summary"]), results["cv_predictions"]


# =============================
# RUN TRAINING
# =============================
results_df, cv_predictions = run_grid_search()

results_df.to_csv(
    "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_grid_search_results(Stats)_7.csv",
    index=False
)
with open(
    "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_cv_predictions(Stats)_7.json",
    "w"
) as f:
    json.dump(cv_predictions, f)

print(f"✅ CV predictions saved: {len(cv_predictions)} records.")
print("✅ Best model + scalers saved to:", save_root)


# ======================================================================================
# AFTER TRAINING: TEST on existing X_test/y_test + EXTERNAL VALIDATION on V3 Sheet15 (NO retraining)
# ======================================================================================

def load_best_artifacts():
    ckpt_path = os.path.join(save_root, "best_model_state.pt")
    cfg_path  = os.path.join(save_root, "best_model_config.json")
    sx_path   = os.path.join(save_root, "scaler_x.joblib")
    sy_path   = os.path.join(save_root, "scaler_y.joblib")

    for p in [ckpt_path, cfg_path, sx_path, sy_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing best artifact: {p}")

    with open(cfg_path, "r") as f:
        cfg = json.load(f)

    sx = joblib.load(sx_path)
    sy = joblib.load(sy_path)

    state = torch.load(ckpt_path, map_location="cpu")
    state_dict = state["state_dict"] if isinstance(state, dict) and "state_dict" in state else state
    return cfg, sx, sy, state_dict


def build_model_from_cfg(cfg, input_dim):
    model_type = cfg["model"]
    hidden_dim = int(cfg["hidden_dim"])
    dropout = float(cfg["dropout"])
    num_layers = int(cfg["num_layers"])

    if model_type == "Mamba":
        m = MambaModel(input_dim, hidden_dim, num_layers, dropout)
    elif model_type == "Transformer":
        m = TransformerModel(input_dim, hidden_dim, num_layers, dropout)
    else:
        m = RNNModel(input_dim, hidden_dim, model_type, num_layers, dropout)
    return m


# --------------------------
# Load best saved model once
# --------------------------
cfg, sx, sy, best_state_dict = load_best_artifacts()

batch_size_best = int(cfg.get("batch_size", 32))
train_input_dim = int(cfg.get("input_dim", X_test.shape[2]))

best_model = build_model_from_cfg(cfg, train_input_dim)
best_model.load_state_dict(best_state_dict)
best_model = best_model.to(device).eval()


# --------------------------
# TEST (held-out X_test/y_test)
# --------------------------
# sanity: X_test dim should match model input_dim
if X_test.shape[2] != train_input_dim:
    raise ValueError(f"Input dim mismatch: X_test has D={X_test.shape[2]} but best model expects D={train_input_dim}")

test_loader = get_loader(X_test, y_test, batch_size_best, shuffle=False)

preds_scaled = []
with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(device)
        pred = best_model(xb).detach().cpu().numpy().reshape(-1)
        preds_scaled.append(pred)

y_pred_scaled = np.concatenate(preds_scaled)
y_pred_test = sy.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_true_test = sy.inverse_transform(y_test.reshape(-1, 1)).flatten()

rmse_test = float(np.sqrt(mean_squared_error(y_true_test, y_pred_test)))
mae_test = float(mean_absolute_error(y_true_test, y_pred_test))

print("\n✅ TEST (held-out split)")
print(f"RMSE: {rmse_test:.4f} | MAE: {mae_test:.4f} | N={len(y_true_test)}")

test_results_df = pd.DataFrame([{
    "model": cfg["model"],
    "rmse_test": rmse_test,
    "mae_test": mae_test,
    "N_test": int(len(y_true_test)),
    "hidden_dim": int(cfg["hidden_dim"]),
    "dropout": float(cfg["dropout"]),
    "num_layers": int(cfg["num_layers"]),
    "batch_size": int(cfg.get("batch_size", -1)),
    "lr": float(cfg.get("lr", np.nan)),
    "weight_decay": float(cfg.get("weight_decay", np.nan)),
    "optimizer": cfg.get("optimizer", "NA"),
    "epochs": int(cfg.get("epochs", -1)),
}])

test_results_csv = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_best_model_test_results(Stats)_7.csv"
test_preds_json  = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_best_model_test_predictions(Stats)_7.json"

test_results_df.to_csv(test_results_csv, index=False)
with open(test_preds_json, "w") as f:
    json.dump([{
        "model": cfg["model"],
        "y_true": y_true_test.tolist(),
        "y_pred": y_pred_test.tolist()
    }], f)

print("✅ Test performance saved:", test_results_csv)
print("✅ Test predictions saved:", test_preds_json)


# --------------------------
# EXTERNAL VALIDATION (V3 Sheet15)
# Fix: V3 may use OSI_V3_12th_avg instead of OSI_V2_12th_avg
# --------------------------
v3_xlsx = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"
v3_sheet = "Sheet15"

df_val = pd.read_excel(v3_xlsx, sheet_name=v3_sheet)

train_target = target              # "OSI_V2_12th_avg"
val_target   = "OSI_V3_12th_avg"   # likely in V3

if train_target in df_val.columns:
    label_col = train_target
elif val_target in df_val.columns:
    print(f"NOTE: '{train_target}' not found in {v3_sheet}; using '{val_target}' instead.")
    label_col = val_target
else:
    raise ValueError(f"Missing target in {v3_sheet}: neither '{train_target}' nor '{val_target}' exists.")

# same feature list as training
all_feature_cols = [c for tw in range(1, 7) for c in tw_features[tw]]
missing_feats = [c for c in all_feature_cols if c not in df_val.columns]
if missing_feats:
    raise ValueError(f"Missing {len(missing_feats)} feature columns in {v3_sheet}. First 30: {missing_feats[:30]}")

df_val_model = df_val.dropna(subset=[label_col] + all_feature_cols).copy()
print(f"\n✅ EXTERNAL VALIDATION rows after dropna ({v3_sheet}): {len(df_val_model)}")
if len(df_val_model) == 0:
    raise ValueError("No rows left after dropna in external validation; cannot run validation.")

# Build X6 then add delta => X7 (must match training timesteps=7)
X_list = [df_val_model[tw_features[tw]].to_numpy(dtype=np.float32) for tw in range(1, 7)]
X6 = np.stack(X_list, axis=1)  # (N,6,D)

delta_val = (X6[:, -1, :] - X6[:, 0, :])[:, np.newaxis, :]
X7_val = np.concatenate([X6, delta_val], axis=1)  # (N,7,D)

y_true_val = df_val_model[label_col].to_numpy(dtype=np.float32)

N, T, D = X7_val.shape
print(f"Validation X shape: {X7_val.shape} (N={N}, T={T}, D={D}) | label_col={label_col}")

# sanity checks
if T != int(cfg.get("timesteps", 7)):
    raise ValueError(f"Timesteps mismatch: got T={T} but best model expects {cfg.get('timesteps', 7)}")
if D != train_input_dim:
    raise ValueError(f"Input dim mismatch: V3 features D={D} but best model expects D={train_input_dim}")

# scale using training-fitted scaler_x (sx)
X7_val_scaled = sx.transform(X7_val.reshape(-1, D)).reshape(N, T, D)

Xv_t = torch.tensor(X7_val_scaled, dtype=torch.float32).to(device)
with torch.no_grad():
    yv_pred_scaled = best_model(Xv_t).detach().cpu().numpy().reshape(-1, 1)

y_pred_val = sy.inverse_transform(yv_pred_scaled).flatten()

rmse_val = float(np.sqrt(mean_squared_error(y_true_val, y_pred_val)))
mae_val  = float(mean_absolute_error(y_true_val, y_pred_val))

print(f"\n✅ EXTERNAL VALIDATION (V3 {v3_sheet})")
print(f"RMSE: {rmse_val:.4f} | MAE: {mae_val:.4f} | N={len(y_true_val)}")

val_out_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/20251105_TBME_Submission/20260301_Major_Revision"
os.makedirs(val_out_dir, exist_ok=True)

val_results_csv = os.path.join(val_out_dir, "Regression_best_model_validation_results(Stats)_7.csv")
val_preds_json  = os.path.join(val_out_dir, "Regression_best_model_validation_predictions(Stats)_7.json")

pd.DataFrame([{
    "model": cfg["model"],
    "sheet": v3_sheet,
    "label_col_used": label_col,
    "rmse_val": rmse_val,
    "mae_val": mae_val,
    "N_val": int(len(y_true_val)),
    "hidden_dim": int(cfg["hidden_dim"]),
    "dropout": float(cfg["dropout"]),
    "num_layers": int(cfg["num_layers"]),
    "batch_size": int(cfg.get("batch_size", -1)),
    "lr": float(cfg.get("lr", np.nan)),
    "weight_decay": float(cfg.get("weight_decay", np.nan)),
    "optimizer": cfg.get("optimizer", "NA"),
    "epochs": int(cfg.get("epochs", -1)),
}]).to_csv(val_results_csv, index=False)

with open(val_preds_json, "w") as f:
    json.dump([{
        "model": cfg["model"],
        "sheet": v3_sheet,
        "label_col_used": label_col,
        "y_true": y_true_val.tolist(),
        "y_pred": y_pred_val.tolist()
    }], f)

print("✅ External validation performance saved:", val_results_csv)
print("✅ External validation predictions saved:", val_preds_json)


### CNN Features Only (Regression) -- training + test + validation

In [ ]:
tw_features = {
    tw: [f"f{i}_TW{tw}" for i in range(1, 257)]
    for tw in range(1, 7)
}

### OSI+CNN Features (Regression) -- training + test + (validation) (_8)

In [ ]:
base_template = ["OSI_mean_TW{}", "OSI_std_TW{}"]
tw_features = {
    tw: [f.format(tw) for f in base_template] + [f"f{i}_TW{tw}" for i in range(1, 257)]
    for tw in range(1, 7)
}

In [ ]:
# ======================================================================================
# TBME Major Revision Utilities (REGRESSION: CNN + optional OSI feature set)
# - Save BEST config+weights per model type (RNN/LSTM/GRU/Transformer/Mamba)
# - Save TRAINING CV predictions + summary (like your previous *_7 regression code)
# - Evaluate BEST per model on TEST + TEMPORAL VALIDATION and save y_true/y_pred
# - Learning curve (20/40/60/80/100% of training patients)
# - PCA sensitivity:
#     * If OSI is used: PCA64 is applied ONLY to CNN block, then concat OSI, then train models
#     * If OSI not used: PCA64 is applied to full feature vector (same as before)
# - Patient-level bootstrap CIs for RMSE/MAE on TEST/VAL preds
# ======================================================================================

import os
import math
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from mamba_ssm import Mamba

# ======================================================================================
# ================================ USER CONTROLS ======================================
# ======================================================================================

RUN_GRID_SEARCH       = True     # trains many configs and saves best-per-model-type
RUN_EVAL_BEST_MODELS  = True     # runs TEST+VALIDATION for best-per-model-type (no retraining)
RUN_LEARNING_CURVES   = True     # trains best-config per model_type on fractions of patients
RUN_PCA_SENSITIVITY   = True     # trains best-config per model_type with PCA-64 (and optional PCA-32)
RUN_BOOTSTRAP_CI_EVAL = True     # patient-level bootstrap CI for RMSE/MAE on TEST/VAL preds

# For learning curves / PCA sensitivity, you probably don't want to redo all 5 models.
LEARNING_MODELS = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]    # or ["Mamba"]
PCA_MODELS      = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]    # or ["Mamba"]
PCA_COMPONENTS_LIST = [64]  # can set [64, 32] if you want both

LEARNING_FRACTIONS = [0.2, 0.4, 0.6, 0.8, 1.0]
LEARNING_SEEDS     = [0]  # repeats per fraction

# Bootstrap settings
BOOTSTRAP_B = 2000

# ---------------------------
# Feature set toggles
# ---------------------------
USE_OSI_FEATURES = True
"""
Set USE_OSI_FEATURES=True if you are building OSI+CNN features per TW.
IMPORTANT: If True, you MUST define `osi_features` below (per TW list of column names),
and PCA sensitivity will apply PCA only on the CNN block then concat OSI.
"""

CNN_DIM_PER_TW = 256  # CNN feature block size per TW (your f1..f256)
PCA_ON_CNN_ONLY_IF_OSI = True  # behavior you asked for

# ======================================================================================
# ================================ PATHS / I/O ========================================
# ======================================================================================

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Training data
xlsx_train  = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1V2_df.xlsx"
sheet_train = "Sheet5"

# Temporal validation data
xlsx_val  = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"
sheet_val = "Sheet15"

# Output roots
out_root_models = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels"
save_root = os.path.join(out_root_models, "SavedModels_OSIandCNN_8")  # NEW folder
os.makedirs(save_root, exist_ok=True)

out_revision_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/20251105_TBME_Submission/20260301_Major_Revision"
os.makedirs(out_revision_dir, exist_ok=True)

# ======================================================================================
# ================================= DATA LOADING ======================================
# ======================================================================================

target_col = "OSI_V2_12th_avg"

# CNN features (per TW): f1_TW1..f256_TW6
tw_cnn_features = {tw: [f"f{i}_TW{tw}" for i in range(1, 257)] for tw in range(1, 7)}

# Optional OSI features per TW
osi_features = {
    tw: [f"OSI_mean_TW{tw}", f"OSI_std_TW{tw}"]
    for tw in range(1, 7)
}

def build_sequences_from_df(
    df: pd.DataFrame,
    label_col: str,
    cnn_features: dict,
    use_osi: bool = False,
    osi_features: dict = None
):
    """
    Returns:
      X6: (N,6,D_perTW)
      y:  (N,)
      groups: (N,)
    """
    if osi_features is None:
        osi_features = {tw: [] for tw in range(1, 7)}

    X, y, groups = [], [], []
    for idx, row in df.iterrows():
        seq = []
        ok = True
        for tw in range(1, 7):
            cols = list(cnn_features[tw])
            if use_osi:
                cols = cols + list(osi_features.get(tw, []))

            if any(c not in df.columns for c in cols):
                ok = False
                break

            if row[cols].isnull().any():
                ok = False
                break

            seq.append(row[cols].values)

        if ok and (not pd.isna(row[label_col])):
            X.append(seq)
            y.append(float(row[label_col]))
            groups.append(row["ResearchID"] if "ResearchID" in df.columns else idx)

    X = np.array(X, dtype=np.float32)  # (N, 6, D)
    y = np.array(y, dtype=np.float32)  # (N,)
    groups = np.array(groups)
    return X, y, groups

def add_delta_window(X6: np.ndarray) -> np.ndarray:
    # X6: (N, 6, D)
    delta = (X6[:, -1, :] - X6[:, 0, :])[:, np.newaxis, :]  # (N,1,D)
    return np.concatenate([X6, delta], axis=1)              # (N,7,D)

# Load train dataframe
df = pd.read_excel(xlsx_train, sheet_name=sheet_train)

X6, y, groups = build_sequences_from_df(
    df,
    target_col,
    tw_cnn_features,
    use_osi=USE_OSI_FEATURES,
    osi_features=osi_features
)
X = add_delta_window(X6)  # (N,7,D_total)

print(f"✅ Total usable samples: {len(X)}")
print(f"✅ Input shape: {X.shape} (N,T,D_total)")

if USE_OSI_FEATURES:
    d_total = X.shape[2]
    osi_dim_per_tw = len(osi_features.get(1, []))
    print(f"✅ USE_OSI_FEATURES=True | CNN_DIM_PER_TW={CNN_DIM_PER_TW} | D_total_perTW={d_total} | OSI_dims_perTW={d_total - CNN_DIM_PER_TW}")

# ======================================================================================
# ============================= PREPROCESSING HELPERS =================================
# ======================================================================================

def fit_scaler_on_train_X(X_train: np.ndarray) -> StandardScaler:
    scaler = StandardScaler()
    N, T, D = X_train.shape
    scaler.fit(X_train.reshape(-1, D))
    return scaler

def apply_scaler_X(scaler: StandardScaler, X_in: np.ndarray) -> np.ndarray:
    N, T, D = X_in.shape
    return scaler.transform(X_in.reshape(-1, D)).reshape(N, T, D).astype(np.float32)

def fit_scaler_on_train_y(y_train: np.ndarray) -> StandardScaler:
    scaler = StandardScaler()
    scaler.fit(y_train.reshape(-1, 1))
    return scaler

def apply_scaler_y(scaler: StandardScaler, y_in: np.ndarray) -> np.ndarray:
    return scaler.transform(y_in.reshape(-1, 1)).flatten().astype(np.float32)

def invert_scaler_y(scaler: StandardScaler, y_scaled: np.ndarray) -> np.ndarray:
    return scaler.inverse_transform(y_scaled.reshape(-1, 1)).flatten().astype(np.float32)

def augment_data(X_in, y_in, groups_in, noise_std=0.005, seed=0):
    rng = np.random.RandomState(seed)
    noise = rng.normal(0, noise_std, X_in.shape).astype(np.float32)
    return (
        np.concatenate([X_in, X_in + noise], axis=0),
        np.concatenate([y_in, y_in], axis=0),
        np.concatenate([groups_in, groups_in], axis=0)
    )

# --- PCA (full vector) ---
def fit_pca_on_train_full(X_train_scaled: np.ndarray, n_components: int) -> PCA:
    N, T, D = X_train_scaled.shape
    pca = PCA(n_components=n_components, random_state=0)
    pca.fit(X_train_scaled.reshape(-1, D))
    return pca

def apply_pca_full(pca: PCA, X_scaled: np.ndarray) -> np.ndarray:
    N, T, D = X_scaled.shape
    X2 = pca.transform(X_scaled.reshape(-1, D))  # (N*T, K)
    return X2.reshape(N, T, -1).astype(np.float32)

# --- PCA (CNN-only then concat OSI) ---
def fit_pca_on_train_cnn_only(X_train_scaled: np.ndarray, cnn_dim: int, n_components: int) -> PCA:
    """
    Fit PCA ONLY on the CNN block (first cnn_dim features in the last dimension), after scaling.
    Assumes features are [CNN | OSI] per time window.
    """
    N, T, D = X_train_scaled.shape
    assert D >= cnn_dim, f"D_total({D}) < cnn_dim({cnn_dim}). Check feature construction."
    X_cnn = X_train_scaled[:, :, :cnn_dim].reshape(-1, cnn_dim)
    pca = PCA(n_components=n_components, random_state=0)
    pca.fit(X_cnn)
    return pca

def apply_pca_cnn_only_and_concat(X_scaled: np.ndarray, pca: PCA, cnn_dim: int) -> np.ndarray:
    """
    Apply PCA ONLY to CNN block, keep OSI block unchanged, then concat.
    Output shape: (N,T,n_components + OSI_dim)
    """
    N, T, D = X_scaled.shape
    assert D >= cnn_dim, f"D_total({D}) < cnn_dim({cnn_dim}). Check feature construction."
    X_cnn = X_scaled[:, :, :cnn_dim].reshape(-1, cnn_dim)  # (N*T, cnn_dim)
    X_osi = X_scaled[:, :, cnn_dim:]                       # (N,T,OSI_dim)

    X_cnn_pca = pca.transform(X_cnn).reshape(N, T, -1).astype(np.float32)
    X_out = np.concatenate([X_cnn_pca, X_osi.astype(np.float32)], axis=-1)
    return X_out

# ======================================================================================
# ================================== MODELS ===========================================
# ======================================================================================

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        w = torch.softmax(self.attn(x), dim=1)  # (B,T,1)
        return torch.sum(w * x, dim=1)          # (B,H)

class RNNRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, rnn_type="RNN", num_layers=1, dropout=0.0):
        super().__init__()
        if num_layers == 1:
            dropout = 0.0
        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[rnn_type]
        self.rnn = rnn_cls(input_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.attn(out)
        return self.fc(out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, nhead, dropout):
        super().__init__()
        enc = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=nhead, dropout=dropout,
            batch_first=True, layer_norm_eps=1e-5
        )
        self.encoder = nn.TransformerEncoder(enc, num_layers=1)

    def forward(self, x):
        return self.encoder(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0, nhead=None):
        super().__init__()
        if nhead is None:
            if hidden_dim % 8 == 0:
                nhead = max(1, hidden_dim // 8)
            elif hidden_dim % 4 == 0:
                nhead = max(1, hidden_dim // 4)
            else:
                nhead = 1

        self.input_proj = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim))
        self.pe = PositionalEncoding(hidden_dim)

        if num_layers == 1:
            self.stack = nn.Sequential(TransformerBlock(hidden_dim, nhead, 0.0))
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(TransformerBlock(hidden_dim, nhead, dropout), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])

        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pe(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

class MambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba = Mamba(d_model=dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return self.norm(self.mamba(x))

class MambaRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        if num_layers == 1:
            self.stack = nn.Sequential(MambaBlock(hidden_dim))
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(MambaBlock(hidden_dim), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.norm(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

def build_model(model_type, input_dim, hidden_dim, num_layers, dropout):
    if model_type == "Mamba":
        return MambaRegressor(input_dim, hidden_dim, num_layers, dropout)
    if model_type == "Transformer":
        return TransformerModel(input_dim, hidden_dim, num_layers, dropout)
    return RNNRegressor(input_dim, hidden_dim, model_type, num_layers, dropout)

def get_loader(X_in, y_in, batch_size, shuffle=True):
    X_tensor = torch.tensor(X_in, dtype=torch.float32)
    y_tensor = torch.tensor(y_in, dtype=torch.float32).view(-1, 1)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=shuffle)

# ======================================================================================
# ============================== REGRESSION METRICS ===================================
# ======================================================================================

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))

def eval_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "RMSE": rmse(y_true, y_pred),
        "MAE": mae(y_true, y_pred),
        "N": int(len(y_true))
    }

# Patient-level bootstrap CI for RMSE/MAE (resample patients/groups with replacement)
def bootstrap_ci_regression_grouped(y_true, y_pred, groups, B=2000, seed=0):
    rng = np.random.RandomState(seed)

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    groups = np.asarray(groups)

    uniq = np.unique(groups)
    point = eval_regression_metrics(y_true, y_pred)

    dist_rmse, dist_mae = [], []
    for _ in range(B):
        sampled_groups = rng.choice(uniq, size=len(uniq), replace=True)
        mask = np.isin(groups, sampled_groups)
        yt = y_true[mask]
        yp = y_pred[mask]
        if len(yt) < 2:
            continue
        dist_rmse.append(rmse(yt, yp))
        dist_mae.append(mae(yt, yp))

    def ci(arr):
        if len(arr) == 0:
            return (np.nan, np.nan)
        return (float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5)))

    rmse_lo, rmse_hi = ci(dist_rmse)
    mae_lo, mae_hi = ci(dist_mae)

    out = pd.DataFrame([
        {"metric": "RMSE", "point": point["RMSE"], "ci_low": rmse_lo, "ci_high": rmse_hi, "n_boot": len(dist_rmse)},
        {"metric": "MAE",  "point": point["MAE"],  "ci_low": mae_lo,  "ci_high": mae_hi,  "n_boot": len(dist_mae)},
    ])
    return out

# ======================================================================================
# ================================ TRAIN/TEST SPLIT ===================================
# ======================================================================================

# Split BEFORE scaling to avoid leakage (X and y)
X_train_raw, X_test_raw, y_train_raw, y_test_raw, groups_train, groups_test = train_test_split(
    X, y, groups, test_size=0.2, random_state=42
)
print(f"Train N={len(X_train_raw)} | Test N={len(X_test_raw)}")

# Fit scalers on TRAIN only
scaler_x = fit_scaler_on_train_X(X_train_raw)
scaler_y = fit_scaler_on_train_y(y_train_raw)

# Apply scaling
X_train = apply_scaler_X(scaler_x, X_train_raw)
X_test  = apply_scaler_X(scaler_x, X_test_raw)

y_train = apply_scaler_y(scaler_y, y_train_raw)
y_test  = apply_scaler_y(scaler_y, y_test_raw)

# Augment train only (do NOT augment test/validation)
X_train_aug, y_train_aug, groups_train_aug = augment_data(X_train, y_train, groups_train, seed=0)
print(f"Train after augmentation N={len(X_train_aug)}")

# ======================================================================================
# ============================== GRID SEARCH TRAINING =================================
# ======================================================================================

def train_one_config_cv_regression(
    model_type, X_train_in, y_train_in, groups_train_in,
    hidden_dim, dropout, num_layers, batch_size, lr, wd, optimizer_name,
    epochs=200, n_splits=5, early_stop_patience=10,
    collect_cv_preds=True,
    scaler_y_for_inverse=None
):
    """
    - Early stop + scheduler on VALIDATION LOSS
    - Save per-fold CV predictions in ORIGINAL y domain (inverse-scaled)
    - Return summary metrics (RMSE/MAE mean/std in ORIGINAL y domain),
      plus "best state dict for this hyperparam combo" chosen by lowest val loss.
    """
    gkf = GroupKFold(n_splits=n_splits)

    rmse_list, mae_list = [], []
    used_epochs_all = []

    combo_best_val_loss = np.inf
    combo_best_state = None

    cv_pred_records = []

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train_in, y_train_in, groups_train_in)):

        model = build_model(model_type, X_train_in.shape[2], hidden_dim, num_layers, dropout).to(device)

        opt_cls = torch.optim.Adam if optimizer_name == "Adam" else torch.optim.SGD
        optimizer = opt_cls(model.parameters(), lr=lr, weight_decay=wd)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5)
        criterion = nn.MSELoss()

        train_loader = get_loader(X_train_in[tr_idx], y_train_in[tr_idx], batch_size=batch_size, shuffle=True)

        # pre-make val tensors
        val_x_t = torch.tensor(X_train_in[val_idx], dtype=torch.float32).to(device)
        val_y_t = torch.tensor(y_train_in[val_idx], dtype=torch.float32).view(-1, 1).to(device)

        best_val_loss = float("inf")
        wait = 0
        used_epochs = 0
        best_state_fold = None

        for _ in range(epochs):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                pred = model(xb)
                loss = criterion(pred, yb)
                if torch.isnan(loss) or torch.isinf(loss):
                    continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            # ---- VALIDATION LOSS (early stop + scheduler) ----
            model.eval()
            with torch.no_grad():
                val_pred = model(val_x_t)
                val_loss = float(criterion(val_pred, val_y_t).item())

            scheduler.step(val_loss)
            used_epochs += 1

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                wait = 0
                best_state_fold = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                wait += 1
                if wait >= early_stop_patience:
                    break

        used_epochs_all.append(used_epochs)

        # restore best fold weights
        if best_state_fold is not None:
            model.load_state_dict(best_state_fold)

        # predict on fold val in SCALED domain
        model.eval()
        with torch.no_grad():
            preds_scaled = model(val_x_t).squeeze().detach().cpu().numpy()

        yt_scaled = y_train_in[val_idx].reshape(-1)

        # inverse-transform to ORIGINAL y domain
        if scaler_y_for_inverse is None:
            raise ValueError("scaler_y_for_inverse is required to compute RMSE/MAE in original domain.")
        y_pred = invert_scaler_y(scaler_y_for_inverse, preds_scaled)
        y_true = invert_scaler_y(scaler_y_for_inverse, yt_scaled)

        rmse_list.append(rmse(y_true, y_pred))
        mae_list.append(mae(y_true, y_pred))

        if collect_cv_preds:
            cv_pred_records.append({
                "model": model_type,
                "hidden_dim": int(hidden_dim),
                "dropout": float(dropout),
                "num_layers": int(num_layers),
                "batch_size": int(batch_size),
                "lr": float(lr),
                "weight_decay": float(wd),
                "optimizer": optimizer_name,
                "fold": int(fold),
                "y_true": y_true.tolist(),
                "y_pred": y_pred.tolist()
            })

        # combo best weights chosen by lowest val loss (scaled domain)
        if best_val_loss < combo_best_val_loss and best_state_fold is not None:
            combo_best_val_loss = best_val_loss
            combo_best_state = best_state_fold

    summary = {
        "RMSE_mean": float(np.mean(rmse_list)) if len(rmse_list) else np.inf,
        "RMSE_std":  float(np.std(rmse_list))  if len(rmse_list) else np.nan,
        "MAE_mean":  float(np.mean(mae_list))  if len(mae_list)  else np.inf,
        "MAE_std":   float(np.std(mae_list))   if len(mae_list)  else np.nan,
        "n_folds_done": int(len(rmse_list)),
        "expected_folds": int(n_splits),
        "used_epochs_mean": float(np.mean(used_epochs_all)) if used_epochs_all else np.nan,
        "used_epochs_std":  float(np.std(used_epochs_all))  if used_epochs_all else np.nan,
        "combo_best_val_loss": float(combo_best_val_loss),
        "has_state": combo_best_state is not None
    }
    return summary, combo_best_state, cv_pred_records

# ======================================================================================
# ============================ GRID SEARCH + SAVE BEST PER MODEL =======================
# ======================================================================================

model_types = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]

# best-per-model based on lowest RMSE_mean (original y domain)
best_per_model = {m: {"rmse": np.inf, "cfg": None, "state": None} for m in model_types}
grid_summary_rows = []

# store CV predictions
cv_predictions_all = []

if RUN_GRID_SEARCH:
    epochs = 200
    n_splits = 5

    for model_type in model_types:
        for hidden_dim in [16, 32, 64, 128]:
            for dropout in [0.0, 0.2]:
                for num_layers in [1, 2, 3]:
                    for batch_size in [16, 32]:
                        for lr in [0.001, 0.01]:
                            for wd in [0, 1e-4]:
                                for optimizer_name in ["Adam", "SGD"]:

                                    summary, best_state, cv_pred_records = train_one_config_cv_regression(
                                        model_type,
                                        X_train_aug, y_train_aug, groups_train_aug,
                                        hidden_dim, dropout, num_layers, batch_size, lr, wd, optimizer_name,
                                        epochs=epochs, n_splits=n_splits, early_stop_patience=10,
                                        collect_cv_preds=True,
                                        scaler_y_for_inverse=scaler_y
                                    )

                                    if cv_pred_records:
                                        cv_predictions_all.extend(cv_pred_records)

                                    row = {
                                        "model": model_type,
                                        "hidden_dim": hidden_dim,
                                        "dropout": dropout,
                                        "num_layers": num_layers,
                                        "batch_size": batch_size,
                                        "lr": lr,
                                        "weight_decay": wd,
                                        "optimizer": optimizer_name,
                                        "epochs": epochs,
                                        "folds": n_splits,
                                        **summary
                                    }
                                    grid_summary_rows.append(row)

                                    # update best for this model type (lowest RMSE_mean)
                                    rmse_mean = row["RMSE_mean"]
                                    if np.isfinite(rmse_mean) and rmse_mean < best_per_model[model_type]["rmse"] and summary["has_state"]:
                                        best_per_model[model_type]["rmse"] = float(rmse_mean)
                                        best_per_model[model_type]["cfg"] = {
                                            "model": model_type,
                                            "hidden_dim": int(hidden_dim),
                                            "dropout": float(dropout),
                                            "num_layers": int(num_layers),
                                            "batch_size": int(batch_size),
                                            "lr": float(lr),
                                            "weight_decay": float(wd),
                                            "optimizer": optimizer_name,
                                            "epochs": int(epochs),
                                            "folds": int(n_splits),
                                            "target_col": target_col,
                                            "timesteps": int(X_train_aug.shape[1]),
                                            "input_dim": int(X_train_aug.shape[2]),
                                            "feature_repr": "CNN+OSI" if USE_OSI_FEATURES else "CNN",
                                            "pca_components": None,
                                            "pca_mode": None,  # "full" or "cnn_only"
                                            "cnn_dim_per_tw": int(CNN_DIM_PER_TW) if USE_OSI_FEATURES else None,
                                        }
                                        best_per_model[model_type]["state"] = best_state
                                        print(f"\n🏆 BEST for {model_type}: RMSE_mean={rmse_mean:.4f}")

        # Save best artifacts per model_type
        bm = best_per_model[model_type]
        if bm["cfg"] is not None:
            model_dir = os.path.join(save_root, f"best_{model_type}")
            os.makedirs(model_dir, exist_ok=True)

            torch.save({"state_dict": bm["state"], "config": bm["cfg"]}, os.path.join(model_dir, "best_model_state.pt"))
            with open(os.path.join(model_dir, "best_model_config.json"), "w") as f:
                json.dump(bm["cfg"], f, indent=2)

            # save scalers (train-only fitted)
            joblib.dump(scaler_x, os.path.join(model_dir, "scaler_x.joblib"))
            joblib.dump(scaler_y, os.path.join(model_dir, "scaler_y.joblib"))

            print(f"✅ Saved best artifacts for {model_type} -> {model_dir}")

    # Save grid summary
    df_grid = pd.DataFrame(grid_summary_rows)
    grid_csv_out = os.path.join(out_root_models, "Regression_grid_search_results(OSIandCNN)_8.csv")
    df_grid.to_csv(grid_csv_out, index=False)
    print("✅ Grid search summary saved:", grid_csv_out)

    # Save CV predictions JSON
    cv_json_out = os.path.join(out_root_models, "Regression_cv_predictions(OSIandCNN)_8.json")
    with open(cv_json_out, "w") as f:
        json.dump(cv_predictions_all, f)
    print(f"✅ CV predictions saved: {len(cv_predictions_all)} records -> {cv_json_out}")

# ======================================================================================
# ============================= LOAD BEST-PER-MODEL ARTIFACTS ==========================
# ======================================================================================

def load_best_for_model(model_type: str):
    model_dir = os.path.join(save_root, f"best_{model_type}")
    ckpt = os.path.join(model_dir, "best_model_state.pt")
    cfgp = os.path.join(model_dir, "best_model_config.json")
    sxp  = os.path.join(model_dir, "scaler_x.joblib")
    syp  = os.path.join(model_dir, "scaler_y.joblib")

    if not (os.path.exists(ckpt) and os.path.exists(cfgp) and os.path.exists(sxp) and os.path.exists(syp)):
        raise FileNotFoundError(f"Missing best artifacts for {model_type} in {model_dir}")

    with open(cfgp, "r") as f:
        cfg = json.load(f)

    sx = joblib.load(sxp)
    sy = joblib.load(syp)
    state = torch.load(ckpt, map_location="cpu")

    return cfg, sx, sy, state["state_dict"]

# ======================================================================================
# =============================== VALIDATION BUILD (V3) ================================
# ======================================================================================

def load_validation_set(scaler_train_x: StandardScaler, scaler_train_y: StandardScaler):
    df_val = pd.read_excel(xlsx_val, sheet_name=sheet_val)
    train_target_col = target_col
    val_target_col = "OSI_V3_12th_avg"

    if train_target_col in df_val.columns:
        label_col = train_target_col
    elif val_target_col in df_val.columns:
        print(f"NOTE: '{train_target_col}' not found in {sheet_val}; using '{val_target_col}' instead.")
        label_col = val_target_col
    else:
        raise ValueError(f"Missing target in {sheet_val}: neither {train_target_col} nor {val_target_col}")

    # build required feature columns
    all_cols = []
    for tw in range(1, 7):
        all_cols += list(tw_cnn_features[tw])
        if USE_OSI_FEATURES:
            all_cols += list(osi_features.get(tw, []))

    missing = [c for c in all_cols if c not in df_val.columns]
    if missing:
        raise ValueError(f"Missing {len(missing)} feature cols in {sheet_val}. First 20: {missing[:20]}")

    df_val_model = df_val.dropna(subset=[label_col] + all_cols).copy()
    if len(df_val_model) == 0:
        raise ValueError("No validation rows left after dropna.")

    # X6
    X_list = []
    for tw in range(1, 7):
        cols = list(tw_cnn_features[tw])
        if USE_OSI_FEATURES:
            cols += list(osi_features.get(tw, []))
        X_list.append(df_val_model[cols].to_numpy(dtype=np.float32))

    X6_val = np.stack(X_list, axis=1)  # (N,6,D_total)
    X7_val = add_delta_window(X6_val)  # (N,7,D_total)

    y_val_raw = df_val_model[label_col].to_numpy(dtype=np.float32)

    # scale using training scalers (no refit)
    X7_val_scaled = apply_scaler_X(scaler_train_x, X7_val)
    y_val_scaled  = apply_scaler_y(scaler_train_y, y_val_raw)

    # groups for bootstrap (if ResearchID exists)
    g_val = df_val_model["ResearchID"].to_numpy() if "ResearchID" in df_val_model.columns else np.arange(len(df_val_model))

    return X7_val_scaled, y_val_scaled, y_val_raw, g_val, label_col

# ======================================================================================
# =============================== EVALUATION: TEST + VAL ===============================
# ======================================================================================

def predict_scaled(model: nn.Module, X_in: np.ndarray, batch_size=32):
    X_tensor = torch.tensor(X_in, dtype=torch.float32)
    loader = DataLoader(TensorDataset(X_tensor), batch_size=batch_size, shuffle=False)
    preds = []
    model.eval()
    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            yhat = model(xb).detach().cpu().numpy().reshape(-1)
            preds.append(yhat)
    return np.concatenate(preds, axis=0)

def evaluate_and_save_best_per_model():
    test_payloads = []
    val_payloads  = []

    metrics_rows_test = []
    metrics_rows_val  = []

    # Original (unscaled) y for test split
    y_test_raw_local = invert_scaler_y(scaler_y, y_test)

    for m in model_types:
        cfg, sx, sy, state_dict = load_best_for_model(m)

        model = build_model(
            cfg["model"],
            input_dim=int(cfg["input_dim"]),
            hidden_dim=int(cfg["hidden_dim"]),
            num_layers=int(cfg["num_layers"]),
            dropout=float(cfg["dropout"])
        ).to(device)
        model.load_state_dict(state_dict)
        model.eval()

        bs = int(cfg.get("batch_size", 32))

        # TEST (X_test is already scaled with global scaler_x; should match sx)
        y_pred_test_scaled = predict_scaled(model, X_test, batch_size=bs)
        y_pred_test = invert_scaler_y(sy, y_pred_test_scaled)
        y_true_test = y_test_raw_local

        test_payloads.append({
            "model": m,
            "config": cfg,
            "y_true": y_true_test.tolist(),
            "y_pred": y_pred_test.tolist(),
            "groups": groups_test.tolist()
        })

        mt = eval_regression_metrics(y_true_test, y_pred_test)
        metrics_rows_test.append({"model": m, **mt})

        # VALIDATION
        X_val_scaled, y_val_scaled, y_val_raw, g_val, label_col_used = load_validation_set(sx, sy)

        y_pred_val_scaled = predict_scaled(model, X_val_scaled, batch_size=bs)
        y_pred_val = invert_scaler_y(sy, y_pred_val_scaled)
        y_true_val = y_val_raw

        val_payloads.append({
            "model": m,
            "config": cfg,
            "label_col_used": label_col_used,
            "y_true": y_true_val.tolist(),
            "y_pred": y_pred_val.tolist(),
            "groups": g_val.tolist()
        })

        mv = eval_regression_metrics(y_true_val, y_pred_val)
        metrics_rows_val.append({"model": m, **mv, "label_col_used": label_col_used})

    # Save predictions JSON (ALL 5 models)
    test_preds_json = os.path.join(out_root_models, "Regression_best_model_test_predictions(OSIandCNN)_8.json")
    val_preds_json  = os.path.join(out_revision_dir, "Regression_best_model_validation_predictions(OSIandCNN)_8.json")

    with open(test_preds_json, "w") as f:
        json.dump(test_payloads, f, indent=2)
    with open(val_preds_json, "w") as f:
        json.dump(val_payloads, f, indent=2)

    # Save metric CSVs (point estimates)
    pd.DataFrame(metrics_rows_test).to_csv(
        os.path.join(out_root_models, "Regression_best_models_TEST_metrics_point(OSIandCNN)_8.csv"),
        index=False
    )
    pd.DataFrame(metrics_rows_val).to_csv(
        os.path.join(out_revision_dir, "Regression_best_models_VALIDATION_metrics_point(OSIandCNN)_8.csv"),
        index=False
    )

    print("✅ Saved ALL5 TEST preds:", test_preds_json)
    print("✅ Saved ALL5 VAL preds:",  val_preds_json)

    return test_preds_json, val_preds_json

# ======================================================================================
# ============================ BOOTSTRAP CI (GROUPED) =================================
# ======================================================================================

def run_bootstrap_ci(preds_json_path: str, split_name: str, out_dir: str):
    with open(preds_json_path, "r") as f:
        payloads = json.load(f)

    all_rows = []
    for item in payloads:
        model_name = item["model"]
        y_true = np.array(item["y_true"], dtype=float)
        y_pred = np.array(item["y_pred"], dtype=float)
        groups = np.array(item.get("groups", np.arange(len(y_true))), dtype=object)

        ci_df = bootstrap_ci_regression_grouped(y_true, y_pred, groups, B=BOOTSTRAP_B, seed=0)
        ci_df.insert(0, "split", split_name)
        ci_df.insert(1, "model", model_name)
        all_rows.append(ci_df)

    out = pd.concat(all_rows, ignore_index=True)
    out_csv = os.path.join(out_dir, f"BootstrapCI_{split_name}_OSIandCNN_8.csv")
    out.to_csv(out_csv, index=False)
    print(f"✅ Bootstrap CI saved: {out_csv}")

# ======================================================================================
# ================================ LEARNING CURVES =====================================
# ======================================================================================

def train_single_model_once_regression(
    model_type: str,
    cfg: dict,
    X_train_in: np.ndarray,
    y_train_in: np.ndarray,      # scaled y
    groups_train_in: np.ndarray,
    X_test_in: np.ndarray,
    y_test_raw: np.ndarray,      # raw y
    X_val_in: np.ndarray,
    y_val_raw: np.ndarray,       # raw y
    sy: StandardScaler,          # y scaler for inverse
    seed: int = 0
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = build_model(
        model_type,
        input_dim=X_train_in.shape[2],
        hidden_dim=int(cfg["hidden_dim"]),
        num_layers=int(cfg["num_layers"]),
        dropout=float(cfg["dropout"])
    ).to(device)

    optimizer_name = cfg["optimizer"]
    lr = float(cfg["lr"])
    wd = float(cfg["weight_decay"])
    batch_size = int(cfg["batch_size"])
    epochs = int(cfg["epochs"])

    opt_cls = torch.optim.Adam if optimizer_name == "Adam" else torch.optim.SGD
    optimizer = opt_cls(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5)
    criterion = nn.MSELoss()

    loader = get_loader(X_train_in, y_train_in, batch_size=batch_size, shuffle=True)

    best_val_proxy = float("inf")
    wait = 0
    best_state = None

    # (We don't have a dedicated val split here; use training loss proxy just to keep this lightweight.
    #  If you prefer, create an inner GroupKFold and early-stop on that.)
    for _ in range(epochs):
        model.train()
        losses = []
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            if torch.isnan(loss) or torch.isinf(loss):
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            losses.append(loss.item())

        avg_loss = float(np.mean(losses)) if len(losses) else np.inf
        scheduler.step(avg_loss)

        if avg_loss < best_val_proxy:
            best_val_proxy = avg_loss
            wait = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= 10:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    # TEST
    y_pred_test_scaled = predict_scaled(model, X_test_in, batch_size=batch_size)
    y_pred_test = invert_scaler_y(sy, y_pred_test_scaled)
    m_test = eval_regression_metrics(y_test_raw, y_pred_test)

    # VAL
    y_pred_val_scaled = predict_scaled(model, X_val_in, batch_size=batch_size)
    y_pred_val = invert_scaler_y(sy, y_pred_val_scaled)
    m_val = eval_regression_metrics(y_val_raw, y_pred_val)

    return m_test, m_val

def run_learning_curves():
    # Use global scalers (same as training split)
    X_val_scaled, y_val_scaled, y_val_raw, g_val, label_col_used = load_validation_set(scaler_x, scaler_y)

    rows = []
    unique_groups = np.unique(groups_train)

    # raw test y
    y_test_raw_local = invert_scaler_y(scaler_y, y_test)

    for model_type in LEARNING_MODELS:
        cfg, _, sy, _ = load_best_for_model(model_type)

        for frac in LEARNING_FRACTIONS:
            n_groups = max(2, int(len(unique_groups) * frac))
            for seed in LEARNING_SEEDS:
                rng = np.random.RandomState(seed)
                chosen_groups = rng.choice(unique_groups, size=n_groups, replace=False)
                mask = np.isin(groups_train, chosen_groups)

                X_sub = X_train[mask]
                y_sub = y_train[mask]
                g_sub = groups_train[mask]

                X_sub_aug, y_sub_aug, g_sub_aug = augment_data(X_sub, y_sub, g_sub, seed=seed)

                m_test, m_val = train_single_model_once_regression(
                    model_type, cfg,
                    X_sub_aug, y_sub_aug, g_sub_aug,
                    X_test, y_test_raw_local,
                    X_val_scaled, y_val_raw,
                    sy=sy,
                    seed=seed
                )

                rows.append({
                    "model": model_type,
                    "fraction": frac,
                    "seed": seed,
                    "label_col_val": label_col_used,
                    **{f"test_{k}": v for k, v in m_test.items()},
                    **{f"val_{k}": v for k, v in m_val.items()},
                    "n_train_patients": int(n_groups),
                    "n_train_samples": int(len(X_sub))
                })

                print(f"✅ LearningCurve {model_type} frac={frac} seed={seed} "
                      f"test_RMSE={m_test['RMSE']:.3f} val_RMSE={m_val['RMSE']:.3f}")

    df_lc = pd.DataFrame(rows)
    lc_csv = os.path.join(out_revision_dir, "LearningCurve_OSIandCNN_8.csv")
    df_lc.to_csv(lc_csv, index=False)
    print("✅ Learning curve results saved:", lc_csv)

# ======================================================================================
# ================================ PCA SENSITIVITY =====================================
# ======================================================================================

def run_pca_sensitivity():
    # Validation (scaled with global scalers)
    X_val_scaled, y_val_scaled, y_val_raw, g_val, label_col_used = load_validation_set(scaler_x, scaler_y)

    # raw test y
    y_test_raw_local = invert_scaler_y(scaler_y, y_test)

    rows = []
    for model_type in PCA_MODELS:
        cfg, _, sy, _ = load_best_for_model(model_type)

        for k in PCA_COMPONENTS_LIST:
            if USE_OSI_FEATURES and PCA_ON_CNN_ONLY_IF_OSI:
                # PCA ONLY on CNN block
                pca = fit_pca_on_train_cnn_only(X_train, cnn_dim=CNN_DIM_PER_TW, n_components=int(k))

                X_train_pca = apply_pca_cnn_only_and_concat(X_train, pca, cnn_dim=CNN_DIM_PER_TW)
                X_test_pca  = apply_pca_cnn_only_and_concat(X_test,  pca, cnn_dim=CNN_DIM_PER_TW)
                X_val_pca   = apply_pca_cnn_only_and_concat(X_val_scaled, pca, cnn_dim=CNN_DIM_PER_TW)

                cfg_pca = dict(cfg)
                cfg_pca["pca_components"] = int(k)
                cfg_pca["pca_mode"] = "cnn_only"
            else:
                # PCA on full vector
                pca = fit_pca_on_train_full(X_train, n_components=int(k))

                X_train_pca = apply_pca_full(pca, X_train)
                X_test_pca  = apply_pca_full(pca, X_test)
                X_val_pca   = apply_pca_full(pca, X_val_scaled)

                cfg_pca = dict(cfg)
                cfg_pca["pca_components"] = int(k)
                cfg_pca["pca_mode"] = "full"

            # augment train only
            X_train_pca_aug, y_train_aug2, g_train_aug2 = augment_data(X_train_pca, y_train, groups_train, seed=0)

            m_test, m_val = train_single_model_once_regression(
                model_type, cfg_pca,
                X_train_pca_aug, y_train_aug2, g_train_aug2,
                X_test_pca, y_test_raw_local,
                X_val_pca, y_val_raw,
                sy=sy,
                seed=0
            )

            rows.append({
                "model": model_type,
                "pca_components": int(k),
                "pca_mode": cfg_pca.get("pca_mode"),
                "label_col_val": label_col_used,
                "input_dim_after_pca": int(X_train_pca.shape[2]),
                **{f"test_{k2}": v for k2, v in m_test.items()},
                **{f"val_{k2}": v for k2, v in m_val.items()},
            })

            print(f"✅ PCA{k} {model_type} ({cfg_pca.get('pca_mode')}): "
                  f"test_RMSE={m_test['RMSE']:.3f} val_RMSE={m_val['RMSE']:.3f} | input_dim={X_train_pca.shape[2]}")

    df_pca = pd.DataFrame(rows)
    pca_csv = os.path.join(out_revision_dir, "PCASensitivity_OSIandCNN_8.csv")
    df_pca.to_csv(pca_csv, index=False)
    print("✅ PCA sensitivity results saved:", pca_csv)

# ======================================================================================
# ===================================== RUN ===========================================
# ======================================================================================

test_preds_json = None
val_preds_json = None

if RUN_EVAL_BEST_MODELS:
    test_preds_json, val_preds_json = evaluate_and_save_best_per_model()

if RUN_BOOTSTRAP_CI_EVAL:
    if test_preds_json is None or val_preds_json is None:
        test_preds_json = os.path.join(out_root_models, "Regression_best_model_test_predictions(OSIandCNN)_8.json")
        val_preds_json  = os.path.join(out_revision_dir, "Regression_best_model_validation_predictions(OSIandCNN)_8.json")
    run_bootstrap_ci(test_preds_json, "TEST", out_root_models)
    run_bootstrap_ci(val_preds_json, "VALIDATION", out_revision_dir)

if RUN_LEARNING_CURVES:
    run_learning_curves()

if RUN_PCA_SENSITIVITY:
    run_pca_sensitivity()

print("✅ All done.")

### OSI+CNN Features (Regression) -- training + test + (validation) (_7)

In [ ]:
# === Try more (Regression) WITH BEST-MODEL WEIGHT + SCALER SAVING + TEST + VALIDATION ===
# Updates vs your original:
#  1) Early-stopping + "best fold weights" are selected using **VALIDATION loss** (not training loss)
#  2) Scheduler steps on **VALIDATION loss**
#  3) Validation block is enabled and supports V3 target name mismatch:
#        if OSI_V2_12th_avg missing -> use OSI_V3_12th_avg
#  4) Fixes a small filename typo in validation json (*_7 not *_6)

import os
import math
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from mamba_ssm import Mamba

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -----------------------------
# Load Data
# -----------------------------
df = pd.read_excel(
    "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1V2_df.xlsx",
    sheet_name="Sheet5"
)

target = "OSI_V2_12th_avg"

base_template = ["OSI_mean_TW{}", "OSI_std_TW{}"]
tw_features = {
    tw: [f.format(tw) for f in base_template] + [f"f{i}_TW{tw}" for i in range(1, 257)]
    for tw in range(1, 7)
}

# -----------------------------
# Build sequences
# -----------------------------
X, y, groups = [], [], []
for idx, row in df.iterrows():
    sequence = []
    for tw in range(1, 7):
        cols = tw_features[tw]
        # if any required column missing or NaN
        if row[cols].isnull().any():
            break
        sequence.append(row[cols].values)
    if len(sequence) == 6 and not pd.isna(row[target]):
        X.append(sequence)
        y.append(row[target])
        groups.append(row["ResearchID"] if "ResearchID" in row else idx)

X = np.array(X, dtype=np.float32)   # (N, 6, D)
y = np.array(y, dtype=np.float32)   # (N,)
groups = np.array(groups)

# -----------------------------
# Feature Engineering: delta (TW6 - TW1) as extra timestep
# -----------------------------
delta_features = (X[:, -1, :] - X[:, 0, :])[:, np.newaxis, :]
X = np.concatenate([X, delta_features], axis=1)  # (N, 7, D)

# -----------------------------
# Scaling
# -----------------------------
scaler_x = StandardScaler()
X_scaled = scaler_x.fit_transform(X.reshape(-1, X.shape[2])).reshape(X.shape)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

# -----------------------------
# Train/Test Split
# -----------------------------
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X_scaled, y_scaled, groups, test_size=0.2, random_state=42
)

# -----------------------------
# Data Augmentation (train only)
# -----------------------------
def augment_data(X, y, groups):
    noise = np.random.normal(0, 0.005, X.shape).astype(np.float32)
    return (
        np.concatenate([X, X + noise]),
        np.concatenate([y, y]),
        np.concatenate([groups, groups])
    )

X_train, y_train, groups_train = augment_data(X_train, y_train, groups_train)

# -----------------------------
# Model Definitions
# -----------------------------
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        w = torch.softmax(self.attn(x), dim=1)
        return torch.sum(w * x, dim=1)

class RNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, rnn_type="RNN", num_layers=1, dropout=0.0):
        super().__init__()
        if num_layers == 1:
            dropout = 0.0
        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[rnn_type]
        self.rnn = rnn_cls(input_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.attn(out)
        return self.fc(out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, nhead, dropout):
        super().__init__()
        enc = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=nhead, dropout=dropout,
            batch_first=True, layer_norm_eps=1e-5
        )
        self.encoder = nn.TransformerEncoder(enc, num_layers=1)

    def forward(self, x):
        return self.encoder(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0, nhead=None):
        super().__init__()
        if nhead is None:
            if hidden_dim % 8 == 0:
                nhead = hidden_dim // 8
            elif hidden_dim % 4 == 0:
                nhead = hidden_dim // 4
            else:
                nhead = 1

        self.input_proj = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim))
        self.pe = PositionalEncoding(hidden_dim)

        if num_layers == 1:
            self.stack = nn.Sequential(TransformerBlock(hidden_dim, nhead, 0.0))
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(TransformerBlock(hidden_dim, nhead, dropout), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])

        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pe(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

class MambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba = Mamba(d_model=dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return self.norm(self.mamba(x))

class MambaModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        if num_layers == 1:
            self.stack = nn.Sequential(*[MambaBlock(hidden_dim) for _ in range(num_layers)])
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(MambaBlock(hidden_dim), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.norm(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

def build_model(model_type, input_dim, hidden_dim, num_layers, dropout):
    if model_type == "Mamba":
        return MambaModel(input_dim, hidden_dim, num_layers, dropout)
    if model_type == "Transformer":
        return TransformerModel(input_dim, hidden_dim, num_layers, dropout)
    return RNNModel(input_dim, hidden_dim, model_type, num_layers, dropout)

# -----------------------------
# Utilities
# -----------------------------
def get_loader(X, y, batch_size, shuffle=True):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=shuffle)

# -----------------------------
# Save directory + best-model tracking
# -----------------------------
save_root = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/SavedModels_Stats_CNN_7"
os.makedirs(save_root, exist_ok=True)

global_best_rmse = np.inf
global_best_info = None
global_best_state_dict = None

# -----------------------------
# Train (Grid Search)
# -----------------------------
def run_grid_search():
    models = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]
    hidden_dims = [16, 32, 64, 128]
    dropouts = [0.0, 0.2]
    num_layers_list = [1, 2, 3]
    batch_sizes = [16, 32]
    lrs = [0.001, 0.01]
    weight_decays = [0, 1e-4]
    optimizers = ["Adam", "SGD"]
    epochs = 200

    results = {"cv_predictions": [], "summary": []}
    gkf = GroupKFold(n_splits=5)

    global global_best_rmse, global_best_info, global_best_state_dict

    for model_type in models:
        for hidden_dim in hidden_dims:
            for dropout in dropouts:
                for num_layers in num_layers_list:
                    for batch_size in batch_sizes:
                        for lr in lrs:
                            for wd in weight_decays:
                                for opt_name in optimizers:
                                    rmse_list, mae_list = [], []
                                    used_epochs_all = []

                                    # Best fold weights for this hyperparam combo are now chosen by **VAL LOSS**
                                    combo_best_val_loss = np.inf
                                    combo_best_state_dict = None

                                    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups_train)):
                                        model = build_model(
                                            model_type, X_train.shape[2],
                                            hidden_dim, num_layers, dropout
                                        ).to(device)

                                        optimizer_cls = torch.optim.Adam if opt_name == "Adam" else torch.optim.SGD
                                        optimizer = optimizer_cls(model.parameters(), lr=lr, weight_decay=wd)
                                        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5)
                                        criterion = nn.MSELoss()

                                        train_loader = get_loader(
                                            X_train[train_idx], y_train[train_idx],
                                            batch_size, shuffle=True
                                        )

                                        # Pre-make val tensors once
                                        val_x_t = torch.tensor(X_train[val_idx], dtype=torch.float32).to(device)
                                        val_y_t = torch.tensor(y_train[val_idx], dtype=torch.float32).view(-1, 1).to(device)

                                        best_val_loss = float("inf")
                                        wait = 0
                                        used_epochs = 0
                                        best_state_dict_fold = None

                                        for epoch in range(epochs):
                                            model.train()
                                            for xb, yb in train_loader:
                                                xb, yb = xb.to(device), yb.to(device)
                                                optimizer.zero_grad()
                                                pred = model(xb)
                                                loss = criterion(pred, yb)
                                                if torch.isnan(loss) or torch.isinf(loss):
                                                    continue
                                                loss.backward()
                                                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                                                optimizer.step()

                                            # ---- VALIDATION LOSS (early stop + scheduler) ----
                                            model.eval()
                                            with torch.no_grad():
                                                val_pred = model(val_x_t)
                                                val_loss = criterion(val_pred, val_y_t).item()

                                            scheduler.step(val_loss)
                                            used_epochs += 1

                                            if val_loss < best_val_loss:
                                                best_val_loss = val_loss
                                                wait = 0
                                                best_state_dict_fold = {k: v.detach().cpu().clone()
                                                                       for k, v in model.state_dict().items()}
                                            else:
                                                wait += 1
                                                if wait >= 10:
                                                    break

                                        used_epochs_all.append(used_epochs)

                                        # Restore best fold weights
                                        if best_state_dict_fold is not None:
                                            model.load_state_dict(best_state_dict_fold)

                                        # Predict (scaled) on fold val -> inverse transform to original y domain
                                        model.eval()
                                        with torch.no_grad():
                                            preds_scaled = model(val_x_t).squeeze().detach().cpu().numpy()

                                        preds = scaler_y.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
                                        y_true = scaler_y.inverse_transform(y_train[val_idx].reshape(-1, 1)).flatten()

                                        results["cv_predictions"].append({
                                            "model": model_type,
                                            "hidden_dim": hidden_dim,
                                            "dropout": dropout,
                                            "num_layers": num_layers,
                                            "batch_size": batch_size,
                                            "lr": lr,
                                            "weight_decay": wd,
                                            "optimizer": opt_name,
                                            "fold": fold,
                                            "y_true": y_true.tolist(),
                                            "y_pred": preds.tolist()
                                        })

                                        rmse = float(np.sqrt(mean_squared_error(y_true, preds)))
                                        mae = float(mean_absolute_error(y_true, preds))
                                        rmse_list.append(rmse)
                                        mae_list.append(mae)

                                        # Best fold weights for this combo chosen by best_val_loss
                                        if best_val_loss < combo_best_val_loss and best_state_dict_fold is not None:
                                            combo_best_val_loss = best_val_loss
                                            combo_best_state_dict = best_state_dict_fold

                                    rmse_mean = float(np.mean(rmse_list)) if len(rmse_list) else float("inf")
                                    rmse_std  = float(np.std(rmse_list))  if len(rmse_list) else float("nan")
                                    mae_mean  = float(np.mean(mae_list))  if len(mae_list)  else float("inf")
                                    mae_std   = float(np.std(mae_list))   if len(mae_list)  else float("nan")

                                    results["summary"].append({
                                        "model": model_type, "hidden_dim": hidden_dim, "dropout": dropout,
                                        "num_layers": num_layers, "batch_size": batch_size, "lr": lr,
                                        "weight_decay": wd, "optimizer": opt_name, "epochs": epochs,
                                        "rmse_mean": rmse_mean, "rmse_std": rmse_std,
                                        "mae_mean": mae_mean, "mae_std": mae_std,
                                        "used_epochs_mean": float(np.mean(used_epochs_all)) if used_epochs_all else float("nan"),
                                        "used_epochs_std": float(np.std(used_epochs_all))  if used_epochs_all else float("nan"),
                                        "combo_best_val_loss": float(combo_best_val_loss),
                                    })

                                    print(f"{model_type} HD={hidden_dim} DO={dropout} NL={num_layers} EP={epochs} OPT={opt_name} "
                                          f"=> RMSE: {rmse_mean:.4f} ± {rmse_std:.4f}, MAE: {mae_mean:.4f} ± {mae_std:.4f}")

                                    # Save global best artifacts by rmse_mean
                                    if rmse_mean < global_best_rmse and combo_best_state_dict is not None:
                                        global_best_rmse = rmse_mean
                                        global_best_info = {
                                            "model": model_type,
                                            "hidden_dim": int(hidden_dim),
                                            "dropout": float(dropout),
                                            "num_layers": int(num_layers),
                                            "batch_size": int(batch_size),
                                            "lr": float(lr),
                                            "weight_decay": float(wd),
                                            "optimizer": opt_name,
                                            "epochs": int(epochs),
                                            "rmse_mean": float(rmse_mean),
                                            "mae_mean": float(mae_mean),
                                            "input_dim": int(X_train.shape[2]),
                                            "timesteps": int(X_train.shape[1]),  # 7
                                            "target": target,
                                        }
                                        global_best_state_dict = combo_best_state_dict

                                        print("\n🏆 NEW GLOBAL BEST (by RMSE) FOUND:", global_best_info)

                                        torch.save(
                                            {"state_dict": global_best_state_dict, "config": global_best_info},
                                            os.path.join(save_root, "best_model_state.pt")
                                        )
                                        with open(os.path.join(save_root, "best_model_config.json"), "w") as f:
                                            json.dump(global_best_info, f, indent=2)

                                        joblib.dump(scaler_x, os.path.join(save_root, "scaler_x.joblib"))
                                        joblib.dump(scaler_y, os.path.join(save_root, "scaler_y.joblib"))

    return pd.DataFrame(results["summary"]), results["cv_predictions"]

# -----------------------------
# Run Training
# -----------------------------
results_df, cv_predictions = run_grid_search()

results_df.to_csv(
    "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_grid_search_results(Stats_CNN)_7.csv",
    index=False
)
print("✅ Grid search completed and results saved.")

with open(
    "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_cv_predictions(Stats_CNN)_7.json",
    "w"
) as f:
    json.dump(cv_predictions, f)
print(f"✅ CV predictions saved: {len(cv_predictions)} records.")
print("✅ Best model + scalers saved to:", save_root)

# ======================================================================================
# TEST + VALIDATION (Best saved model, NO retraining)
# ======================================================================================
def load_best_artifacts(save_root):
    ckpt_path = os.path.join(save_root, "best_model_state.pt")
    cfg_path  = os.path.join(save_root, "best_model_config.json")
    sx_path   = os.path.join(save_root, "scaler_x.joblib")
    sy_path   = os.path.join(save_root, "scaler_y.joblib")

    for p in [ckpt_path, cfg_path, sx_path, sy_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing best artifact: {p}")

    with open(cfg_path, "r") as f:
        cfg = json.load(f)

    sx = joblib.load(sx_path)
    sy = joblib.load(sy_path)

    state = torch.load(ckpt_path, map_location="cpu")
    state_dict = state["state_dict"] if isinstance(state, dict) and "state_dict" in state else state
    return cfg, sx, sy, state_dict

cfg_best, sx_best, sy_best, state_dict_best = load_best_artifacts(save_root)

# Build model with saved input_dim to avoid any mismatch surprises
best_model = build_model(
    cfg_best["model"],
    input_dim=int(cfg_best.get("input_dim", X_test.shape[2])),
    hidden_dim=int(cfg_best["hidden_dim"]),
    num_layers=int(cfg_best["num_layers"]),
    dropout=float(cfg_best["dropout"])
).to(device)

best_model.load_state_dict(state_dict_best)
best_model.eval()

batch_size_best = int(cfg_best.get("batch_size", 32))

# --------------------------
# TEST on held-out test split
# --------------------------
test_loader = get_loader(X_test, y_test, batch_size_best, shuffle=False)

preds_scaled = []
with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(device)
        pred = best_model(xb).detach().cpu().numpy().reshape(-1)
        preds_scaled.append(pred)

y_pred_scaled = np.concatenate(preds_scaled)
y_pred_test = sy_best.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_true_test = sy_best.inverse_transform(y_test.reshape(-1, 1)).flatten()

rmse_test = float(np.sqrt(mean_squared_error(y_true_test, y_pred_test)))
mae_test  = float(mean_absolute_error(y_true_test, y_pred_test))

print("\n✅ TEST (held-out split)")
print(f"RMSE: {rmse_test:.4f} | MAE: {mae_test:.4f} | N={len(y_true_test)}")

test_results_df = pd.DataFrame([{
    "model": cfg_best["model"],
    "rmse_test": rmse_test,
    "mae_test": mae_test,
    "N_test": int(len(y_true_test)),
    "hidden_dim": int(cfg_best["hidden_dim"]),
    "dropout": float(cfg_best["dropout"]),
    "num_layers": int(cfg_best["num_layers"]),
    "batch_size": int(cfg_best.get("batch_size", -1)),
    "lr": float(cfg_best.get("lr", np.nan)),
    "weight_decay": float(cfg_best.get("weight_decay", np.nan)),
    "optimizer": cfg_best.get("optimizer", "NA"),
    "epochs": int(cfg_best.get("epochs", -1)),
}])

test_results_csv = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_best_model_test_results(Stats_CNN)_7.csv"
test_preds_json  = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig7_RegressionModels/Regression_best_model_test_predictions(Stats_CNN)_7.json"

test_results_df.to_csv(test_results_csv, index=False)
with open(test_preds_json, "w") as f:
    json.dump([{
        "model": cfg_best["model"],
        "y_true": y_true_test.tolist(),
        "y_pred": y_pred_test.tolist()
    }], f)

print("✅ Test performance saved:", test_results_csv)
print("✅ Test predictions saved:", test_preds_json)

# --------------------------
# VALIDATION on V3 Sheet15 (NO retrain)
# Fix: V3 uses OSI_V3_12th_avg instead of OSI_V2_12th_avg
# --------------------------
v3_xlsx  = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"
v3_sheet = "Sheet15"
df_val = pd.read_excel(v3_xlsx, sheet_name=v3_sheet)

train_target = target              # "OSI_V2_12th_avg"
val_target   = "OSI_V3_12th_avg"   # present in V3

if train_target in df_val.columns:
    label_col = train_target
elif val_target in df_val.columns:
    print(f"NOTE: '{train_target}' not found in {v3_sheet}; using '{val_target}' instead.")
    label_col = val_target
else:
    raise ValueError(f"Missing target in {v3_sheet}: neither '{train_target}' nor '{val_target}' exists.")

# Rebuild X like training
all_feature_cols = [c for tw in range(1, 7) for c in tw_features[tw]]

missing_feats = [c for c in all_feature_cols if c not in df_val.columns]
if missing_feats:
    raise ValueError(f"Missing {len(missing_feats)} feature columns in {v3_sheet}. First 30: {missing_feats[:30]}")

df_val_model = df_val.dropna(subset=[label_col] + all_feature_cols).copy()
print(f"\n✅ VALIDATION rows after dropna ({v3_sheet}): {len(df_val_model)}")
if len(df_val_model) == 0:
    raise ValueError("No rows left after dropna in validation; cannot run validation.")

X_list = [df_val_model[tw_features[tw]].to_numpy(dtype=np.float32) for tw in range(1, 7)]
X6_val = np.stack(X_list, axis=1)  # (N, 6, D)
delta_val = (X6_val[:, -1, :] - X6_val[:, 0, :])[:, np.newaxis, :]
X7_val = np.concatenate([X6_val, delta_val], axis=1)  # (N, 7, D)

y_true_val = df_val_model[label_col].to_numpy(dtype=np.float32)

N, T, D = X7_val.shape
# scale with TRAIN scaler_x (do NOT refit)
X7_val_scaled = sx_best.transform(X7_val.reshape(-1, D)).reshape(N, T, D)

Xv_t = torch.tensor(X7_val_scaled, dtype=torch.float32).to(device)
with torch.no_grad():
    yv_pred_scaled = best_model(Xv_t).detach().cpu().numpy().reshape(-1, 1)

y_pred_val = sy_best.inverse_transform(yv_pred_scaled).flatten()

rmse_val = float(np.sqrt(mean_squared_error(y_true_val, y_pred_val)))
mae_val  = float(mean_absolute_error(y_true_val, y_pred_val))

print(f"\n✅ VALIDATION (V3 {v3_sheet}) | label_col={label_col}")
print(f"RMSE: {rmse_val:.4f} | MAE: {mae_val:.4f} | N={len(y_true_val)}")

val_out_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/20251105_TBME_Submission/20260301_Major_Revision"
os.makedirs(val_out_dir, exist_ok=True)

val_results_csv = os.path.join(val_out_dir, "Regression_best_model_validation_results(Stats_CNN)_7.csv")
val_preds_json  = os.path.join(val_out_dir, "Regression_best_model_validation_predictions(Stats_CNN)_7.json")

pd.DataFrame([{
    "model": cfg_best["model"],
    "sheet": v3_sheet,
    "label_col_used": label_col,
    "rmse_val": rmse_val,
    "mae_val": mae_val,
    "N_val": int(len(y_true_val)),
    "hidden_dim": int(cfg_best["hidden_dim"]),
    "dropout": float(cfg_best["dropout"]),
    "num_layers": int(cfg_best["num_layers"]),
    "batch_size": int(cfg_best.get("batch_size", -1)),
    "lr": float(cfg_best.get("lr", np.nan)),
    "weight_decay": float(cfg_best.get("weight_decay", np.nan)),
    "optimizer": cfg_best.get("optimizer", "NA"),
    "epochs": int(cfg_best.get("epochs", -1)),
}]).to_csv(val_results_csv, index=False)

with open(val_preds_json, "w") as f:
    json.dump([{
        "model": cfg_best["model"],
        "sheet": v3_sheet,
        "label_col_used": label_col,
        "y_true": y_true_val.tolist(),
        "y_pred": y_pred_val.tolist()
    }], f)

print("✅ Validation performance saved:", val_results_csv)
print("✅ Validation predictions saved:", val_preds_json)


# Classification (RNN, GRU, LSTM, Mamba) Models

### OSI Features Only (Classification) -- training + test + validation

In [ ]:
base_template = [
    "OSI_mean_TW{}", "OSI_std_TW{}"
]

tw_features = {
    i: [f.format(i) for f in base_template]
    for i in range(1, 7)
}

### Vent_Time Features Only (Classification) -- training + test + validation

In [ ]:
base_template = [
    "PIP_mean_TW{}", "PIP_std_TW{}",
    "PEEP_mean_TW{}", "PEEP_std_TW{}", "TV_mean_TW{}(mL/Kg)", "TV_std_TW{}(mL/Kg)",
    "Avg_NegFlowDur_TW{}", "Std_NegFlowDur_TW{}", "Avg_PeakInterval_TW{}", "Std_PeakInterval_TW{}"
]

tw_features = {
    i: [f.format(i) for f in base_template]
    for i in range(1, 7)
}

### Vent_Freq Features Only (Classification) -- training + test + validation

In [ ]:
base_template = [
    "PIP_mean_TW{}", "PIP_std_TW{}",
    "PEEP_mean_TW{}", "PEEP_std_TW{}", "TV_mean_TW{}(mL/Kg)", "TV_std_TW{}(mL/Kg)"
]

tw_features = {
    i: [f.format(i) for f in base_template] + [f"w{i}_SubBandEnergy_row{j}" for j in range(1, 17)]
    for i in range(1, 7)
}

### Vent_All Features Only (Classification) -- training + test + validation

In [ ]:
base_template = [
    "PIP_mean_TW{}", "PIP_std_TW{}",
    "PEEP_mean_TW{}", "PEEP_std_TW{}", "TV_mean_TW{}(mL/Kg)", "TV_std_TW{}(mL/Kg)",
    "Avg_NegFlowDur_TW{}", "Std_NegFlowDur_TW{}", "Avg_PeakInterval_TW{}", "Std_PeakInterval_TW{}"
]

tw_features = {
    i: [f.format(i) for f in base_template] + [f"w{i}_SubBandEnergy_row{j}" for j in range(1, 17)]
    for i in range(1, 7)
}

### OSI+Vent_Time Features Only (Classification) -- training + test + validation

In [ ]:
base_template = [
    "OSI_mean_TW{}", "OSI_std_TW{}", "PIP_mean_TW{}", "PIP_std_TW{}",
    "PEEP_mean_TW{}", "PEEP_std_TW{}", "TV_mean_TW{}(mL/Kg)", "TV_std_TW{}(mL/Kg)",
    "Avg_NegFlowDur_TW{}", "Std_NegFlowDur_TW{}", "Avg_PeakInterval_TW{}", "Std_PeakInterval_TW{}"
]

tw_features = {
    i: [f.format(i) for f in base_template]
    for i in range(1, 7)
}

### OSI+Vent_Freq Features Only (Classification) -- training + test + validation

In [ ]:
base_template = [
    "OSI_mean_TW{}", "OSI_std_TW{}", "PIP_mean_TW{}", "PIP_std_TW{}",
    "PEEP_mean_TW{}", "PEEP_std_TW{}", "TV_mean_TW{}(mL/Kg)", "TV_std_TW{}(mL/Kg)"
]

tw_features = {
    i: [f.format(i) for f in base_template] + [f"w{i}_SubBandEnergy_row{j}" for j in range(1, 17)]
    for i in range(1, 7)
}

### OSI+Vent_All Features Only (Classification) -- training + test + validation (_8)

In [ ]:
base_template = [
    "OSI_mean_TW{}", "OSI_std_TW{}", "PIP_mean_TW{}", "PIP_std_TW{}",
    "PEEP_mean_TW{}", "PEEP_std_TW{}", "TV_mean_TW{}(mL/Kg)", "TV_std_TW{}(mL/Kg)",
    "Avg_NegFlowDur_TW{}", "Std_NegFlowDur_TW{}", "Avg_PeakInterval_TW{}", "Std_PeakInterval_TW{}"
]

tw_features = {
    i: [f.format(i) for f in base_template] + [f"w{i}_SubBandEnergy_row{j}" for j in range(1, 17)]
    for i in range(1, 7)
}

In [ ]:
#!/usr/bin/env python3
# ======================================================================================
# TBME Major Revision Utilities (OSI + VentALL non-CNN feature set)
# - Save BEST config+weights per model type (RNN/LSTM/GRU/Transformer/Mamba)
# - Save TRAINING CV predictions per config (for downstream ROC/PRC/F1 plots)
# - Evaluate BEST per model on TEST + TEMPORAL VALIDATION and save y_true/y_prob
# - Learning curve (20/40/60/80/100% of training patients)
# - Bootstrap CIs + Sens@fixed specificity + Calibration metrics/curve
#
# NOTE: OSI+VentALL (stats + optional wavelet) => NO PCA64. Keeps the rest the same.
# ======================================================================================

import os
import math
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, auc, f1_score,
    roc_curve, brier_score_loss
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from mamba_ssm import Mamba

# ======================================================================================
# ================================ USER CONTROLS ======================================
# ======================================================================================

RUN_GRID_SEARCH       = True
RUN_EVAL_BEST_MODELS  = True
RUN_LEARNING_CURVES   = True
RUN_PCA_SENSITIVITY   = False   # non-CNN => disabled
RUN_BOOTSTRAP_CI_EVAL = True

# Feature toggles (still non-CNN)
USE_WAVELET = True  # set False for stats-only (OSI+Vent stats w/o wavelets)

# Learning curves
LEARNING_MODELS    = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]  # or ["Mamba"]
LEARNING_FRACTIONS = [0.2, 0.4, 0.6, 0.8, 1.0]
LEARNING_SEEDS     = [0]

# Bootstrap settings
BOOTSTRAP_B   = 2000
SPEC_TARGETS  = [0.90, 0.95]
CALIB_BINS    = 10

# Stratified sampling inside CV
SUBSAMPLE_RATIO     = 0.5
SUBSAMPLE_POS_RATIO = 1/3

# ======================================================================================
# ================================ PATHS / I/O ========================================
# ======================================================================================

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Training data
xlsx_train  = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1V2_df.xlsx"
sheet_train = "Sheet5"

# Temporal validation data
xlsx_val  = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"
sheet_val = "Sheet15"

# Output roots
out_root_models = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels"
save_root = os.path.join(out_root_models, "SavedModels_OSIandVentALL_8")
os.makedirs(save_root, exist_ok=True)

out_revision_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/20251105_TBME_Submission/20260301_Major_Revision"
os.makedirs(out_revision_dir, exist_ok=True)

# ======================================================================================
# ================================= DATA LOADING ======================================
# ======================================================================================

target_col = "OSI_V2_12th_avg"
threshold = 7.5

# ----- Stats features per TW -----
base_template = [
    "OSI_mean_TW{}", "OSI_std_TW{}",
    "PIP_mean_TW{}", "PIP_std_TW{}",
    "PEEP_mean_TW{}", "PEEP_std_TW{}",
    "TV_mean_TW{}(mL/Kg)", "TV_std_TW{}(mL/Kg)",
    "Avg_NegFlowDur_TW{}", "Std_NegFlowDur_TW{}",
    "Avg_PeakInterval_TW{}", "Std_PeakInterval_TW{}",
]

def wavelet_cols_for_tw(tw: int, n_bands: int = 16):
    return [f"w{tw}_SubBandEnergy_row{j}" for j in range(1, n_bands + 1)]

# TW -> list of columns
tw_features = {}
for tw in range(1, 7):
    cols = [tmpl.format(tw) for tmpl in base_template]
    if USE_WAVELET:
        cols += wavelet_cols_for_tw(tw, n_bands=16)
    tw_features[tw] = cols

def build_sequences_from_df_noncnn(
    df: pd.DataFrame,
    label_col: str,
    tw_features: dict,
    threshold: float,
    group_col: str = "ResearchID",
):
    """
    Build sequences:
      X6: (N, 6, D) where D = len(tw_features[tw]) (stats [+ wavelets])
      y : (N,) binary label
      groups: (N,) patient/group id
    """
    # sanity: all required feature cols must exist
    all_required_cols = sorted({c for tw in range(1, 7) for c in tw_features[tw]})
    missing_global = [c for c in all_required_cols if c not in df.columns]
    if missing_global:
        raise ValueError(
            f"Missing {len(missing_global)} required feature columns in TRAIN sheet. "
            f"First 30: {missing_global[:30]}"
        )

    X, y, groups = [], [], []
    for idx, row in df.iterrows():
        if pd.isna(row.get(label_col, np.nan)):
            continue

        seq = []
        ok = True
        for tw in range(1, 7):
            cols = tw_features[tw]
            vals = row[cols]
            if vals.isnull().any():
                ok = False
                break
            seq.append(vals.to_numpy(dtype=np.float32))

        if not ok:
            continue

        X.append(seq)
        y.append(1 if float(row[label_col]) >= threshold else 0)
        groups.append(row[group_col] if group_col in df.columns else idx)

    X = np.asarray(X, dtype=np.float32)       # (N,6,D)
    y = np.asarray(y, dtype=np.float32)       # (N,)
    groups = np.asarray(groups)
    return X, y, groups

def add_delta_window(X6: np.ndarray) -> np.ndarray:
    # X6: (N, 6, D) -> X7: (N, 7, D) by appending (TW6 - TW1)
    delta = (X6[:, -1, :] - X6[:, 0, :])[:, np.newaxis, :]
    return np.concatenate([X6, delta], axis=1)

# Load train
df_train = pd.read_excel(xlsx_train, sheet_name=sheet_train)
X6, y, groups = build_sequences_from_df_noncnn(df_train, target_col, tw_features, threshold)
X = add_delta_window(X6)

print(f"✅ Total usable samples: {len(X)} | Positives: {int(y.sum())} | Negatives: {int((1-y).sum())}")
print(f"✅ Feature dim per TW: {X.shape[2]}  (USE_WAVELET={USE_WAVELET})")

# ======================================================================================
# ============================= PREPROCESSING HELPERS =================================
# ======================================================================================

def fit_scaler_on_train(X_train: np.ndarray) -> StandardScaler:
    scaler = StandardScaler()
    N, T, D = X_train.shape
    scaler.fit(X_train.reshape(-1, D))
    return scaler

def apply_scaler(scaler: StandardScaler, X_in: np.ndarray) -> np.ndarray:
    N, T, D = X_in.shape
    return scaler.transform(X_in.reshape(-1, D)).reshape(N, T, D)

def augment_data(X_in, y_in, groups_in, noise_std=0.005, seed=0):
    rng = np.random.RandomState(seed)
    noise = rng.normal(0, noise_std, X_in.shape).astype(np.float32)
    return (
        np.concatenate([X_in, X_in + noise]),
        np.concatenate([y_in, y_in]),
        np.concatenate([groups_in, groups_in]),
    )

# ======================================================================================
# ================================== MODELS ===========================================
# ======================================================================================

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        w = torch.softmax(self.attn(x), dim=1)  # (B,T,1)
        return torch.sum(w * x, dim=1)          # (B,H)

class RNNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, rnn_type="RNN", num_layers=1, dropout=0.0):
        super().__init__()
        if num_layers == 1:
            dropout = 0.0
        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[rnn_type]
        self.rnn = rnn_cls(input_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.attn(out)
        return self.fc(out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, nhead, dropout):
        super().__init__()
        enc = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=nhead, dropout=dropout,
            batch_first=True, layer_norm_eps=1e-5
        )
        self.encoder = nn.TransformerEncoder(enc, num_layers=1)

    def forward(self, x):
        return self.encoder(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0, nhead=None):
        super().__init__()
        if nhead is None:
            if hidden_dim % 8 == 0:
                nhead = max(1, hidden_dim // 8)
            elif hidden_dim % 4 == 0:
                nhead = max(1, hidden_dim // 4)
            else:
                nhead = 1

        self.input_proj = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim))
        self.pe = PositionalEncoding(hidden_dim)

        if num_layers == 1:
            self.stack = nn.Sequential(TransformerBlock(hidden_dim, nhead, 0.0))
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(TransformerBlock(hidden_dim, nhead, dropout), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])

        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pe(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

class MambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba = Mamba(d_model=dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return self.norm(self.mamba(x))

class MambaClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        if num_layers == 1:
            self.stack = nn.Sequential(MambaBlock(hidden_dim))
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(MambaBlock(hidden_dim), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.norm(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

def build_model(model_type, input_dim, hidden_dim, num_layers, dropout):
    if model_type == "Mamba":
        return MambaClassifier(input_dim, hidden_dim, num_layers, dropout)
    if model_type == "Transformer":
        return TransformerModel(input_dim, hidden_dim, num_layers, dropout)
    return RNNClassifier(input_dim, hidden_dim, model_type, num_layers, dropout)

def get_loader(X_in, y_in, batch_size, shuffle=True):
    X_tensor = torch.tensor(X_in, dtype=torch.float32)
    y_tensor = torch.tensor(y_in, dtype=torch.float32).view(-1, 1)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=shuffle)

def auprc_score(y_true, y_prob):
    p, r, _ = precision_recall_curve(y_true, y_prob)
    return auc(r, p)

# ======================================================================================
# ============================ CLINICAL METRICS HELPERS ===============================
# ======================================================================================

def sens_at_fixed_spec(y_true, y_prob, spec_target=0.90):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    spec = 1.0 - fpr
    idx = np.where(spec >= spec_target)[0]
    if len(idx) == 0:
        return np.nan, np.nan
    best_i = idx[np.argmax(tpr[idx])]
    return float(tpr[best_i]), float(thr[best_i])

def calibration_curve_bins(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    rows = []
    for b in range(n_bins):
        mask = (bin_ids == b)
        if mask.sum() == 0:
            rows.append((b, 0, np.nan, np.nan))
        else:
            rows.append((b, int(mask.sum()), float(np.mean(y_prob[mask])), float(np.mean(y_true[mask]))))
    return pd.DataFrame(rows, columns=["bin", "count", "mean_pred", "frac_pos"])

def eval_point_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    out = {}
    out["AUC"] = float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan
    out["AUPRC"] = float(auprc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan
    out["F1"] = float(f1_score(y_true, y_pred)) if len(np.unique(y_true)) > 1 else np.nan
    out["Brier"] = float(brier_score_loss(y_true, y_prob))
    return out

def bootstrap_ci_metrics(y_true, y_prob, B=2000, seed=0, spec_targets=(0.90,)):
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    n = len(y_true)

    point = eval_point_metrics(y_true, y_prob, threshold=0.5)
    for s in spec_targets:
        sens, thr = sens_at_fixed_spec(y_true, y_prob, spec_target=s)
        point[f"Sens@Spec{int(s*100)}"] = sens
        point[f"Thr@Spec{int(s*100)}"] = thr

    dist = {k: [] for k in point.keys()}
    valid = 0

    for _ in range(B):
        idx = rng.randint(0, n, size=n)
        yt = y_true[idx]
        yp = y_prob[idx]
        if len(np.unique(yt)) < 2:
            continue
        valid += 1
        m = eval_point_metrics(yt, yp, threshold=0.5)
        for s in spec_targets:
            sens, thr = sens_at_fixed_spec(yt, yp, spec_target=s)
            m[f"Sens@Spec{int(s*100)}"] = sens
            m[f"Thr@Spec{int(s*100)}"] = thr
        for k, v in m.items():
            dist[k].append(v)

    rows = []
    for k, v_point in point.items():
        arr = np.asarray(dist[k], dtype=float)
        if arr.size == 0:
            rows.append((k, v_point, np.nan, np.nan, valid))
        else:
            lo = float(np.nanpercentile(arr, 2.5))
            hi = float(np.nanpercentile(arr, 97.5))
            rows.append((k, v_point, lo, hi, valid))
    return pd.DataFrame(rows, columns=["metric", "point", "ci_low", "ci_high", "n_valid_bootstrap"])

# ======================================================================================
# ============================== GRID SEARCH TRAINING =================================
# ======================================================================================

def stratified_sample(y_train, ratio=0.5, pos_ratio=1/3, seed=0):
    rng = np.random.RandomState(seed)
    y_flat = y_train.flatten()
    idx_0 = np.where(y_flat == 0)[0]
    idx_1 = np.where(y_flat == 1)[0]

    total = int(len(y_flat) * ratio)
    n1 = int(total * pos_ratio)
    n0 = total - n1
    if len(idx_0) < n0 or len(idx_1) < n1:
        return None

    idx = np.concatenate([
        rng.choice(idx_0, n0, replace=False),
        rng.choice(idx_1, n1, replace=False)
    ])
    rng.shuffle(idx)
    return idx

def train_one_config_cv(
    model_type, X_train, y_train, groups_train,
    hidden_dim, dropout, num_layers, batch_size, lr, wd, optimizer_name,
    epochs=200, n_repeats=5, n_splits=3, early_stop_patience=10,
    save_cv_records=True
):
    expected_metrics = n_repeats * n_splits
    all_aucs, all_auprcs, all_f1s = [], [], []
    used_epochs_all = []
    combo_best_val_loss = np.inf
    combo_best_state = None
    cv_records = []

    for rep in range(n_repeats):
        sampled_idx = stratified_sample(
            y_train, ratio=SUBSAMPLE_RATIO, pos_ratio=SUBSAMPLE_POS_RATIO, seed=rep
        )
        if sampled_idx is None:
            continue

        X_sub = X_train[sampled_idx]
        y_sub = y_train[sampled_idx]
        g_sub = groups_train[sampled_idx]

        gkf = GroupKFold(n_splits=n_splits)
        for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_sub, y_sub, g_sub)):

            model = build_model(model_type, X_train.shape[2], hidden_dim, num_layers, dropout).to(device)
            opt_cls = torch.optim.Adam if optimizer_name == "Adam" else torch.optim.SGD
            optimizer = opt_cls(model.parameters(), lr=lr, weight_decay=wd)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5)
            criterion = nn.BCEWithLogitsLoss()

            loader = get_loader(X_sub[tr_idx], y_sub[tr_idx], batch_size, shuffle=True)

            best_loss = float("inf")
            wait = 0
            used_epochs = 0
            best_state_fold = None

            for _ in range(epochs):
                model.train()
                losses = []
                for xb, yb in loader:
                    xb, yb = xb.to(device), yb.to(device)
                    optimizer.zero_grad()
                    logits = model(xb)
                    loss = criterion(logits, yb)
                    if torch.isnan(loss) or torch.isinf(loss):
                        continue
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    losses.append(loss.item())

                avg_loss = float(np.mean(losses)) if len(losses) else np.inf
                scheduler.step(avg_loss)
                used_epochs += 1

                if avg_loss < best_loss:
                    best_loss = avg_loss
                    wait = 0
                    best_state_fold = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                else:
                    wait += 1
                    if wait >= early_stop_patience:
                        break

            used_epochs_all.append(used_epochs)
            if best_state_fold is not None:
                model.load_state_dict(best_state_fold)

            model.eval()
            with torch.no_grad():
                val_x = torch.tensor(X_sub[val_idx], dtype=torch.float32).to(device)
                logits = model(val_x).cpu().flatten()
                prob = torch.sigmoid(logits).numpy()
                yt = y_sub[val_idx].flatten().astype(int)

            valid_mask = np.isfinite(prob)
            if valid_mask.sum() == 0:
                continue
            prob = prob[valid_mask]
            yt = yt[valid_mask]

            if len(np.unique(yt)) < 2:
                continue

            all_aucs.append(float(roc_auc_score(yt, prob)))
            all_auprcs.append(float(auprc_score(yt, prob)))
            all_f1s.append(float(f1_score(yt, (prob >= 0.5).astype(int))))

            if save_cv_records:
                cv_records.append({
                    "model": model_type,
                    "hidden_dim": int(hidden_dim),
                    "dropout": float(dropout),
                    "num_layers": int(num_layers),
                    "batch_size": int(batch_size),
                    "lr": float(lr),
                    "weight_decay": float(wd),
                    "optimizer": optimizer_name,
                    "repeat": int(rep),
                    "fold": int(fold),
                    "y_true": yt.tolist(),
                    "y_prob": prob.tolist()
                })

            if best_loss < combo_best_val_loss and best_state_fold is not None:
                combo_best_val_loss = best_loss
                combo_best_state = best_state_fold

    is_complete = (len(all_aucs) == expected_metrics)
    summary = {
        "AUC_mean": float(np.mean(all_aucs)) if is_complete else np.nan,
        "AUC_std":  float(np.std(all_aucs))  if is_complete else np.nan,
        "AUPRC_mean": float(np.mean(all_auprcs)) if is_complete else np.nan,
        "AUPRC_std":  float(np.std(all_auprcs))  if is_complete else np.nan,
        "F1_mean": float(np.mean(all_f1s)) if is_complete else np.nan,
        "F1_std":  float(np.std(all_f1s))  if is_complete else np.nan,
        "n_metrics": int(len(all_aucs)),
        "expected_metrics": int(expected_metrics),
        "used_epochs_mean": float(np.mean(used_epochs_all)) if used_epochs_all else np.nan,
        "used_epochs_std": float(np.std(used_epochs_all)) if used_epochs_all else np.nan,
        "best_val_loss": float(combo_best_val_loss) if np.isfinite(combo_best_val_loss) else np.inf,
        "has_state": combo_best_state is not None
    }
    return summary, combo_best_state, cv_records

# ======================================================================================
# ================================ TRAIN/TEST SPLIT ===================================
# ======================================================================================

X_train_raw, X_test_raw, y_train, y_test, groups_train, groups_test = train_test_split(
    X, y, groups, test_size=0.2, random_state=42, stratify=y
)
print(f"Train N={len(X_train_raw)} | Test N={len(X_test_raw)}")

scaler_x = fit_scaler_on_train(X_train_raw)
X_train = apply_scaler(scaler_x, X_train_raw)
X_test  = apply_scaler(scaler_x, X_test_raw)

X_train_aug, y_train_aug, groups_train_aug = augment_data(X_train, y_train, groups_train, seed=0)
print(f"Train after augmentation N={len(X_train_aug)}")

# ======================================================================================
# ============================ GRID SEARCH + SAVE BEST PER MODEL =======================
# ======================================================================================

model_types = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]

best_per_model = {m: {"auc": -np.inf, "cfg": None, "state": None} for m in model_types}
grid_summary_rows = []
all_cv_predictions = []

FEATURE_REPR = "OSI_VENT_ALL_WAVELET" if USE_WAVELET else "OSI_VENT_ALL_STATS"

if RUN_GRID_SEARCH:
    epochs = 200
    n_repeats = 5
    n_splits = 3

    for model_type in model_types:
        for hidden_dim in [16, 32, 64, 128]:
            for dropout in [0.0, 0.2]:
                for num_layers in [1, 2, 3]:
                    for batch_size in [16, 32]:
                        for lr in [0.001, 0.01]:
                            for wd in [0, 1e-4]:
                                for optimizer_name in ["Adam", "SGD"]:

                                    summary, best_state, cv_records = train_one_config_cv(
                                        model_type,
                                        X_train_aug, y_train_aug, groups_train_aug,
                                        hidden_dim, dropout, num_layers, batch_size, lr, wd, optimizer_name,
                                        epochs=epochs, n_repeats=n_repeats, n_splits=n_splits,
                                        save_cv_records=True
                                    )

                                    all_cv_predictions.extend(cv_records)

                                    row = {
                                        "model": model_type,
                                        "hidden_dim": hidden_dim,
                                        "dropout": dropout,
                                        "num_layers": num_layers,
                                        "batch_size": batch_size,
                                        "lr": lr,
                                        "weight_decay": wd,
                                        "optimizer": optimizer_name,
                                        "epochs": epochs,
                                        "repeats": n_repeats,
                                        "folds": n_splits,
                                        "feature_repr": FEATURE_REPR,
                                        **summary
                                    }
                                    grid_summary_rows.append(row)

                                    auc_mean = row["AUC_mean"]
                                    if np.isfinite(auc_mean) and auc_mean > best_per_model[model_type]["auc"] and summary["has_state"]:
                                        best_per_model[model_type]["auc"] = float(auc_mean)
                                        best_per_model[model_type]["cfg"] = {
                                            "model": model_type,
                                            "hidden_dim": int(hidden_dim),
                                            "dropout": float(dropout),
                                            "num_layers": int(num_layers),
                                            "batch_size": int(batch_size),
                                            "lr": float(lr),
                                            "weight_decay": float(wd),
                                            "optimizer": optimizer_name,
                                            "epochs": int(epochs),
                                            "repeats": int(n_repeats),
                                            "folds": int(n_splits),
                                            "threshold": float(threshold),
                                            "target_col": target_col,
                                            "timesteps": int(X_train_aug.shape[1]),
                                            "input_dim": int(X_train_aug.shape[2]),
                                            "feature_repr": FEATURE_REPR,
                                            "pca_components": None,
                                            "use_wavelet": bool(USE_WAVELET),
                                        }
                                        best_per_model[model_type]["state"] = best_state
                                        print(f"\n🏆 BEST for {model_type}: AUC_mean={auc_mean:.4f} | repr={FEATURE_REPR}")

        bm = best_per_model[model_type]
        if bm["cfg"] is not None:
            model_dir = os.path.join(save_root, f"best_{model_type}")
            os.makedirs(model_dir, exist_ok=True)
            torch.save({"state_dict": bm["state"], "config": bm["cfg"]}, os.path.join(model_dir, "best_model_state.pt"))
            with open(os.path.join(model_dir, "best_model_config.json"), "w") as f:
                json.dump(bm["cfg"], f, indent=2)
            joblib.dump(scaler_x, os.path.join(model_dir, "scaler_x.joblib"))
            print(f"✅ Saved best artifacts for {model_type} -> {model_dir}")

    df_grid = pd.DataFrame(grid_summary_rows)
    grid_csv_out = os.path.join(out_root_models, "Classification_grid_search_results(OSIandVentALL)_8.csv")
    df_grid.to_csv(grid_csv_out, index=False)
    print("✅ Grid search summary saved:", grid_csv_out)

    cv_json_out = os.path.join(out_root_models, "Classification_cv_predictions(OSIandVentALL)_8.json")
    with open(cv_json_out, "w") as f:
        json.dump(all_cv_predictions, f)
    print(f"✅ CV predictions saved: {len(all_cv_predictions)} records -> {cv_json_out}")

# ======================================================================================
# ============================= LOAD BEST-PER-MODEL ARTIFACTS ==========================
# ======================================================================================

def load_best_for_model(model_type: str):
    model_dir = os.path.join(save_root, f"best_{model_type}")
    ckpt = os.path.join(model_dir, "best_model_state.pt")
    cfgp = os.path.join(model_dir, "best_model_config.json")
    sxp  = os.path.join(model_dir, "scaler_x.joblib")
    if not (os.path.exists(ckpt) and os.path.exists(cfgp) and os.path.exists(sxp)):
        raise FileNotFoundError(f"Missing best artifacts for {model_type} in {model_dir}")
    with open(cfgp, "r") as f:
        cfg = json.load(f)
    sx = joblib.load(sxp)
    state = torch.load(ckpt, map_location="cpu")
    return cfg, sx, state["state_dict"]

# ======================================================================================
# =============================== VALIDATION BUILD (V3) ================================
# ======================================================================================

def load_validation_set_noncnn(scaler_train: StandardScaler):
    df_val = pd.read_excel(xlsx_val, sheet_name=sheet_val)
    train_target_col = target_col
    val_target_col = "OSI_V3_12th_avg"

    if train_target_col not in df_val.columns:
        if val_target_col in df_val.columns:
            label_col = val_target_col
        else:
            raise ValueError(f"Missing target in {sheet_val}: neither {train_target_col} nor {val_target_col}")
    else:
        label_col = train_target_col

    all_feature_cols = sorted({c for tw in range(1, 7) for c in tw_features[tw]})
    missing = [c for c in all_feature_cols if c not in df_val.columns]
    if missing:
        raise ValueError(f"Missing {len(missing)} feature cols in {sheet_val}. First 30: {missing[:30]}")

    df_val_model = df_val.dropna(subset=[label_col] + all_feature_cols).copy()
    if len(df_val_model) == 0:
        raise ValueError("No validation rows left after dropna.")

    X_list = [df_val_model[tw_features[tw]].to_numpy(dtype=np.float32) for tw in range(1, 7)]
    X6_val = np.stack(X_list, axis=1)  # (N,6,D)
    X7_val = add_delta_window(X6_val)  # (N,7,D)

    y_val = (df_val_model[label_col].to_numpy(dtype=np.float32) >= threshold).astype(int)
    X7_val_scaled = apply_scaler(scaler_train, X7_val)
    return X7_val_scaled, y_val, label_col

# ======================================================================================
# =============================== EVALUATION: TEST + VAL ===============================
# ======================================================================================

def predict_probs(model: nn.Module, X_in: np.ndarray, y_in: np.ndarray, batch_size=32):
    loader = get_loader(X_in, y_in, batch_size=batch_size, shuffle=False)
    probs, ys = [], []
    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb).detach().cpu().flatten()
            p = torch.sigmoid(logits).numpy()
            probs.append(p)
            ys.append(yb.numpy().flatten())
    y_prob = np.concatenate(probs)
    y_true = np.concatenate(ys).astype(int)
    return y_true, y_prob

def evaluate_and_save_best_per_model():
    test_payloads = []
    val_payloads  = []
    metrics_rows_test = []
    metrics_rows_val  = []

    for m in model_types:
        cfg, sx, state_dict = load_best_for_model(m)

        model = build_model(
            cfg["model"],
            input_dim=int(cfg["input_dim"]),
            hidden_dim=int(cfg["hidden_dim"]),
            num_layers=int(cfg["num_layers"]),
            dropout=float(cfg["dropout"])
        ).to(device)
        model.load_state_dict(state_dict)

        bs = int(cfg.get("batch_size", 32))

        # TEST
        y_true_test, y_prob_test = predict_probs(model, X_test, y_test, batch_size=bs)
        test_payloads.append({"model": m, "config": cfg, "y_true": y_true_test.tolist(), "y_prob": y_prob_test.tolist()})

        pm = eval_point_metrics(y_true_test, y_prob_test)
        rowt = {"model": m, **pm, "N": int(len(y_true_test))}
        for s in SPEC_TARGETS:
            sens, thr = sens_at_fixed_spec(y_true_test, y_prob_test, s)
            rowt[f"Sens@Spec{int(s*100)}"] = sens
            rowt[f"Thr@Spec{int(s*100)}"] = thr
        metrics_rows_test.append(rowt)

        # VALIDATION
        X_val_scaled, y_val, label_col_used = load_validation_set_noncnn(sx)
        y_true_val, y_prob_val = predict_probs(model, X_val_scaled, y_val, batch_size=bs)
        val_payloads.append({
            "model": m, "config": cfg, "label_col_used": label_col_used,
            "y_true": y_true_val.tolist(), "y_prob": y_prob_val.tolist()
        })

        pmv = eval_point_metrics(y_true_val, y_prob_val)
        rowv = {"model": m, **pmv, "N": int(len(y_true_val)), "label_col_used": label_col_used}
        for s in SPEC_TARGETS:
            sens, thr = sens_at_fixed_spec(y_true_val, y_prob_val, s)
            rowv[f"Sens@Spec{int(s*100)}"] = sens
            rowv[f"Thr@Spec{int(s*100)}"] = thr
        metrics_rows_val.append(rowv)

    test_preds_json = os.path.join(out_root_models, "Classification_best_model_test_predictions(OSIandVentALL)_8.json")
    val_preds_json  = os.path.join(out_revision_dir, "Classification_best_model_validation_predictions(OSIandVentALL)_8.json")

    with open(test_preds_json, "w") as f:
        json.dump(test_payloads, f, indent=2)
    with open(val_preds_json, "w") as f:
        json.dump(val_payloads, f, indent=2)

    pd.DataFrame(metrics_rows_test).to_csv(
        os.path.join(out_root_models, "Classification_best_models_TEST_metrics_point(OSIandVentALL)_8.csv"),
        index=False
    )
    pd.DataFrame(metrics_rows_val).to_csv(
        os.path.join(out_revision_dir, "Classification_best_models_VALIDATION_metrics_point(OSIandVentALL)_8.csv"),
        index=False
    )

    print("✅ Saved ALL5 TEST preds:", test_preds_json)
    print("✅ Saved ALL5 VAL preds:",  val_preds_json)
    return test_preds_json, val_preds_json

# ======================================================================================
# ============================ BOOTSTRAP CI + CALIBRATION ==============================
# ======================================================================================

def run_bootstrap_and_calibration(preds_json_path: str, split_name: str, out_dir: str):
    with open(preds_json_path, "r") as f:
        payloads = json.load(f)

    ci_rows = []
    calib_rows = []

    for item in payloads:
        model_name = item["model"]
        y_true = np.array(item["y_true"], dtype=int)
        y_prob = np.array(item["y_prob"], dtype=float)

        ci_df = bootstrap_ci_metrics(
            y_true, y_prob, B=BOOTSTRAP_B, seed=0, spec_targets=tuple(SPEC_TARGETS)
        )
        ci_df.insert(0, "split", split_name)
        ci_df.insert(1, "model", model_name)
        ci_rows.append(ci_df)

        cal_df = calibration_curve_bins(y_true, y_prob, n_bins=CALIB_BINS)
        cal_df.insert(0, "split", split_name)
        cal_df.insert(1, "model", model_name)
        calib_rows.append(cal_df)

    ci_all = pd.concat(ci_rows, ignore_index=True)
    cal_all = pd.concat(calib_rows, ignore_index=True)

    ci_csv  = os.path.join(out_dir, f"BootstrapCI_{split_name}_OSIandVentALL_8.csv")
    cal_csv = os.path.join(out_dir, f"CalibrationCurve_{split_name}_OSIandVentALL_8.csv")

    ci_all.to_csv(ci_csv, index=False)
    cal_all.to_csv(cal_csv, index=False)

    print(f"✅ Bootstrap CI saved: {ci_csv}")
    print(f"✅ Calibration curve bins saved: {cal_csv}")

# ======================================================================================
# ================================ LEARNING CURVES =====================================
# ======================================================================================

def train_single_model_once(
    model_type: str,
    cfg: dict,
    X_train_in: np.ndarray,
    y_train_in: np.ndarray,
    groups_train_in: np.ndarray,
    X_test_in: np.ndarray,
    y_test_in: np.ndarray,
    X_val_in: np.ndarray,
    y_val_in: np.ndarray,
    seed: int = 0
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = build_model(
        model_type,
        input_dim=X_train_in.shape[2],
        hidden_dim=int(cfg["hidden_dim"]),
        num_layers=int(cfg["num_layers"]),
        dropout=float(cfg["dropout"])
    ).to(device)

    optimizer_name = cfg["optimizer"]
    lr = float(cfg["lr"])
    wd = float(cfg["weight_decay"])
    batch_size = int(cfg["batch_size"])
    epochs = int(cfg["epochs"])

    opt_cls = torch.optim.Adam if optimizer_name == "Adam" else torch.optim.SGD
    optimizer = opt_cls(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5)
    criterion = nn.BCEWithLogitsLoss()

    loader = get_loader(X_train_in, y_train_in, batch_size=batch_size, shuffle=True)

    best_loss = float("inf")
    wait = 0
    best_state = None

    for _ in range(epochs):
        model.train()
        losses = []
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            if torch.isnan(loss) or torch.isinf(loss):
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            losses.append(loss.item())

        avg_loss = float(np.mean(losses)) if len(losses) else np.inf
        scheduler.step(avg_loss)

        if avg_loss < best_loss:
            best_loss = avg_loss
            wait = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= 10:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    yt_test, yp_test = predict_probs(model, X_test_in, y_test_in, batch_size=batch_size)
    pm_test = eval_point_metrics(yt_test, yp_test)
    for s in SPEC_TARGETS:
        sens, thr = sens_at_fixed_spec(yt_test, yp_test, s)
        pm_test[f"Sens@Spec{int(s*100)}"] = sens
        pm_test[f"Thr@Spec{int(s*100)}"] = thr

    yt_val, yp_val = predict_probs(model, X_val_in, y_val_in, batch_size=batch_size)
    pm_val = eval_point_metrics(yt_val, yp_val)
    for s in SPEC_TARGETS:
        sens, thr = sens_at_fixed_spec(yt_val, yp_val, s)
        pm_val[f"Sens@Spec{int(s*100)}"] = sens
        pm_val[f"Thr@Spec{int(s*100)}"] = thr

    return pm_test, pm_val

def run_learning_curves():
    X_val_scaled, y_val, label_col_used = load_validation_set_noncnn(scaler_x)

    rows = []
    unique_groups = np.unique(groups_train)

    for model_type in LEARNING_MODELS:
        cfg, _, _ = load_best_for_model(model_type)

        for frac in LEARNING_FRACTIONS:
            n_groups = max(2, int(len(unique_groups) * frac))
            for seed in LEARNING_SEEDS:
                rng = np.random.RandomState(seed)
                chosen_groups = rng.choice(unique_groups, size=n_groups, replace=False)
                mask = np.isin(groups_train, chosen_groups)

                X_sub = X_train[mask]
                y_sub = y_train[mask]
                g_sub = groups_train[mask]

                X_sub_aug, y_sub_aug, g_sub_aug = augment_data(X_sub, y_sub, g_sub, seed=seed)

                pm_test, pm_val = train_single_model_once(
                    model_type, cfg,
                    X_sub_aug, y_sub_aug, g_sub_aug,
                    X_test, y_test,
                    X_val_scaled, y_val,
                    seed=seed
                )

                rows.append({
                    "model": model_type,
                    "feature_repr": FEATURE_REPR,
                    "fraction": frac,
                    "seed": seed,
                    "label_col_val": label_col_used,
                    **{f"test_{k}": v for k, v in pm_test.items()},
                    **{f"val_{k}": v for k, v in pm_val.items()},
                    "n_train_patients": int(n_groups),
                    "n_train_samples": int(len(X_sub))
                })

                print(f"✅ LearningCurve {model_type} frac={frac} seed={seed} "
                      f"test_AUC={pm_test['AUC']:.3f} val_AUC={pm_val['AUC']:.3f}")

    df_lc = pd.DataFrame(rows)
    lc_csv = os.path.join(out_revision_dir, "LearningCurve_OSIandVentALL_8.csv")
    df_lc.to_csv(lc_csv, index=False)
    print("✅ Learning curve results saved:", lc_csv)

# ======================================================================================
# ===================================== RUN ===========================================
# ======================================================================================

test_preds_json = None
val_preds_json  = None

if RUN_EVAL_BEST_MODELS:
    test_preds_json, val_preds_json = evaluate_and_save_best_per_model()

if RUN_BOOTSTRAP_CI_EVAL:
    if test_preds_json is None or val_preds_json is None:
        test_preds_json = os.path.join(out_root_models, "Classification_best_model_test_predictions(OSIandVentALL)_8.json")
        val_preds_json  = os.path.join(out_revision_dir, "Classification_best_model_validation_predictions(OSIandVentALL)_8.json")
    run_bootstrap_and_calibration(test_preds_json, "TEST", out_root_models)
    run_bootstrap_and_calibration(val_preds_json, "VALIDATION", out_revision_dir)

if RUN_LEARNING_CURVES:
    run_learning_curves()

print("✅ All done.")

### OSI+Vent_All Features Only (Classification) -- training + test + validation (_7)

In [ ]:
# === Try more (with best-model weight saving) + TEST + VALIDATION (NO retraining) ===
# UPDATED for repeats=5 and folds=3:
#   ✅ No hard-coded "50" anymore (uses expected = repeats * folds)
#   ✅ Saves repeats/folds/expected/n_metrics in summary + best config
#   ✅ More robust: skips folds with single-class validation labels (AUC undefined)
#   ✅ Keeps your V3 validation label mapping (OSI_V3_12th_avg fallback)

import os
import math
import json
import joblib  # for saving/loading scaler
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from mamba_ssm import Mamba

# ========== Device ==========
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ========== Load Data ==========
df = pd.read_excel(
    "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1V2_df.xlsx",
    sheet_name="Sheet5"
)

target_col = "OSI_V2_12th_avg"
threshold = 7.5

base_template = [
    "OSI_mean_TW{}", "OSI_std_TW{}", "PIP_mean_TW{}", "PIP_std_TW{}",
    "PEEP_mean_TW{}", "PEEP_std_TW{}", "TV_mean_TW{}(mL/Kg)", "TV_std_TW{}(mL/Kg)",
    "Avg_NegFlowDur_TW{}", "Std_NegFlowDur_TW{}", "Avg_PeakInterval_TW{}", "Std_PeakInterval_TW{}"
]

tw_features = {
    i: [f.format(i) for f in base_template] + [f"w{i}_SubBandEnergy_row{j}" for j in range(1, 17)]
    for i in range(1, 7)
}

# ========== Build sequences ==========
X, y, groups = [], [], []
for idx, row in df.iterrows():
    sequence = []
    for tw in range(1, 7):
        cols = tw_features[tw]
        if row[cols].isnull().any():
            break
        sequence.append(row[cols].values)

    if len(sequence) == 6 and not pd.isna(row[target_col]):
        X.append(sequence)
        y.append(1 if row[target_col] >= threshold else 0)
        groups.append(row["ResearchID"] if "ResearchID" in df.columns else idx)

X = np.array(X, dtype=np.float32)  # (N, 6, D)
y = np.array(y, dtype=np.float32)  # (N,)
groups = np.array(groups)

print(f"✅ Total usable samples: {len(X)} | Positives: {int(y.sum())} | Negatives: {int((1-y).sum())}")

# ========== Feature Engineering ==========
delta_features = X[:, -1, :] - X[:, 0, :]
delta_features = delta_features[:, np.newaxis, :]
X = np.concatenate([X, delta_features], axis=1)  # (N, 7, D)

# ========== Scaling ==========
scaler_x = StandardScaler()
X_scaled = scaler_x.fit_transform(X.reshape(-1, X.shape[2])).reshape(X.shape)

# ========== Train/Test Split ==========
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X_scaled, y, groups, test_size=0.2, random_state=42
)

print(f"Training samples before augmentation: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# ========== Data Augmentation ==========
def augment_data(X, y, groups):
    noise = np.random.normal(0, 0.005, X.shape).astype(np.float32)
    return (
        np.concatenate([X, X + noise]),
        np.concatenate([y, y]),
        np.concatenate([groups, groups])
    )

X_train, y_train, groups_train = augment_data(X_train, y_train, groups_train)
print(f"Training samples after augmentation: {len(X_train)}")

# ========== Model Definitions ==========
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return torch.sum(weights * x, dim=1)

class RNNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, rnn_type="RNN", num_layers=1, dropout=0.0):
        super().__init__()
        if num_layers == 1:
            dropout = 0.0
        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[rnn_type]
        self.rnn = rnn_cls(input_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.attn(out)
        return self.fc(out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, nhead, dropout):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=nhead,
            dropout=dropout,
            batch_first=True,
            layer_norm_eps=1e-5
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)

    def forward(self, x):
        return self.encoder(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0, nhead=None):
        super().__init__()
        if nhead is None:
            if hidden_dim % 8 == 0:
                nhead = hidden_dim // 8
            elif hidden_dim % 4 == 0:
                nhead = hidden_dim // 4
            else:
                nhead = 1

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )
        self.pe = PositionalEncoding(hidden_dim)

        if num_layers == 1:
            self.stack = nn.Sequential(*[TransformerBlock(hidden_dim, nhead, 0.0)])
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(TransformerBlock(hidden_dim, nhead, dropout), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])

        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pe(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

class MambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba = Mamba(d_model=dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return self.norm(self.mamba(x))

class MambaClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        if num_layers == 1:
            self.stack = nn.Sequential(*[MambaBlock(hidden_dim)])
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(MambaBlock(hidden_dim), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.norm(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

def build_model(model_type, input_dim, hidden_dim, num_layers, dropout):
    if model_type == "Mamba":
        return MambaClassifier(input_dim, hidden_dim, num_layers, dropout)
    if model_type == "Transformer":
        return TransformerModel(input_dim, hidden_dim, num_layers, dropout)
    return RNNClassifier(input_dim, hidden_dim, model_type, num_layers, dropout)

# ========== Utility ==========
def get_loader(X, y, batch_size, shuffle=True):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=shuffle)

def auprc(y_true, y_prob):
    p, r, _ = precision_recall_curve(y_true, y_prob)
    return auc(r, p)

def stratified_sample(y_train, ratio=0.5, pos_ratio=1/3):
    y_flat = y_train.flatten()
    idx_0 = np.where(y_flat == 0)[0]
    idx_1 = np.where(y_flat == 1)[0]
    total = int(len(y_flat) * ratio)
    n1 = int(total * pos_ratio)
    n0 = total - n1
    if len(idx_0) < n0 or len(idx_1) < n1:
        return None, None
    idx = np.concatenate([
        np.random.choice(idx_0, n0, replace=False),
        np.random.choice(idx_1, n1, replace=False)
    ])
    np.random.shuffle(idx)
    return idx, total

# ========== Where to save best model ==========
save_root = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/SavedModels_Stats_7"
os.makedirs(save_root, exist_ok=True)

global_best_auc = -np.inf
global_best_info = None
global_best_state_dict = None

# ========== Train Loop Controls ==========
epochs = 200
n_repeats = 5
n_splits = 3
expected_metrics = n_repeats * n_splits  # 15

# ========== Train Loop ==========
results = {"cv_predictions": [], "summary": []}
model_types = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]

for model_type in model_types:
    for hidden_dim in [16, 32, 64, 128]:
        for dropout in [0.0, 0.2]:
            for num_layers in [1, 2, 3]:
                for batch_size in [16, 32]:
                    for lr in [0.001, 0.01]:
                        for wd in [0, 1e-4]:
                            for optimizer_name in ["Adam", "SGD"]:

                                all_aucs, all_auprcs, all_f1s = [], [], []
                                used_epochs_all = []

                                combo_best_val_loss = np.inf
                                combo_best_state_dict = None

                                for repeat in range(n_repeats):
                                    sampled_idx, _ = stratified_sample(y_train)
                                    if sampled_idx is None:
                                        print(f"[{model_type}] Skipping repeat {repeat+1}: not enough pos/neg after stratified_sample().")
                                        continue

                                    X_sub = X_train[sampled_idx]
                                    y_sub = y_train[sampled_idx]
                                    groups_sub = groups_train[sampled_idx]

                                    gkf = GroupKFold(n_splits=n_splits)

                                    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_sub, y_sub, groups_sub)):
                                        print(f"[Repeat {repeat+1}/{n_repeats} | Fold {fold+1}/{n_splits}] "
                                              f"Train size: {len(tr_idx)} | Val size: {len(val_idx)} | Test size: {len(y_test)}")

                                        model = build_model(model_type, X_train.shape[2], hidden_dim, num_layers, dropout).to(device)

                                        optimizer_cls = torch.optim.Adam if optimizer_name == "Adam" else torch.optim.SGD
                                        optimizer = optimizer_cls(model.parameters(), lr=lr, weight_decay=wd)
                                        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5)
                                        criterion = nn.BCEWithLogitsLoss()

                                        loader = get_loader(X_sub[tr_idx], y_sub[tr_idx], batch_size, shuffle=True)

                                        best_loss = float("inf")
                                        wait = 0
                                        used_epochs = 0
                                        best_state_dict_fold = None

                                        for epoch in range(epochs):
                                            model.train()
                                            epoch_losses = []
                                            for xb, yb in loader:
                                                xb, yb = xb.to(device), yb.to(device)
                                                optimizer.zero_grad()
                                                pred = model(xb)
                                                loss = criterion(pred, yb)
                                                if torch.isnan(loss) or torch.isinf(loss):
                                                    continue
                                                loss.backward()
                                                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                                                optimizer.step()
                                                epoch_losses.append(loss.item())

                                            avg_loss = float(np.mean(epoch_losses)) if len(epoch_losses) else np.inf
                                            scheduler.step(avg_loss)
                                            used_epochs += 1

                                            if avg_loss < best_loss:
                                                best_loss = avg_loss
                                                wait = 0
                                                best_state_dict_fold = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                                            else:
                                                wait += 1
                                                if wait >= 10:
                                                    break

                                        used_epochs_all.append(used_epochs)

                                        # restore best fold weights
                                        if best_state_dict_fold is not None:
                                            model.load_state_dict(best_state_dict_fold)

                                        # Evaluate on validation
                                        model.eval()
                                        with torch.no_grad():
                                            val_input = torch.tensor(X_sub[val_idx], dtype=torch.float32).to(device)
                                            logits_tensor = model(val_input).cpu().flatten()
                                            probs = torch.sigmoid(logits_tensor).numpy()
                                            y_true = y_sub[val_idx].flatten()

                                        valid_mask = np.isfinite(probs)
                                        if valid_mask.sum() == 0:
                                            continue

                                        probs = probs[valid_mask]
                                        y_true = y_true[valid_mask]
                                        preds = (probs >= 0.5).astype(int)

                                        # AUC undefined if only one class present
                                        if len(np.unique(y_true)) < 2:
                                            continue

                                        all_aucs.append(float(roc_auc_score(y_true, probs)))
                                        all_auprcs.append(float(auprc(y_true, probs)))
                                        all_f1s.append(float(f1_score(y_true, preds)))

                                        results["cv_predictions"].append({
                                            "model": model_type,
                                            "hidden_dim": hidden_dim,
                                            "dropout": dropout,
                                            "num_layers": num_layers,
                                            "batch_size": batch_size,
                                            "lr": lr,
                                            "weight_decay": wd,
                                            "optimizer": optimizer_name,
                                            "repeat": repeat,
                                            "fold": fold,
                                            "y_true": y_true.tolist(),
                                            "y_prob": probs.tolist()
                                        })

                                        # best fold for combo by best_loss
                                        if best_loss < combo_best_val_loss and best_state_dict_fold is not None:
                                            combo_best_val_loss = best_loss
                                            combo_best_state_dict = best_state_dict_fold

                                # ===== Summary (dynamic completeness) =====
                                is_complete = (len(all_aucs) == expected_metrics)

                                auc_mean = float(np.mean(all_aucs)) if is_complete else np.nan
                                auc_std  = float(np.std(all_aucs))  if is_complete else np.nan
                                auprc_mean = float(np.mean(all_auprcs)) if is_complete else np.nan
                                auprc_std  = float(np.std(all_auprcs))  if is_complete else np.nan
                                f1_mean = float(np.mean(all_f1s)) if is_complete else np.nan
                                f1_std  = float(np.std(all_f1s))  if is_complete else np.nan

                                used_epochs_mean = float(np.mean(used_epochs_all)) if used_epochs_all else np.nan
                                used_epochs_std  = float(np.std(used_epochs_all))  if used_epochs_all else np.nan

                                results["summary"].append({
                                    "model": model_type,
                                    "hidden_dim": hidden_dim,
                                    "dropout": dropout,
                                    "num_layers": num_layers,
                                    "batch_size": batch_size,
                                    "lr": lr,
                                    "weight_decay": wd,
                                    "optimizer": optimizer_name,
                                    "epochs": epochs,
                                    "repeats": n_repeats,
                                    "folds": n_splits,
                                    "expected_metrics": int(expected_metrics),
                                    "n_metrics": int(len(all_aucs)),
                                    "AUC_mean": auc_mean,
                                    "AUC_std": auc_std,
                                    "AUPRC_mean": auprc_mean,
                                    "AUPRC_std": auprc_std,
                                    "F1_mean": f1_mean,
                                    "F1_std": f1_std,
                                    "used_epochs_mean": used_epochs_mean,
                                    "used_epochs_std": used_epochs_std,
                                })

                                # Track global best by AUC_mean (only if complete)
                                if np.isfinite(auc_mean) and auc_mean > global_best_auc and combo_best_state_dict is not None:
                                    global_best_auc = float(auc_mean)
                                    global_best_info = {
                                        "model": model_type,
                                        "hidden_dim": int(hidden_dim),
                                        "dropout": float(dropout),
                                        "num_layers": int(num_layers),
                                        "batch_size": int(batch_size),
                                        "lr": float(lr),
                                        "weight_decay": float(wd),
                                        "optimizer": optimizer_name,
                                        "epochs": int(epochs),
                                        "repeats": int(n_repeats),
                                        "folds": int(n_splits),
                                        "expected_metrics": int(expected_metrics),
                                        "n_metrics": int(len(all_aucs)),
                                        "AUC_mean": float(auc_mean),
                                        "AUPRC_mean": float(auprc_mean) if np.isfinite(auprc_mean) else None,
                                        "F1_mean": float(f1_mean) if np.isfinite(f1_mean) else None,
                                        "input_dim": int(X_train.shape[2]),
                                        "timesteps": int(X_train.shape[1]),
                                        "target_col": target_col,
                                        "threshold": float(threshold),
                                    }
                                    global_best_state_dict = combo_best_state_dict
                                    print("\n🏆 NEW GLOBAL BEST MODEL FOUND:", global_best_info)

                                    torch.save(
                                        {"state_dict": global_best_state_dict, "config": global_best_info},
                                        os.path.join(save_root, "best_model_state.pt")
                                    )
                                    with open(os.path.join(save_root, "best_model_config.json"), "w") as f:
                                        json.dump(global_best_info, f, indent=2)

                                    joblib.dump(scaler_x, os.path.join(save_root, "scaler_x.joblib"))

# ========== Save Training Results ==========
df_results = pd.DataFrame(results["summary"])
grid_csv_out = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_grid_search_results(Stats)_7.csv"
df_results.to_csv(grid_csv_out, index=False)
print("✅ Grid search completed and saved:", grid_csv_out)

cv_json_out = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_cv_predictions(Stats)_7.json"
with open(cv_json_out, "w") as f:
    json.dump(results["cv_predictions"], f)
print(f"✅ CV predictions saved: {len(results['cv_predictions'])} records -> {cv_json_out}")
print("✅ Best model artifacts saved to:", save_root)

# ======================================================================================
# TEST + VALIDATION (Best saved model, NO retraining)
# ======================================================================================

def load_best_artifacts(save_root):
    ckpt_path = os.path.join(save_root, "best_model_state.pt")
    cfg_path = os.path.join(save_root, "best_model_config.json")
    sx_path = os.path.join(save_root, "scaler_x.joblib")

    for p in [ckpt_path, cfg_path, sx_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing best artifact: {p}")

    with open(cfg_path, "r") as f:
        cfg = json.load(f)

    sx = joblib.load(sx_path)

    state = torch.load(ckpt_path, map_location="cpu")
    state_dict = state["state_dict"] if isinstance(state, dict) and "state_dict" in state else state
    return cfg, sx, state_dict

best_cfg, best_scaler_x, best_state_dict = load_best_artifacts(save_root)

best_model = build_model(
    best_cfg["model"],
    input_dim=X_test.shape[2],
    hidden_dim=int(best_cfg["hidden_dim"]),
    num_layers=int(best_cfg["num_layers"]),
    dropout=float(best_cfg["dropout"])
).to(device)

best_model.load_state_dict(best_state_dict)
best_model.eval()

best_bs = int(best_cfg.get("batch_size", 32))

# --------------------------
# TEST (held-out X_test)
# --------------------------
test_loader = get_loader(X_test, y_test, batch_size=best_bs, shuffle=False)

all_probs, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = best_model(xb).detach().cpu().flatten()
        probs = torch.sigmoid(logits).numpy()
        all_probs.append(probs)
        all_true.append(yb.numpy().flatten())

y_prob_test = np.concatenate(all_probs)
y_true_test = np.concatenate(all_true).astype(int)

test_auc = float(roc_auc_score(y_true_test, y_prob_test)) if len(np.unique(y_true_test)) > 1 else float("nan")
test_auprc = float(auprc(y_true_test, y_prob_test)) if len(np.unique(y_true_test)) > 1 else float("nan")
test_pred = (y_prob_test >= 0.5).astype(int)
test_f1 = float(f1_score(y_true_test, test_pred)) if len(np.unique(y_true_test)) > 1 else float("nan")

print("\n✅ TEST (held-out split)")
print(f"AUC: {test_auc:.4f} | AUPRC: {test_auprc:.4f} | F1: {test_f1:.4f} | N={len(y_true_test)}")

test_results_df = pd.DataFrame([{
    "model": best_cfg["model"],
    "AUC_test": test_auc,
    "AUPRC_test": test_auprc,
    "F1_test": test_f1,
    "N_test": int(len(y_true_test)),
    **{k: best_cfg.get(k) for k in ["hidden_dim", "dropout", "num_layers", "batch_size", "lr", "weight_decay", "optimizer", "epochs", "repeats", "folds"]}
}])

test_results_csv = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_best_model_test_results(Stats)_7.csv"
test_preds_json = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_best_model_test_predictions(Stats)_7.json"
test_pred_csv = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_best_model_test_predictions_per_sample(Stats)_7.csv"

test_results_df.to_csv(test_results_csv, index=False)

with open(test_preds_json, "w") as f:
    json.dump([{
        "model": best_cfg["model"],
        "y_true": y_true_test.tolist(),
        "y_prob": y_prob_test.tolist()
    }], f)

pd.DataFrame({
    "y_true": y_true_test.astype(int),
    "y_prob": y_prob_test,
    "y_pred": test_pred.astype(int),
}).to_csv(test_pred_csv, index=False)

print("✅ Test performance saved:", test_results_csv)
print("✅ Test predictions saved:", test_preds_json)
print("✅ Test per-sample CSV saved:", test_pred_csv)

# --------------------------
# VALIDATION on V3 Sheet15 (65 samples)
# FIX: uses OSI_V3_12th_avg if OSI_V2_12th_avg is not present
# --------------------------
xlsx_val = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"
sheet_val = "Sheet15"
df_val = pd.read_excel(xlsx_val, sheet_name=sheet_val)

train_target_col = target_col              # "OSI_V2_12th_avg"
val_target_col   = "OSI_V3_12th_avg"       # actual in V3 sheet

if train_target_col not in df_val.columns:
    if val_target_col in df_val.columns:
        print(f"NOTE: '{train_target_col}' not found in {sheet_val}; using '{val_target_col}' instead.")
        label_col = val_target_col
    else:
        raise ValueError(f"Missing target in {sheet_val}: neither '{train_target_col}' nor '{val_target_col}' exists.")
else:
    label_col = train_target_col

all_feature_cols = [c for tw in range(1, 7) for c in tw_features[tw]]
missing_feats = [c for c in all_feature_cols if c not in df_val.columns]
if missing_feats:
    raise ValueError(f"Missing {len(missing_feats)} feature columns in {sheet_val}. First 30: {missing_feats[:30]}")

df_val_model = df_val.dropna(subset=[label_col] + all_feature_cols).copy()
print(f"\n✅ VALIDATION rows after dropna ({sheet_val}): {len(df_val_model)}")
if len(df_val_model) == 0:
    raise ValueError("No rows left after dropna in validation; cannot run validation.")

X_list = [df_val_model[tw_features[tw]].to_numpy(dtype=np.float32) for tw in range(1, 7)]
X6_val = np.stack(X_list, axis=1)  # (N, 6, D)
delta_val = (X6_val[:, -1, :] - X6_val[:, 0, :])[:, np.newaxis, :]
X7_val = np.concatenate([X6_val, delta_val], axis=1)  # (N, 7, D)

y_true_val = (df_val_model[label_col].to_numpy(dtype=np.float32) >= threshold).astype(int)

N, T, D = X7_val.shape
X7_val_scaled = best_scaler_x.transform(X7_val.reshape(-1, D)).reshape(N, T, D)

val_loader = DataLoader(
    TensorDataset(
        torch.tensor(X7_val_scaled, dtype=torch.float32),
        torch.tensor(y_true_val, dtype=torch.float32).view(-1, 1),
    ),
    batch_size=best_bs,
    shuffle=False,
)

val_probs, val_true = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        logits = best_model(xb).detach().cpu().flatten()
        probs = torch.sigmoid(logits).numpy()
        val_probs.append(probs)
        val_true.append(yb.numpy().flatten())

y_prob_val = np.concatenate(val_probs)
y_true_val = np.concatenate(val_true).astype(int)

val_auc = float(roc_auc_score(y_true_val, y_prob_val)) if len(np.unique(y_true_val)) > 1 else float("nan")
val_auprc = float(auprc(y_true_val, y_prob_val)) if len(np.unique(y_true_val)) > 1 else float("nan")
val_pred = (y_prob_val >= 0.5).astype(int)
val_f1 = float(f1_score(y_true_val, val_pred)) if len(np.unique(y_true_val)) > 1 else float("nan")

print(f"\n✅ VALIDATION (V3 {sheet_val})")
print(f"AUC: {val_auc:.4f} | AUPRC: {val_auprc:.4f} | F1: {val_f1:.4f} | N={len(y_true_val)} | label_col={label_col}")

val_out_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/20251105_TBME_Submission/20260301_Major_Revision"
os.makedirs(val_out_dir, exist_ok=True)

val_results_csv = os.path.join(val_out_dir, "Classification_best_model_validation_results(Stats)_7.csv")
val_preds_json  = os.path.join(val_out_dir, "Classification_best_model_validation_predictions(Stats)_7.json")
val_pred_csv    = os.path.join(val_out_dir, "Classification_best_model_validation_predictions_per_sample(Stats)_7.csv")

pd.DataFrame([{
    "model": best_cfg["model"],
    "sheet": sheet_val,
    "label_col_used": label_col,
    "AUC_val": val_auc,
    "AUPRC_val": val_auprc,
    "F1_val": val_f1,
    "N_val": int(len(y_true_val)),
    **{k: best_cfg.get(k) for k in ["hidden_dim", "dropout", "num_layers", "batch_size", "lr", "weight_decay", "optimizer", "epochs", "repeats", "folds"]}
}]).to_csv(val_results_csv, index=False)

with open(val_preds_json, "w") as f:
    json.dump([{
        "model": best_cfg["model"],
        "sheet": sheet_val,
        "label_col_used": label_col,
        "y_true": y_true_val.tolist(),
        "y_prob": y_prob_val.tolist()
    }], f)

pd.DataFrame({
    "y_true": y_true_val.astype(int),
    "y_prob": y_prob_val,
    "y_pred": val_pred.astype(int),
}).to_csv(val_pred_csv, index=False)

print("✅ Validation performance saved:", val_results_csv)
print("✅ Validation predictions saved:", val_preds_json)
print("✅ Validation per-sample CSV saved:", val_pred_csv)


### CNN Features Only (Classification) -- training + test + validation (_8)

In [ ]:
tw_features = {
    tw: [f"f{i}_TW{tw}" for i in range(1, 257)]
    for tw in range(1, 7)
}

### OSI+CNN Features (Classification) -- training + test + validation (_8)

In [ ]:
base_template = ["OSI_mean_TW{}", "OSI_std_TW{}"]
tw_features = {
    tw: [f.format(tw) for f in base_template] + [f"f{i}_TW{tw}" for i in range(1, 257)]
    for tw in range(1, 7)
}

In [ ]:
# ======================================================================================
# TBME Major Revision Utilities (CNN + optional OSI feature set)
# - Save BEST config+weights per model type (RNN/LSTM/GRU/Transformer/Mamba)
# - Save TRAINING CV predictions + summary (like your previous *_7 code)
# - Evaluate BEST per model on TEST + TEMPORAL VALIDATION and save y_true/y_prob
# - Learning curve (20/40/60/80/100% of training patients)
# - PCA sensitivity:
#     * If OSI is used: PCA64 is applied ONLY to CNN block, then concat OSI, then train models
#     * If OSI not used: PCA64 is applied to full feature vector (same as before)
# - Patient-level bootstrap CIs + Sens@fixed specificity + Calibration metrics/curve
# ======================================================================================

import os
import math
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, auc, f1_score,
    roc_curve, brier_score_loss
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from mamba_ssm import Mamba

# ======================================================================================
# ================================ USER CONTROLS ======================================
# ======================================================================================

RUN_GRID_SEARCH       = True     # trains many configs and saves best-per-model-type
RUN_EVAL_BEST_MODELS  = True     # runs TEST+VALIDATION for best-per-model-type (no retraining)
RUN_LEARNING_CURVES   = True     # trains best-config per model_type on fractions of patients
RUN_PCA_SENSITIVITY   = True     # trains best-config per model_type with PCA-64 (and optional PCA-32)
RUN_BOOTSTRAP_CI_EVAL = True     # compute patient-level CI + sens@spec + calibration on TEST/VAL preds

# For learning curves / PCA sensitivity, you probably don't want to redo all 5 models.
LEARNING_MODELS = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]    # or ["Mamba"] to keep it light
PCA_MODELS      = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]    # or ["Mamba"]
PCA_COMPONENTS_LIST = [64]                  # can set [64, 32] if you want both

LEARNING_FRACTIONS = [0.2, 0.4, 0.6, 0.8, 1.0]
LEARNING_SEEDS     = [0, 1, 2]              # repeats per fraction (3 is enough)

# Bootstrap settings
BOOTSTRAP_B = 2000
SPEC_TARGETS = [0.90, 0.95]   # sensitivity at these specificities
CALIB_BINS = 10

# ---------------------------
# Feature set toggles
# ---------------------------
USE_OSI_FEATURES = True
"""
Set USE_OSI_FEATURES=True if you are building OSI+CNN features per TW.
IMPORTANT: If True, you MUST define `osi_features` below (per TW list of column names),
and PCA sensitivity will apply PCA only on the CNN block then concat OSI.
"""

CNN_DIM_PER_TW = 256  # CNN feature block size per TW (your f1..f256)

PCA_ON_CNN_ONLY_IF_OSI = True  # the behavior you asked for

# ======================================================================================
# ================================ PATHS / I/O ========================================
# ======================================================================================

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Training data
xlsx_train = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1V2_df.xlsx"
sheet_train = "Sheet5"

# Temporal validation data
xlsx_val = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"
sheet_val = "Sheet15"

# Output roots
out_root_models = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels"
save_root = os.path.join(out_root_models, "SavedModels_OSIandCNN_8")  # NEW folder
os.makedirs(save_root, exist_ok=True)

out_revision_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/20251105_TBME_Submission/20260301_Major_Revision"
os.makedirs(out_revision_dir, exist_ok=True)

# ======================================================================================
# ================================= DATA LOADING ======================================
# ======================================================================================

target_col = "OSI_V2_12th_avg"
threshold = 7.5

# CNN features (per TW): f1_TW1..f256_TW6
tw_cnn_features = {tw: [f"f{i}_TW{tw}" for i in range(1, 257)] for tw in range(1, 7)}

osi_features = {
    tw: [f"OSI_mean_TW{tw}", f"OSI_std_TW{tw}"]
    for tw in range(1, 7)
}

def build_sequences_from_df(
    df: pd.DataFrame,
    label_col: str,
    cnn_features: dict,
    threshold: float,
    use_osi: bool = False,
    osi_features: dict = None
):
    """
    Returns:
      X6: (N,6,D_perTW)
      y:  (N,)
      groups: (N,)
    """
    if osi_features is None:
        osi_features = {tw: [] for tw in range(1, 7)}

    X, y, groups = [], [], []
    for idx, row in df.iterrows():
        seq = []
        ok = True
        for tw in range(1, 7):
            cols = list(cnn_features[tw])
            if use_osi:
                cols = cols + list(osi_features.get(tw, []))

            if any(c not in df.columns for c in cols):
                ok = False
                break

            if row[cols].isnull().any():
                ok = False
                break

            seq.append(row[cols].values)

        if ok and (not pd.isna(row[label_col])):
            X.append(seq)
            y.append(1 if row[label_col] >= threshold else 0)
            groups.append(row["ResearchID"] if "ResearchID" in df.columns else idx)

    X = np.array(X, dtype=np.float32)  # (N, 6, D)
    y = np.array(y, dtype=np.float32)  # (N,)
    groups = np.array(groups)
    return X, y, groups

def add_delta_window(X6: np.ndarray) -> np.ndarray:
    # X6: (N, 6, D)
    delta = (X6[:, -1, :] - X6[:, 0, :])[:, np.newaxis, :]  # (N,1,D)
    return np.concatenate([X6, delta], axis=1)              # (N,7,D)

# Load train
df = pd.read_excel(xlsx_train, sheet_name=sheet_train)

X6, y, groups = build_sequences_from_df(
    df,
    target_col,
    tw_cnn_features,
    threshold,
    use_osi=USE_OSI_FEATURES,
    osi_features=osi_features
)
X = add_delta_window(X6)  # (N,7,D_total)

print(f"✅ Total usable samples: {len(X)} | Positives: {int(y.sum())} | Negatives: {int((1-y).sum())}")
print(f"✅ Input shape: {X.shape} (N,T,D_total)")

if USE_OSI_FEATURES:
    # sanity: CNN block must be first in each TW if you want PCA slicing by position
    # Our build_sequences_from_df appends OSI after CNN, so this is true.
    d_total = X.shape[2]
    # OSI per TW dims:
    osi_dim_per_tw = len(osi_features.get(1, []))
    # But delta window also contains CNN+OSI dims; PCA slicing assumes first CNN_DIM_PER_TW dims correspond to CNN,
    # and the remaining dims correspond to OSI, per window.
    print(f"✅ USE_OSI_FEATURES=True | CNN_DIM_PER_TW={CNN_DIM_PER_TW} | D_total_perTW={d_total} | OSI_dims_perTW={d_total - CNN_DIM_PER_TW}")

# ======================================================================================
# ============================= PREPROCESSING HELPERS =================================
# ======================================================================================

def fit_scaler_on_train(X_train: np.ndarray) -> StandardScaler:
    scaler = StandardScaler()
    N, T, D = X_train.shape
    scaler.fit(X_train.reshape(-1, D))
    return scaler

def apply_scaler(scaler: StandardScaler, X_in: np.ndarray) -> np.ndarray:
    N, T, D = X_in.shape
    return scaler.transform(X_in.reshape(-1, D)).reshape(N, T, D)

def augment_data(X_in, y_in, groups_in, noise_std=0.005, seed=0):
    rng = np.random.RandomState(seed)
    noise = rng.normal(0, noise_std, X_in.shape).astype(np.float32)
    return (
        np.concatenate([X_in, X_in + noise]),
        np.concatenate([y_in, y_in]),
        np.concatenate([groups_in, groups_in])
    )

# --- PCA (full vector) ---
def fit_pca_on_train_full(X_train_scaled: np.ndarray, n_components: int) -> PCA:
    N, T, D = X_train_scaled.shape
    pca = PCA(n_components=n_components, random_state=0)
    pca.fit(X_train_scaled.reshape(-1, D))
    return pca

def apply_pca_full(pca: PCA, X_scaled: np.ndarray) -> np.ndarray:
    N, T, D = X_scaled.shape
    X2 = pca.transform(X_scaled.reshape(-1, D))  # (N*T, K)
    return X2.reshape(N, T, -1).astype(np.float32)

# --- PCA (CNN-only then concat OSI) ---
def fit_pca_on_train_cnn_only(X_train_scaled: np.ndarray, cnn_dim: int, n_components: int) -> PCA:
    """
    Fit PCA ONLY on the CNN block (first cnn_dim features in the last dimension), after scaling.
    Assumes features are [CNN | OSI] per time window.
    """
    N, T, D = X_train_scaled.shape
    assert D >= cnn_dim, f"D_total({D}) < cnn_dim({cnn_dim}). Check feature construction."
    X_cnn = X_train_scaled[:, :, :cnn_dim].reshape(-1, cnn_dim)
    pca = PCA(n_components=n_components, random_state=0)
    pca.fit(X_cnn)
    return pca

def apply_pca_cnn_only_and_concat(X_scaled: np.ndarray, pca: PCA, cnn_dim: int) -> np.ndarray:
    """
    Apply PCA ONLY to CNN block, keep OSI block unchanged, then concat.
    Output shape: (N,T,n_components + OSI_dim)
    """
    N, T, D = X_scaled.shape
    assert D >= cnn_dim, f"D_total({D}) < cnn_dim({cnn_dim}). Check feature construction."
    X_cnn = X_scaled[:, :, :cnn_dim].reshape(-1, cnn_dim)  # (N*T, cnn_dim)
    X_osi = X_scaled[:, :, cnn_dim:]                       # (N,T,OSI_dim)

    X_cnn_pca = pca.transform(X_cnn).reshape(N, T, -1).astype(np.float32)
    X_out = np.concatenate([X_cnn_pca, X_osi.astype(np.float32)], axis=-1)
    return X_out

# ======================================================================================
# ================================== MODELS ===========================================
# ======================================================================================

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        w = torch.softmax(self.attn(x), dim=1)  # (B,T,1)
        return torch.sum(w * x, dim=1)          # (B,H)

class RNNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, rnn_type="RNN", num_layers=1, dropout=0.0):
        super().__init__()
        if num_layers == 1:
            dropout = 0.0
        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[rnn_type]
        self.rnn = rnn_cls(input_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.attn(out)
        return self.fc(out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, nhead, dropout):
        super().__init__()
        enc = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=nhead, dropout=dropout,
            batch_first=True, layer_norm_eps=1e-5
        )
        self.encoder = nn.TransformerEncoder(enc, num_layers=1)

    def forward(self, x):
        return self.encoder(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0, nhead=None):
        super().__init__()
        if nhead is None:
            if hidden_dim % 8 == 0:
                nhead = max(1, hidden_dim // 8)
            elif hidden_dim % 4 == 0:
                nhead = max(1, hidden_dim // 4)
            else:
                nhead = 1

        self.input_proj = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim))
        self.pe = PositionalEncoding(hidden_dim)

        if num_layers == 1:
            self.stack = nn.Sequential(TransformerBlock(hidden_dim, nhead, 0.0))
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(TransformerBlock(hidden_dim, nhead, dropout), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])

        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pe(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

class MambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba = Mamba(d_model=dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return self.norm(self.mamba(x))

class MambaClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        if num_layers == 1:
            self.stack = nn.Sequential(MambaBlock(hidden_dim))
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(MambaBlock(hidden_dim), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.norm(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

def build_model(model_type, input_dim, hidden_dim, num_layers, dropout):
    if model_type == "Mamba":
        return MambaClassifier(input_dim, hidden_dim, num_layers, dropout)
    if model_type == "Transformer":
        return TransformerModel(input_dim, hidden_dim, num_layers, dropout)
    return RNNClassifier(input_dim, hidden_dim, model_type, num_layers, dropout)

def get_loader(X_in, y_in, batch_size, shuffle=True):
    X_tensor = torch.tensor(X_in, dtype=torch.float32)
    y_tensor = torch.tensor(y_in, dtype=torch.float32).view(-1, 1)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=shuffle)

def auprc_score(y_true, y_prob):
    p, r, _ = precision_recall_curve(y_true, y_prob)
    return auc(r, p)

# ======================================================================================
# ============================ CLINICAL METRICS HELPERS ===============================
# ======================================================================================

def sens_at_fixed_spec(y_true, y_prob, spec_target=0.90):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    spec = 1.0 - fpr
    idx = np.where(spec >= spec_target)[0]
    if len(idx) == 0:
        return np.nan, np.nan
    best_i = idx[np.argmax(tpr[idx])]
    return float(tpr[best_i]), float(thr[best_i])

def calibration_curve_bins(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    rows = []
    for b in range(n_bins):
        mask = (bin_ids == b)
        if mask.sum() == 0:
            rows.append((b, 0, np.nan, np.nan))
        else:
            rows.append((b, int(mask.sum()), float(np.mean(y_prob[mask])), float(np.mean(y_true[mask]))))
    return pd.DataFrame(rows, columns=["bin", "count", "mean_pred", "frac_pos"])

def eval_point_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    out = {}
    out["AUC"] = float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan
    out["AUPRC"] = float(auprc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan
    out["F1"] = float(f1_score(y_true, y_pred)) if len(np.unique(y_true)) > 1 else np.nan
    out["Brier"] = float(brier_score_loss(y_true, y_prob))
    return out

def bootstrap_ci_metrics(y_true, y_prob, B=2000, seed=0, spec_targets=(0.90,)):
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    n = len(y_true)

    point = eval_point_metrics(y_true, y_prob, threshold=0.5)
    for s in spec_targets:
        sens, thr = sens_at_fixed_spec(y_true, y_prob, spec_target=s)
        point[f"Sens@Spec{int(s*100)}"] = sens
        point[f"Thr@Spec{int(s*100)}"] = thr

    dist = {k: [] for k in point.keys()}
    valid = 0

    for _ in range(B):
        idx = rng.randint(0, n, size=n)
        yt = y_true[idx]
        yp = y_prob[idx]
        if len(np.unique(yt)) < 2:
            continue
        valid += 1
        m = eval_point_metrics(yt, yp, threshold=0.5)
        for s in spec_targets:
            sens, thr = sens_at_fixed_spec(yt, yp, spec_target=s)
            m[f"Sens@Spec{int(s*100)}"] = sens
            m[f"Thr@Spec{int(s*100)}"] = thr
        for k, v in m.items():
            dist[k].append(v)

    rows = []
    for k, v_point in point.items():
        arr = np.asarray(dist[k], dtype=float)
        if arr.size == 0:
            rows.append((k, v_point, np.nan, np.nan, valid))
        else:
            lo = float(np.nanpercentile(arr, 2.5))
            hi = float(np.nanpercentile(arr, 97.5))
            rows.append((k, v_point, lo, hi, valid))
    return pd.DataFrame(rows, columns=["metric", "point", "ci_low", "ci_high", "n_valid_bootstrap"])

# ======================================================================================
# ============================== GRID SEARCH TRAINING =================================
# ======================================================================================

def stratified_sample(y_train, ratio=0.5, pos_ratio=1/3, seed=0):
    rng = np.random.RandomState(seed)
    y_flat = y_train.flatten()
    idx_0 = np.where(y_flat == 0)[0]
    idx_1 = np.where(y_flat == 1)[0]

    total = int(len(y_flat) * ratio)
    n1 = int(total * pos_ratio)
    n0 = total - n1
    if len(idx_0) < n0 or len(idx_1) < n1:
        return None

    idx = np.concatenate([
        rng.choice(idx_0, n0, replace=False),
        rng.choice(idx_1, n1, replace=False)
    ])
    rng.shuffle(idx)
    return idx

def train_one_config_cv(
    model_type, X_train, y_train, groups_train,
    hidden_dim, dropout, num_layers, batch_size, lr, wd, optimizer_name,
    epochs=200, n_repeats=5, n_splits=3, early_stop_patience=10,
    collect_cv_preds=True
):
    expected_metrics = n_repeats * n_splits
    all_aucs, all_auprcs, all_f1s = [], [], []
    used_epochs_all = []
    combo_best_val_loss = np.inf
    combo_best_state = None

    cv_pred_records = []

    for rep in range(n_repeats):
        sampled_idx = stratified_sample(y_train, ratio=0.5, pos_ratio=1/3, seed=rep)
        if sampled_idx is None:
            continue

        X_sub = X_train[sampled_idx]
        y_sub = y_train[sampled_idx]
        g_sub = groups_train[sampled_idx]

        gkf = GroupKFold(n_splits=n_splits)
        for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_sub, y_sub, g_sub)):

            model = build_model(model_type, X_train.shape[2], hidden_dim, num_layers, dropout).to(device)
            opt_cls = torch.optim.Adam if optimizer_name == "Adam" else torch.optim.SGD
            optimizer = opt_cls(model.parameters(), lr=lr, weight_decay=wd)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5)
            criterion = nn.BCEWithLogitsLoss()

            loader = get_loader(X_sub[tr_idx], y_sub[tr_idx], batch_size, shuffle=True)

            best_loss = float("inf")
            wait = 0
            used_epochs = 0
            best_state_fold = None

            for _ in range(epochs):
                model.train()
                losses = []
                for xb, yb in loader:
                    xb, yb = xb.to(device), yb.to(device)
                    optimizer.zero_grad()
                    logits = model(xb)
                    loss = criterion(logits, yb)
                    if torch.isnan(loss) or torch.isinf(loss):
                        continue
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    losses.append(loss.item())

                avg_loss = float(np.mean(losses)) if len(losses) else np.inf
                scheduler.step(avg_loss)
                used_epochs += 1

                if avg_loss < best_loss:
                    best_loss = avg_loss
                    wait = 0
                    best_state_fold = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                else:
                    wait += 1
                    if wait >= early_stop_patience:
                        break

            used_epochs_all.append(used_epochs)
            if best_state_fold is not None:
                model.load_state_dict(best_state_fold)

            # Validation metrics + (optional) save fold predictions
            model.eval()
            with torch.no_grad():
                val_x = torch.tensor(X_sub[val_idx], dtype=torch.float32).to(device)
                logits = model(val_x).cpu().flatten()
                prob = torch.sigmoid(logits).numpy()
                yt = y_sub[val_idx].flatten().astype(int)

            if len(np.unique(yt)) < 2:
                continue

            all_aucs.append(float(roc_auc_score(yt, prob)))
            all_auprcs.append(float(auprc_score(yt, prob)))
            all_f1s.append(float(f1_score(yt, (prob >= 0.5).astype(int))))

            if collect_cv_preds:
                cv_pred_records.append({
                    "model": model_type,
                    "hidden_dim": int(hidden_dim),
                    "dropout": float(dropout),
                    "num_layers": int(num_layers),
                    "batch_size": int(batch_size),
                    "lr": float(lr),
                    "weight_decay": float(wd),
                    "optimizer": optimizer_name,
                    "repeat": int(rep),
                    "fold": int(fold),
                    "y_true": yt.tolist(),
                    "y_prob": prob.tolist(),
                })

            if best_loss < combo_best_val_loss and best_state_fold is not None:
                combo_best_val_loss = best_loss
                combo_best_state = best_state_fold

    is_complete = (len(all_aucs) == expected_metrics)
    summary = {
        "AUC_mean": float(np.mean(all_aucs)) if is_complete else np.nan,
        "AUC_std":  float(np.std(all_aucs))  if is_complete else np.nan,
        "AUPRC_mean": float(np.mean(all_auprcs)) if is_complete else np.nan,
        "AUPRC_std":  float(np.std(all_auprcs))  if is_complete else np.nan,
        "F1_mean": float(np.mean(all_f1s)) if is_complete else np.nan,
        "F1_std":  float(np.std(all_f1s))  if is_complete else np.nan,
        "n_metrics": int(len(all_aucs)),
        "expected_metrics": int(expected_metrics),
        "used_epochs_mean": float(np.mean(used_epochs_all)) if used_epochs_all else np.nan,
        "used_epochs_std": float(np.std(used_epochs_all)) if used_epochs_all else np.nan,
        "best_val_loss": float(combo_best_val_loss) if np.isfinite(combo_best_val_loss) else np.inf,
        "has_state": combo_best_state is not None
    }
    return summary, combo_best_state, cv_pred_records

# ======================================================================================
# ================================ TRAIN/TEST SPLIT ===================================
# ======================================================================================

# Split BEFORE scaling/PCA to avoid leakage
X_train_raw, X_test_raw, y_train, y_test, groups_train, groups_test = train_test_split(
    X, y, groups, test_size=0.2, random_state=42, stratify=y
)
print(f"Train N={len(X_train_raw)} | Test N={len(X_test_raw)}")

# Fit scaler on train only, then apply to train/test
scaler_x = fit_scaler_on_train(X_train_raw)
X_train = apply_scaler(scaler_x, X_train_raw)
X_test  = apply_scaler(scaler_x, X_test_raw)

# Augment train only (do NOT augment test/validation)
X_train_aug, y_train_aug, groups_train_aug = augment_data(X_train, y_train, groups_train, seed=0)
print(f"Train after augmentation N={len(X_train_aug)}")

# ======================================================================================
# ============================ GRID SEARCH + SAVE BEST PER MODEL =======================
# ======================================================================================

model_types = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]

best_per_model = {m: {"auc": -np.inf, "cfg": None, "state": None} for m in model_types}
grid_summary_rows = []

# NEW: store CV predictions like your *_7 code
cv_predictions_all = []

if RUN_GRID_SEARCH:
    epochs = 200
    n_repeats = 5
    n_splits = 3

    for model_type in model_types:
        for hidden_dim in [16, 32, 64, 128]:
            for dropout in [0.0, 0.2]:
                for num_layers in [1, 2, 3]:
                    for batch_size in [16, 32]:
                        for lr in [0.001, 0.01]:
                            for wd in [0, 1e-4]:
                                for optimizer_name in ["Adam", "SGD"]:

                                    summary, best_state, cv_pred_records = train_one_config_cv(
                                        model_type,
                                        X_train_aug, y_train_aug, groups_train_aug,
                                        hidden_dim, dropout, num_layers, batch_size, lr, wd, optimizer_name,
                                        epochs=epochs, n_repeats=n_repeats, n_splits=n_splits,
                                        collect_cv_preds=True
                                    )

                                    # append CV predictions
                                    if cv_pred_records:
                                        cv_predictions_all.extend(cv_pred_records)

                                    row = {
                                        "model": model_type,
                                        "hidden_dim": hidden_dim,
                                        "dropout": dropout,
                                        "num_layers": num_layers,
                                        "batch_size": batch_size,
                                        "lr": lr,
                                        "weight_decay": wd,
                                        "optimizer": optimizer_name,
                                        "epochs": epochs,
                                        "repeats": n_repeats,
                                        "folds": n_splits,
                                        **summary
                                    }
                                    grid_summary_rows.append(row)

                                    # update best for this model type (require complete metrics)
                                    auc_mean = row["AUC_mean"]
                                    if np.isfinite(auc_mean) and auc_mean > best_per_model[model_type]["auc"] and summary["has_state"]:
                                        best_per_model[model_type]["auc"] = float(auc_mean)
                                        best_per_model[model_type]["cfg"] = {
                                            "model": model_type,
                                            "hidden_dim": int(hidden_dim),
                                            "dropout": float(dropout),
                                            "num_layers": int(num_layers),
                                            "batch_size": int(batch_size),
                                            "lr": float(lr),
                                            "weight_decay": float(wd),
                                            "optimizer": optimizer_name,
                                            "epochs": int(epochs),
                                            "repeats": int(n_repeats),
                                            "folds": int(n_splits),
                                            "threshold": float(threshold),
                                            "target_col": target_col,
                                            "timesteps": int(X_train_aug.shape[1]),
                                            "input_dim": int(X_train_aug.shape[2]),
                                            "feature_repr": "CNN+OSI" if USE_OSI_FEATURES else "CNN",
                                            "pca_components": None,
                                            "pca_mode": None,  # "full" or "cnn_only"
                                            "cnn_dim_per_tw": int(CNN_DIM_PER_TW) if USE_OSI_FEATURES else None,
                                        }
                                        best_per_model[model_type]["state"] = best_state
                                        print(f"\n🏆 BEST for {model_type}: AUC_mean={auc_mean:.4f}")

        # Save best artifacts per model_type
        bm = best_per_model[model_type]
        if bm["cfg"] is not None:
            model_dir = os.path.join(save_root, f"best_{model_type}")
            os.makedirs(model_dir, exist_ok=True)
            torch.save({"state_dict": bm["state"], "config": bm["cfg"]}, os.path.join(model_dir, "best_model_state.pt"))
            with open(os.path.join(model_dir, "best_model_config.json"), "w") as f:
                json.dump(bm["cfg"], f, indent=2)
            joblib.dump(scaler_x, os.path.join(model_dir, "scaler_x.joblib"))
            print(f"✅ Saved best artifacts for {model_type} -> {model_dir}")

    # Save grid summary
    df_grid = pd.DataFrame(grid_summary_rows)
    grid_csv_out = os.path.join(out_root_models, "Classification_grid_search_results(OSIandCNN)_8.csv")
    df_grid.to_csv(grid_csv_out, index=False)
    print("✅ Grid search summary saved:", grid_csv_out)

    # NEW: Save CV predictions JSON (for your downstream ROC/PR/F1 plotting scripts)
    cv_json_out = os.path.join(out_root_models, "Classification_cv_predictions(OSIandCNN)_8.json")
    with open(cv_json_out, "w") as f:
        json.dump(cv_predictions_all, f)
    print(f"✅ CV predictions saved: {len(cv_predictions_all)} records -> {cv_json_out}")

# ======================================================================================
# ============================= LOAD BEST-PER-MODEL ARTIFACTS ==========================
# ======================================================================================

def load_best_for_model(model_type: str):
    model_dir = os.path.join(save_root, f"best_{model_type}")
    ckpt = os.path.join(model_dir, "best_model_state.pt")
    cfgp = os.path.join(model_dir, "best_model_config.json")
    sxp  = os.path.join(model_dir, "scaler_x.joblib")
    if not (os.path.exists(ckpt) and os.path.exists(cfgp) and os.path.exists(sxp)):
        raise FileNotFoundError(f"Missing best artifacts for {model_type} in {model_dir}")
    with open(cfgp, "r") as f:
        cfg = json.load(f)
    sx = joblib.load(sxp)
    state = torch.load(ckpt, map_location="cpu")
    return cfg, sx, state["state_dict"]

# ======================================================================================
# =============================== VALIDATION BUILD (V3) ================================
# ======================================================================================

def load_validation_set(scaler_train: StandardScaler):
    df_val = pd.read_excel(xlsx_val, sheet_name=sheet_val)
    train_target_col = target_col
    val_target_col = "OSI_V3_12th_avg"

    if train_target_col not in df_val.columns:
        if val_target_col in df_val.columns:
            label_col = val_target_col
        else:
            raise ValueError(f"Missing target in {sheet_val}: neither {train_target_col} nor {val_target_col}")
    else:
        label_col = train_target_col

    # build required columns
    all_cols = []
    for tw in range(1, 7):
        all_cols += list(tw_cnn_features[tw])
        if USE_OSI_FEATURES:
            all_cols += list(osi_features.get(tw, []))

    missing = [c for c in all_cols if c not in df_val.columns]
    if missing:
        raise ValueError(f"Missing {len(missing)} feature cols in {sheet_val}. First 20: {missing[:20]}")

    df_val_model = df_val.dropna(subset=[label_col] + all_cols).copy()
    if len(df_val_model) == 0:
        raise ValueError("No validation rows left after dropna.")

    # X6
    X_list = []
    for tw in range(1, 7):
        cols = list(tw_cnn_features[tw])
        if USE_OSI_FEATURES:
            cols += list(osi_features.get(tw, []))
        X_list.append(df_val_model[cols].to_numpy(dtype=np.float32))

    X6_val = np.stack(X_list, axis=1)  # (N,6,D_total)
    X7_val = add_delta_window(X6_val)  # (N,7,D_total)

    y_val = (df_val_model[label_col].to_numpy(dtype=np.float32) >= threshold).astype(int)

    # scale using training scaler (no refit)
    X7_val_scaled = apply_scaler(scaler_train, X7_val)
    return X7_val_scaled, y_val, label_col

# ======================================================================================
# =============================== EVALUATION: TEST + VAL ===============================
# ======================================================================================

def predict_probs(model: nn.Module, X_in: np.ndarray, y_in: np.ndarray, batch_size=32):
    loader = get_loader(X_in, y_in, batch_size=batch_size, shuffle=False)
    probs, ys = [], []
    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb).detach().cpu().flatten()
            p = torch.sigmoid(logits).numpy()
            probs.append(p)
            ys.append(yb.numpy().flatten())
    y_prob = np.concatenate(probs)
    y_true = np.concatenate(ys).astype(int)
    return y_true, y_prob

def evaluate_and_save_best_per_model():
    test_payloads = []
    val_payloads  = []

    metrics_rows_test = []
    metrics_rows_val  = []

    for m in model_types:
        cfg, sx, state_dict = load_best_for_model(m)

        model = build_model(
            cfg["model"],
            input_dim=int(cfg["input_dim"]),
            hidden_dim=int(cfg["hidden_dim"]),
            num_layers=int(cfg["num_layers"]),
            dropout=float(cfg["dropout"])
        ).to(device)
        model.load_state_dict(state_dict)

        bs = int(cfg.get("batch_size", 32))

        # TEST
        y_true_test, y_prob_test = predict_probs(model, X_test, y_test, batch_size=bs)
        test_payloads.append({"model": m, "config": cfg, "y_true": y_true_test.tolist(), "y_prob": y_prob_test.tolist()})

        pm = eval_point_metrics(y_true_test, y_prob_test)
        rowt = {"model": m, **pm, "N": int(len(y_true_test))}
        for s in SPEC_TARGETS:
            sens, thr = sens_at_fixed_spec(y_true_test, y_prob_test, s)
            rowt[f"Sens@Spec{int(s*100)}"] = sens
            rowt[f"Thr@Spec{int(s*100)}"] = thr
        metrics_rows_test.append(rowt)

        # VALIDATION
        X_val_scaled, y_val, label_col_used = load_validation_set(sx)

        y_true_val, y_prob_val = predict_probs(model, X_val_scaled, y_val, batch_size=bs)
        val_payloads.append({"model": m, "config": cfg, "label_col_used": label_col_used,
                             "y_true": y_true_val.tolist(), "y_prob": y_prob_val.tolist()})

        pmv = eval_point_metrics(y_true_val, y_prob_val)
        rowv = {"model": m, **pmv, "N": int(len(y_true_val)), "label_col_used": label_col_used}
        for s in SPEC_TARGETS:
            sens, thr = sens_at_fixed_spec(y_true_val, y_prob_val, s)
            rowv[f"Sens@Spec{int(s*100)}"] = sens
            rowv[f"Thr@Spec{int(s*100)}"] = thr
        metrics_rows_val.append(rowv)

    # Save predictions JSON (ALL 5 models)
    test_preds_json = os.path.join(out_root_models, "Classification_best_model_test_predictions(OSIandCNN)_8.json")
    val_preds_json  = os.path.join(out_revision_dir, "Classification_best_model_validation_predictions(OSIandCNN)_8.json")

    with open(test_preds_json, "w") as f:
        json.dump(test_payloads, f, indent=2)
    with open(val_preds_json, "w") as f:
        json.dump(val_payloads, f, indent=2)

    # Save metric CSVs (point estimates)
    pd.DataFrame(metrics_rows_test).to_csv(
        os.path.join(out_root_models, "Classification_best_models_TEST_metrics_point(OSIandCNN)_8.csv"),
        index=False
    )
    pd.DataFrame(metrics_rows_val).to_csv(
        os.path.join(out_revision_dir, "Classification_best_models_VALIDATION_metrics_point(OSIandCNN)_8.csv"),
        index=False
    )

    print("✅ Saved ALL5 TEST preds:", test_preds_json)
    print("✅ Saved ALL5 VAL preds:",  val_preds_json)

    return test_preds_json, val_preds_json

# ======================================================================================
# ============================ BOOTSTRAP CI + CALIBRATION ==============================
# ======================================================================================

def run_bootstrap_and_calibration(preds_json_path: str, split_name: str, out_dir: str):
    with open(preds_json_path, "r") as f:
        payloads = json.load(f)

    ci_rows = []
    calib_rows = []

    for item in payloads:
        model_name = item["model"]
        y_true = np.array(item["y_true"], dtype=int)
        y_prob = np.array(item["y_prob"], dtype=float)

        ci_df = bootstrap_ci_metrics(
            y_true, y_prob, B=BOOTSTRAP_B, seed=0, spec_targets=tuple(SPEC_TARGETS)
        )
        ci_df.insert(0, "split", split_name)
        ci_df.insert(1, "model", model_name)
        ci_rows.append(ci_df)

        cal_df = calibration_curve_bins(y_true, y_prob, n_bins=CALIB_BINS)
        cal_df.insert(0, "split", split_name)
        cal_df.insert(1, "model", model_name)
        calib_rows.append(cal_df)

    ci_all = pd.concat(ci_rows, ignore_index=True)
    cal_all = pd.concat(calib_rows, ignore_index=True)

    ci_csv  = os.path.join(out_dir, f"BootstrapCI_{split_name}_OSIandCNN_8.csv")
    cal_csv = os.path.join(out_dir, f"CalibrationCurve_{split_name}_OSIandCNN_8.csv")

    ci_all.to_csv(ci_csv, index=False)
    cal_all.to_csv(cal_csv, index=False)

    print(f"✅ Bootstrap CI saved: {ci_csv}")
    print(f"✅ Calibration curve bins saved: {cal_csv}")

# ======================================================================================
# ================================ LEARNING CURVES =====================================
# ======================================================================================

def train_single_model_once(
    model_type: str,
    cfg: dict,
    X_train_in: np.ndarray,
    y_train_in: np.ndarray,
    groups_train_in: np.ndarray,
    X_test_in: np.ndarray,
    y_test_in: np.ndarray,
    X_val_in: np.ndarray,
    y_val_in: np.ndarray,
    seed: int = 0
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = build_model(
        model_type,
        input_dim=X_train_in.shape[2],
        hidden_dim=int(cfg["hidden_dim"]),
        num_layers=int(cfg["num_layers"]),
        dropout=float(cfg["dropout"])
    ).to(device)

    optimizer_name = cfg["optimizer"]
    lr = float(cfg["lr"])
    wd = float(cfg["weight_decay"])
    batch_size = int(cfg["batch_size"])
    epochs = int(cfg["epochs"])

    opt_cls = torch.optim.Adam if optimizer_name == "Adam" else torch.optim.SGD
    optimizer = opt_cls(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5)
    criterion = nn.BCEWithLogitsLoss()

    loader = get_loader(X_train_in, y_train_in, batch_size=batch_size, shuffle=True)

    best_loss = float("inf")
    wait = 0
    best_state = None

    for _ in range(epochs):
        model.train()
        losses = []
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            if torch.isnan(loss) or torch.isinf(loss):
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            losses.append(loss.item())

        avg_loss = float(np.mean(losses)) if len(losses) else np.inf
        scheduler.step(avg_loss)

        if avg_loss < best_loss:
            best_loss = avg_loss
            wait = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= 10:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    yt_test, yp_test = predict_probs(model, X_test_in, y_test_in, batch_size=batch_size)
    pm_test = eval_point_metrics(yt_test, yp_test)
    for s in SPEC_TARGETS:
        sens, thr = sens_at_fixed_spec(yt_test, yp_test, s)
        pm_test[f"Sens@Spec{int(s*100)}"] = sens
        pm_test[f"Thr@Spec{int(s*100)}"] = thr

    yt_val, yp_val = predict_probs(model, X_val_in, y_val_in, batch_size=batch_size)
    pm_val = eval_point_metrics(yt_val, yp_val)
    for s in SPEC_TARGETS:
        sens, thr = sens_at_fixed_spec(yt_val, yp_val, s)
        pm_val[f"Sens@Spec{int(s*100)}"] = sens
        pm_val[f"Thr@Spec{int(s*100)}"] = thr

    return pm_test, pm_val

def run_learning_curves():
    X_val_scaled, y_val, label_col_used = load_validation_set(scaler_x)

    rows = []
    unique_groups = np.unique(groups_train)

    for model_type in LEARNING_MODELS:
        cfg, _, _ = load_best_for_model(model_type)

        for frac in LEARNING_FRACTIONS:
            n_groups = max(2, int(len(unique_groups) * frac))
            for seed in LEARNING_SEEDS:
                rng = np.random.RandomState(seed)
                chosen_groups = rng.choice(unique_groups, size=n_groups, replace=False)
                mask = np.isin(groups_train, chosen_groups)

                X_sub = X_train[mask]
                y_sub = y_train[mask]
                g_sub = groups_train[mask]

                X_sub_aug, y_sub_aug, g_sub_aug = augment_data(X_sub, y_sub, g_sub, seed=seed)

                pm_test, pm_val = train_single_model_once(
                    model_type, cfg,
                    X_sub_aug, y_sub_aug, g_sub_aug,
                    X_test, y_test,
                    X_val_scaled, y_val,
                    seed=seed
                )

                rows.append({
                    "model": model_type,
                    "fraction": frac,
                    "seed": seed,
                    "label_col_val": label_col_used,
                    **{f"test_{k}": v for k, v in pm_test.items()},
                    **{f"val_{k}": v for k, v in pm_val.items()},
                    "n_train_patients": int(n_groups),
                    "n_train_samples": int(len(X_sub))
                })

                print(f"✅ LearningCurve {model_type} frac={frac} seed={seed} "
                      f"test_AUC={pm_test['AUC']:.3f} val_AUC={pm_val['AUC']:.3f}")

    df_lc = pd.DataFrame(rows)
    lc_csv = os.path.join(out_revision_dir, "LearningCurve_OSIandCNN_8.csv")
    df_lc.to_csv(lc_csv, index=False)
    print("✅ Learning curve results saved:", lc_csv)

# ======================================================================================
# ================================ PCA SENSITIVITY =====================================
# ======================================================================================

def run_pca_sensitivity():
    X_val_scaled, y_val, label_col_used = load_validation_set(scaler_x)

    rows = []
    for model_type in PCA_MODELS:
        cfg, _, _ = load_best_for_model(model_type)

        for k in PCA_COMPONENTS_LIST:
            if USE_OSI_FEATURES and PCA_ON_CNN_ONLY_IF_OSI:
                # PCA ONLY on CNN block
                pca = fit_pca_on_train_cnn_only(X_train, cnn_dim=CNN_DIM_PER_TW, n_components=int(k))

                X_train_pca = apply_pca_cnn_only_and_concat(X_train, pca, cnn_dim=CNN_DIM_PER_TW)
                X_test_pca  = apply_pca_cnn_only_and_concat(X_test,  pca, cnn_dim=CNN_DIM_PER_TW)
                X_val_pca   = apply_pca_cnn_only_and_concat(X_val_scaled, pca, cnn_dim=CNN_DIM_PER_TW)

                cfg_pca = dict(cfg)
                cfg_pca["pca_components"] = int(k)
                cfg_pca["pca_mode"] = "cnn_only"

            else:
                # PCA on full feature vector (original behavior)
                pca = fit_pca_on_train_full(X_train, n_components=int(k))

                X_train_pca = apply_pca_full(pca, X_train)
                X_test_pca  = apply_pca_full(pca, X_test)
                X_val_pca   = apply_pca_full(pca, X_val_scaled)

                cfg_pca = dict(cfg)
                cfg_pca["pca_components"] = int(k)
                cfg_pca["pca_mode"] = "full"

            X_train_pca_aug, y_train_aug2, g_train_aug2 = augment_data(X_train_pca, y_train, groups_train, seed=0)

            pm_test, pm_val = train_single_model_once(
                model_type, cfg_pca,
                X_train_pca_aug, y_train_aug2, g_train_aug2,
                X_test_pca, y_test,
                X_val_pca, y_val,
                seed=0
            )

            rows.append({
                "model": model_type,
                "pca_components": int(k),
                "pca_mode": cfg_pca.get("pca_mode"),
                "label_col_val": label_col_used,
                "input_dim_after_pca": int(X_train_pca.shape[2]),
                **{f"test_{k2}": v for k2, v in pm_test.items()},
                **{f"val_{k2}": v for k2, v in pm_val.items()},
            })

            print(f"✅ PCA{k} {model_type} ({cfg_pca.get('pca_mode')}): "
                  f"test_AUC={pm_test['AUC']:.3f} val_AUC={pm_val['AUC']:.3f} | input_dim={X_train_pca.shape[2]}")

    df_pca = pd.DataFrame(rows)
    pca_csv = os.path.join(out_revision_dir, "PCASensitivity_OSIandCNN_8.csv")
    df_pca.to_csv(pca_csv, index=False)
    print("✅ PCA sensitivity results saved:", pca_csv)

# ======================================================================================
# ===================================== RUN ===========================================
# ======================================================================================

test_preds_json = None
val_preds_json = None

if RUN_EVAL_BEST_MODELS:
    test_preds_json, val_preds_json = evaluate_and_save_best_per_model()

if RUN_BOOTSTRAP_CI_EVAL:
    if test_preds_json is None or val_preds_json is None:
        test_preds_json = os.path.join(out_root_models, "Classification_best_model_test_predictions(OSIandCNN)_8.json")
        val_preds_json  = os.path.join(out_revision_dir, "Classification_best_model_validation_predictions(OSIandCNN)_8.json")
    run_bootstrap_and_calibration(test_preds_json, "TEST", out_root_models)
    run_bootstrap_and_calibration(val_preds_json, "VALIDATION", out_revision_dir)

if RUN_LEARNING_CURVES:
    run_learning_curves()

if RUN_PCA_SENSITIVITY:
    run_pca_sensitivity()

print("✅ All done.")

### OSI+CNN Features (Classification) -- training + test + validation (_7)

In [ ]:
# === Try more (with best-model weight saving) + TEST + VALIDATION (NO retraining) ===
# Updated for:
#   ✅ repeats=5, folds=3 (dynamic expected_metrics; no more hard-coded 50)
#   ✅ safer metric aggregation + logging of expected vs actual
#   ✅ keeps your V3 validation label mapping (OSI_V3_12th_avg if needed)

import os
import math
import json
import joblib  # for saving/loading scaler
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from mamba_ssm import Mamba

# ========== Device ==========
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ========== Load Data ==========
df = pd.read_excel(
    "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1V2_df.xlsx",
    sheet_name="Sheet5"
)

target_col = "OSI_V2_12th_avg"
threshold = 7.5

base_template = ["OSI_mean_TW{}", "OSI_std_TW{}"]
tw_features = {
    tw: [f.format(tw) for f in base_template] + [f"f{i}_TW{tw}" for i in range(1, 257)]
    for tw in range(1, 7)
}

# ========== Build sequences ==========
X, y, groups = [], [], []
for idx, row in df.iterrows():
    sequence = []
    for tw in range(1, 7):
        cols = tw_features[tw]
        # If any required feature is missing for this TW, drop this sample
        if row[cols].isnull().any():
            break
        sequence.append(row[cols].values)

    if len(sequence) == 6 and not pd.isna(row[target_col]):
        X.append(sequence)
        y.append(1 if row[target_col] >= threshold else 0)
        groups.append(row["ResearchID"] if "ResearchID" in df.columns else idx)

X = np.array(X, dtype=np.float32)  # (N, 6, D)
y = np.array(y, dtype=np.float32)  # (N,)
groups = np.array(groups)

print(f"✅ Total usable samples: {len(X)} | Positives: {int(y.sum())} | Negatives: {int((1-y).sum())}")

# ========== Feature Engineering ==========
delta_features = X[:, -1, :] - X[:, 0, :]
delta_features = delta_features[:, np.newaxis, :]
X = np.concatenate([X, delta_features], axis=1)  # (N, 7, D)

# ========== Scaling ==========
scaler_x = StandardScaler()
X_scaled = scaler_x.fit_transform(X.reshape(-1, X.shape[2])).reshape(X.shape)

# ========== Train/Test Split ==========
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X_scaled, y, groups, test_size=0.2, random_state=42
)

print(f"Training samples before augmentation: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# ========== Data Augmentation ==========
def augment_data(X, y, groups):
    noise = np.random.normal(0, 0.005, X.shape).astype(np.float32)
    return (
        np.concatenate([X, X + noise]),
        np.concatenate([y, y]),
        np.concatenate([groups, groups])
    )

X_train, y_train, groups_train = augment_data(X_train, y_train, groups_train)
print(f"Training samples after augmentation: {len(X_train)}")

# ========== Model Definitions ==========
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return torch.sum(weights * x, dim=1)

class RNNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, rnn_type="RNN", num_layers=1, dropout=0.0):
        super().__init__()
        if num_layers == 1:
            dropout = 0.0
        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[rnn_type]
        self.rnn = rnn_cls(input_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.attn(out)
        return self.fc(out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, nhead, dropout):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=nhead,
            dropout=dropout,
            batch_first=True,
            layer_norm_eps=1e-5
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)

    def forward(self, x):
        return self.encoder(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0, nhead=None):
        super().__init__()
        if nhead is None:
            if hidden_dim % 8 == 0:
                nhead = hidden_dim // 8
            elif hidden_dim % 4 == 0:
                nhead = hidden_dim // 4
            else:
                nhead = 1

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )
        self.pe = PositionalEncoding(hidden_dim)

        if num_layers == 1:
            self.stack = nn.Sequential(*[TransformerBlock(hidden_dim, nhead, 0.0)])
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(TransformerBlock(hidden_dim, nhead, dropout), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])

        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pe(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

class MambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba = Mamba(d_model=dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return self.norm(self.mamba(x))

class MambaClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        if num_layers == 1:
            self.stack = nn.Sequential(*[MambaBlock(hidden_dim)])
        else:
            self.stack = nn.Sequential(*[
                nn.Sequential(MambaBlock(hidden_dim), nn.Dropout(dropout))
                for _ in range(num_layers)
            ])
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.apply(init_weights)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.norm(x)
        x = self.stack(x)
        x = self.attn(x)
        return self.fc(x)

def build_model(model_type, input_dim, hidden_dim, num_layers, dropout):
    if model_type == "Mamba":
        return MambaClassifier(input_dim, hidden_dim, num_layers, dropout)
    if model_type == "Transformer":
        return TransformerModel(input_dim, hidden_dim, num_layers, dropout)
    return RNNClassifier(input_dim, hidden_dim, model_type, num_layers, dropout)

# ========== Utility ==========
def get_loader(X, y, batch_size, shuffle=True):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=shuffle)

def auprc(y_true, y_prob):
    p, r, _ = precision_recall_curve(y_true, y_prob)
    return auc(r, p)

def stratified_sample(y_train, ratio=0.5, pos_ratio=1/3):
    """
    Subsample training labels to keep compute manageable:
      - take ratio fraction of total
      - within that, pos_ratio fraction are positives
    Returns:
      idx (np.ndarray) or (None, None) if not enough samples
    """
    y_flat = y_train.flatten()
    idx_0 = np.where(y_flat == 0)[0]
    idx_1 = np.where(y_flat == 1)[0]

    total = int(len(y_flat) * ratio)
    n1 = int(total * pos_ratio)
    n0 = total - n1

    if len(idx_0) < n0 or len(idx_1) < n1:
        return None, None

    idx = np.concatenate([
        np.random.choice(idx_0, n0, replace=False),
        np.random.choice(idx_1, n1, replace=False)
    ])
    np.random.shuffle(idx)
    return idx, total

# ========== Save directory + best tracking ==========
save_root = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/SavedModels_Stats_CNN_7"
os.makedirs(save_root, exist_ok=True)

global_best_auc = -np.inf
global_best_info = None
global_best_state_dict = None

# ========== Training schedule controls ==========
epochs = 200
n_repeats = 5
n_splits = 3
expected_metrics = n_repeats * n_splits

# ========== Train Loop ==========
results = {"cv_predictions": [], "summary": []}
model_types = ["RNN", "LSTM", "GRU", "Transformer", "Mamba"]

for model_type in model_types:
    for hidden_dim in [16, 32, 64, 128]:
        for dropout in [0.0, 0.2]:
            for num_layers in [1, 2, 3]:
                for batch_size in [16, 32]:
                    for lr in [0.001, 0.01]:
                        for wd in [0, 1e-4]:
                            for optimizer_name in ["Adam", "SGD"]:

                                all_aucs, all_auprcs, all_f1s = [], [], []
                                used_epochs_all = []

                                combo_best_val_loss = np.inf
                                combo_best_state_dict = None

                                for repeat in range(n_repeats):
                                    sampled_idx, _ = stratified_sample(y_train)
                                    if sampled_idx is None:
                                        print(f"[{model_type}] Skipping repeat {repeat+1}: not enough pos/neg after stratified_sample().")
                                        continue

                                    X_sub = X_train[sampled_idx]
                                    y_sub = y_train[sampled_idx]
                                    groups_sub = groups_train[sampled_idx]

                                    gkf = GroupKFold(n_splits=n_splits)

                                    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_sub, y_sub, groups_sub)):
                                        print(f"[Repeat {repeat+1}/{n_repeats} | Fold {fold+1}/{n_splits}] "
                                              f"Train size: {len(tr_idx)} | Val size: {len(val_idx)} | Test size: {len(y_test)}")

                                        model = build_model(model_type, X_train.shape[2], hidden_dim, num_layers, dropout).to(device)

                                        optimizer_cls = torch.optim.Adam if optimizer_name == "Adam" else torch.optim.SGD
                                        optimizer = optimizer_cls(model.parameters(), lr=lr, weight_decay=wd)
                                        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5)
                                        criterion = nn.BCEWithLogitsLoss()

                                        loader = get_loader(X_sub[tr_idx], y_sub[tr_idx], batch_size, shuffle=True)

                                        best_loss = float("inf")
                                        wait = 0
                                        used_epochs = 0
                                        best_state_dict_fold = None

                                        for epoch in range(epochs):
                                            model.train()
                                            epoch_losses = []
                                            for xb, yb in loader:
                                                xb, yb = xb.to(device), yb.to(device)
                                                optimizer.zero_grad()
                                                pred = model(xb)
                                                loss = criterion(pred, yb)
                                                if torch.isnan(loss) or torch.isinf(loss):
                                                    continue
                                                loss.backward()
                                                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                                                optimizer.step()
                                                epoch_losses.append(loss.item())

                                            avg_loss = float(np.mean(epoch_losses)) if len(epoch_losses) else np.inf
                                            scheduler.step(avg_loss)
                                            used_epochs += 1

                                            if avg_loss < best_loss:
                                                best_loss = avg_loss
                                                wait = 0
                                                best_state_dict_fold = {k: v.detach().cpu().clone()
                                                                       for k, v in model.state_dict().items()}
                                            else:
                                                wait += 1
                                                if wait >= 10:
                                                    break

                                        used_epochs_all.append(used_epochs)

                                        # Validation metrics (use best fold weights)
                                        if best_state_dict_fold is not None:
                                            model.load_state_dict(best_state_dict_fold)

                                        model.eval()
                                        with torch.no_grad():
                                            val_input = torch.tensor(X_sub[val_idx], dtype=torch.float32).to(device)
                                            logits_tensor = model(val_input).cpu().flatten()
                                            probs = torch.sigmoid(logits_tensor).numpy()
                                            y_true = y_sub[val_idx].flatten()

                                        valid_mask = np.isfinite(probs)
                                        if valid_mask.sum() == 0:
                                            print("⚠️  Skipping metrics: no finite probabilities.")
                                            continue

                                        probs = probs[valid_mask]
                                        y_true = y_true[valid_mask]
                                        preds = (probs >= 0.5).astype(int)

                                        # If a fold ends up with only one class, roc_auc_score will error
                                        if len(np.unique(y_true)) < 2:
                                            print("⚠️  Skipping AUC/AUPRC/F1: validation has <2 classes after masking.")
                                            continue

                                        all_aucs.append(float(roc_auc_score(y_true, probs)))
                                        all_auprcs.append(float(auprc(y_true, probs)))
                                        all_f1s.append(float(f1_score(y_true, preds)))

                                        results["cv_predictions"].append({
                                            "model": model_type,
                                            "hidden_dim": hidden_dim,
                                            "dropout": dropout,
                                            "num_layers": num_layers,
                                            "batch_size": batch_size,
                                            "lr": lr,
                                            "weight_decay": wd,
                                            "optimizer": optimizer_name,
                                            "repeat": repeat,
                                            "fold": fold,
                                            "y_true": y_true.tolist(),
                                            "y_prob": probs.tolist()
                                        })

                                        # Track best fold model for this combo (by best_loss)
                                        if best_loss < combo_best_val_loss and best_state_dict_fold is not None:
                                            combo_best_val_loss = best_loss
                                            combo_best_state_dict = best_state_dict_fold

                                # -------- Summary (dynamic completeness) --------
                                is_complete = (len(all_aucs) == expected_metrics)

                                auc_mean = float(np.mean(all_aucs)) if is_complete else np.nan
                                auc_std  = float(np.std(all_aucs))  if is_complete else np.nan
                                auprc_mean = float(np.mean(all_auprcs)) if is_complete else np.nan
                                auprc_std  = float(np.std(all_auprcs))  if is_complete else np.nan
                                f1_mean = float(np.mean(all_f1s)) if is_complete else np.nan
                                f1_std  = float(np.std(all_f1s))  if is_complete else np.nan

                                # epochs stats: still useful even if some metric folds were skipped
                                used_epochs_mean = float(np.mean(used_epochs_all)) if used_epochs_all else np.nan
                                used_epochs_std  = float(np.std(used_epochs_all))  if used_epochs_all else np.nan

                                results["summary"].append({
                                    "model": model_type,
                                    "hidden_dim": hidden_dim,
                                    "dropout": dropout,
                                    "num_layers": num_layers,
                                    "batch_size": batch_size,
                                    "lr": lr,
                                    "weight_decay": wd,
                                    "optimizer": optimizer_name,
                                    "epochs": epochs,
                                    "repeats": n_repeats,
                                    "folds": n_splits,
                                    "expected_metrics": int(expected_metrics),
                                    "n_metrics": int(len(all_aucs)),
                                    "AUC_mean": auc_mean,
                                    "AUC_std": auc_std,
                                    "AUPRC_mean": auprc_mean,
                                    "AUPRC_std": auprc_std,
                                    "F1_mean": f1_mean,
                                    "F1_std": f1_std,
                                    "used_epochs_mean": used_epochs_mean,
                                    "used_epochs_std": used_epochs_std,
                                })

                                # Track global best by AUC_mean (only if complete)
                                if np.isfinite(auc_mean) and auc_mean > global_best_auc and combo_best_state_dict is not None:
                                    global_best_auc = float(auc_mean)
                                    global_best_info = {
                                        "model": model_type,
                                        "hidden_dim": int(hidden_dim),
                                        "dropout": float(dropout),
                                        "num_layers": int(num_layers),
                                        "batch_size": int(batch_size),
                                        "lr": float(lr),
                                        "weight_decay": float(wd),
                                        "optimizer": optimizer_name,
                                        "epochs": int(epochs),
                                        "repeats": int(n_repeats),
                                        "folds": int(n_splits),
                                        "AUC_mean": float(auc_mean),
                                        "AUPRC_mean": float(auprc_mean) if np.isfinite(auprc_mean) else None,
                                        "F1_mean": float(f1_mean) if np.isfinite(f1_mean) else None,
                                        "n_metrics": int(len(all_aucs)),
                                        "expected_metrics": int(expected_metrics),
                                        "input_dim": int(X_train.shape[2]),
                                        "timesteps": int(X_train.shape[1]),
                                        "threshold": float(threshold),
                                        "target_col": target_col,
                                    }
                                    global_best_state_dict = combo_best_state_dict

                                    print("\n🏆 NEW GLOBAL BEST MODEL FOUND:", global_best_info)

                                    torch.save(
                                        {"state_dict": global_best_state_dict, "config": global_best_info},
                                        os.path.join(save_root, "best_model_state.pt")
                                    )
                                    with open(os.path.join(save_root, "best_model_config.json"), "w") as f:
                                        json.dump(global_best_info, f, indent=2)

                                    joblib.dump(scaler_x, os.path.join(save_root, "scaler_x.joblib"))

# ========== Save Training Results ==========
df_results = pd.DataFrame(results["summary"])
grid_csv_out = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_grid_search_results(Stats_CNN)_7.csv"
df_results.to_csv(grid_csv_out, index=False)
print("✅ Grid search completed and saved:", grid_csv_out)

cv_json_out = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_cv_predictions(Stats_CNN)_7.json"
with open(cv_json_out, "w") as f:
    json.dump(results["cv_predictions"], f)
print(f"✅ CV predictions saved: {len(results['cv_predictions'])} records -> {cv_json_out}")
print("✅ Best model artifacts saved to:", save_root)

# ======================================================================================
# TEST + VALIDATION (Best saved model, NO retraining)
# ======================================================================================

def load_best_artifacts(save_root):
    ckpt_path = os.path.join(save_root, "best_model_state.pt")
    cfg_path = os.path.join(save_root, "best_model_config.json")
    sx_path = os.path.join(save_root, "scaler_x.joblib")

    for p in [ckpt_path, cfg_path, sx_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing best artifact: {p}")

    with open(cfg_path, "r") as f:
        cfg = json.load(f)

    sx = joblib.load(sx_path)

    state = torch.load(ckpt_path, map_location="cpu")
    state_dict = state["state_dict"] if isinstance(state, dict) and "state_dict" in state else state
    return cfg, sx, state_dict

best_cfg, best_scaler_x, best_state_dict = load_best_artifacts(save_root)

best_model = build_model(
    best_cfg["model"],
    input_dim=X_test.shape[2],
    hidden_dim=int(best_cfg["hidden_dim"]),
    num_layers=int(best_cfg["num_layers"]),
    dropout=float(best_cfg["dropout"])
).to(device)

best_model.load_state_dict(best_state_dict)
best_model.eval()

best_bs = int(best_cfg.get("batch_size", 32))

# --------------------------
# TEST (held-out X_test)
# --------------------------
test_loader = get_loader(X_test, y_test, batch_size=best_bs, shuffle=False)

all_probs, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = best_model(xb).detach().cpu().flatten()
        probs = torch.sigmoid(logits).numpy()
        all_probs.append(probs)
        all_true.append(yb.numpy().flatten())

y_prob_test = np.concatenate(all_probs)
y_true_test = np.concatenate(all_true).astype(int)

test_auc = float(roc_auc_score(y_true_test, y_prob_test)) if len(np.unique(y_true_test)) > 1 else float("nan")
test_auprc = float(auprc(y_true_test, y_prob_test)) if len(np.unique(y_true_test)) > 1 else float("nan")
test_pred = (y_prob_test >= 0.5).astype(int)
test_f1 = float(f1_score(y_true_test, test_pred)) if len(np.unique(y_true_test)) > 1 else float("nan")

print("\n✅ TEST (held-out split)")
print(f"AUC: {test_auc:.4f} | AUPRC: {test_auprc:.4f} | F1: {test_f1:.4f} | N={len(y_true_test)}")

test_results_df = pd.DataFrame([{
    "model": best_cfg["model"],
    "AUC_test": test_auc,
    "AUPRC_test": test_auprc,
    "F1_test": test_f1,
    "N_test": int(len(y_true_test)),
    **{k: best_cfg.get(k) for k in ["hidden_dim", "dropout", "num_layers", "batch_size", "lr", "weight_decay", "optimizer", "epochs", "repeats", "folds"]}
}])

test_results_csv = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_best_model_test_results(Stats_CNN)_7.csv"
test_preds_json = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_best_model_test_predictions(Stats_CNN)_7.json"
test_pred_csv = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/Fig8_ClassificationModels/Classification_best_model_test_predictions_per_sample(Stats_CNN)_7.csv"

test_results_df.to_csv(test_results_csv, index=False)

with open(test_preds_json, "w") as f:
    json.dump([{
        "model": best_cfg["model"],
        "y_true": y_true_test.tolist(),
        "y_prob": y_prob_test.tolist()
    }], f)

pd.DataFrame({
    "y_true": y_true_test.astype(int),
    "y_prob": y_prob_test,
    "y_pred": test_pred.astype(int),
}).to_csv(test_pred_csv, index=False)

print("✅ Test performance saved:", test_results_csv)
print("✅ Test predictions saved:", test_preds_json)
print("✅ Test per-sample CSV saved:", test_pred_csv)

# --------------------------
# VALIDATION on V3 Sheet15
# FIX: uses OSI_V3_12th_avg if OSI_V2_12th_avg is not present
# --------------------------
xlsx_val = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"
sheet_val = "Sheet15"
df_val = pd.read_excel(xlsx_val, sheet_name=sheet_val)

train_target_col = target_col        # "OSI_V2_12th_avg"
val_target_col = "OSI_V3_12th_avg"   # V3 sheet

if train_target_col not in df_val.columns:
    if val_target_col in df_val.columns:
        print(f"NOTE: '{train_target_col}' not found in {sheet_val}; using '{val_target_col}' instead.")
        label_col = val_target_col
    else:
        raise ValueError(f"Missing target in {sheet_val}: neither '{train_target_col}' nor '{val_target_col}' exists.")
else:
    label_col = train_target_col

# Required feature columns
all_feature_cols = [c for tw in range(1, 7) for c in tw_features[tw]]
missing_feats = [c for c in all_feature_cols if c not in df_val.columns]
if missing_feats:
    raise ValueError(f"Missing {len(missing_feats)} feature columns in {sheet_val}. First 30: {missing_feats[:30]}")

df_val_model = df_val.dropna(subset=[label_col] + all_feature_cols).copy()
print(f"\n✅ VALIDATION rows after dropna ({sheet_val}): {len(df_val_model)}")
if len(df_val_model) == 0:
    raise ValueError("No rows left after dropna in validation; cannot run validation.")

# Build X6 then add delta -> X7
X_list = [df_val_model[tw_features[tw]].to_numpy(dtype=np.float32) for tw in range(1, 7)]
X6_val = np.stack(X_list, axis=1)  # (N, 6, D)
delta_val = (X6_val[:, -1, :] - X6_val[:, 0, :])[:, np.newaxis, :]
X7_val = np.concatenate([X6_val, delta_val], axis=1)  # (N, 7, D)

y_true_val = (df_val_model[label_col].to_numpy(dtype=np.float32) >= threshold).astype(int)

# Scale using training-fitted scaler_x (NO refit)
N, T, D = X7_val.shape
X7_val_scaled = best_scaler_x.transform(X7_val.reshape(-1, D)).reshape(N, T, D)

val_loader = DataLoader(
    TensorDataset(
        torch.tensor(X7_val_scaled, dtype=torch.float32),
        torch.tensor(y_true_val, dtype=torch.float32).view(-1, 1),
    ),
    batch_size=best_bs,
    shuffle=False,
)

val_probs, val_true = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        logits = best_model(xb).detach().cpu().flatten()
        probs = torch.sigmoid(logits).numpy()
        val_probs.append(probs)
        val_true.append(yb.numpy().flatten())

y_prob_val = np.concatenate(val_probs)
y_true_val = np.concatenate(val_true).astype(int)

val_auc = float(roc_auc_score(y_true_val, y_prob_val)) if len(np.unique(y_true_val)) > 1 else float("nan")
val_auprc = float(auprc(y_true_val, y_prob_val)) if len(np.unique(y_true_val)) > 1 else float("nan")
val_pred = (y_prob_val >= 0.5).astype(int)
val_f1 = float(f1_score(y_true_val, val_pred)) if len(np.unique(y_true_val)) > 1 else float("nan")

print(f"\n✅ VALIDATION (V3 {sheet_val})")
print(f"AUC: {val_auc:.4f} | AUPRC: {val_auprc:.4f} | F1: {val_f1:.4f} | N={len(y_true_val)} | label_col={label_col}")

val_out_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/04-Results/PARDS_Risk_V2/20251105_TBME_Submission/20260301_Major_Revision"
os.makedirs(val_out_dir, exist_ok=True)

val_results_csv = os.path.join(val_out_dir, "Classification_best_model_validation_results(Stats_CNN)_7.csv")
val_preds_json = os.path.join(val_out_dir, "Classification_best_model_validation_predictions(Stats_CNN)_7.json")
val_pred_csv = os.path.join(val_out_dir, "Classification_best_model_validation_predictions_per_sample(Stats_CNN)_7.csv")

pd.DataFrame([{
    "model": best_cfg["model"],
    "sheet": sheet_val,
    "label_col_used": label_col,
    "AUC_val": val_auc,
    "AUPRC_val": val_auprc,
    "F1_val": val_f1,
    "N_val": int(len(y_true_val)),
    **{k: best_cfg.get(k) for k in ["hidden_dim", "dropout", "num_layers", "batch_size", "lr", "weight_decay", "optimizer", "epochs", "repeats", "folds"]}
}]).to_csv(val_results_csv, index=False)

with open(val_preds_json, "w") as f:
    json.dump([{
        "model": best_cfg["model"],
        "sheet": sheet_val,
        "label_col_used": label_col,
        "y_true": y_true_val.tolist(),
        "y_prob": y_prob_val.tolist()
    }], f)

pd.DataFrame({
    "y_true": y_true_val.astype(int),
    "y_prob": y_prob_val,
    "y_pred": val_pred.astype(int),
}).to_csv(val_pred_csv, index=False)

print("✅ Validation performance saved:", val_results_csv)
print("✅ Validation predictions saved:", val_preds_json)
print("✅ Validation per-sample CSV saved:", val_pred_csv)
